# domain_adapt **v16-hybrid** — RadJEPA (where) × frozen CheXzero (what)

Hybrid causal-patch pipeline. The v15 single-model gate is split across two **frozen** backbones: a self-supervised **RadJEPA** ViT provides sharp, domain-invariant *spatial* grounding (backbone frozen, **only a linear probe is trained** on NIH), and **CheXzero (CLIP) stays 100% frozen / zero-shot** for the *semantic* match — its last layer is **never** trained, to preserve NIH→CheXpert transfer. A patch is **causal** only if it passes **both** gates (their overlap). All v15 metrics, ablations, and figures are preserved; full per-image records (image path, sex/age/view/labels/all metadata + every causal/spurious patch, scores and fp16 embedding) are saved for the next (CheXpert) round.


## Cell 1 — Setup, config & prompts

Imports, device/seed, and **all switches**. Key v15 settings: `COHORT_MODE="single_label"` (keep only images with **exactly one** of the 5 findings positive), `ALLOW_FALLBACK=False` (honest gate), `BINARY_PROB_TEMP=14.0` (de-saturate the softmax), plus the POS/NEG CheXzero prompt pairs. Also defines the Wilson-CI and Benjamini–Hochberg helpers.

In [1]:
!pip install -q open_clip_torch ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q "transformers==5.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
# =============================================================================
# domain_adapt_v15.py
# DOMAIN-ADAPTIVE PATCH-MINING PIPELINE FOR ZERO-SHOT CHEST-X-RAY FINDINGS
# =============================================================================
# This is the fully-remediated ("v15") single-file version of the 8-cell
# domain_adapt_v14 notebook. Every change vs v14 is tagged  [v15 FIX #n]  where
# n is the numbered item in the code/methodology review, and is explained in
# FIXES_v15.md. Nothing here executes on import except the (data-free) self
# test; the heavy Stage-2/3 and ablation drivers are guarded by RUN_* flags at
# the bottom so the module is safe to import for unit testing.
#
# ---------------------------------------------------------------------------
# WHAT CHANGED, IN ONE PARAGRAPH
# ---------------------------------------------------------------------------
# The v14 headline AUROC was circular (the report parser that *chooses* which
# classes to query is derived from the same reports the NIH labels come from),
# pooled train+val with no held-out split, evaluated on a non-comparable
# multi-label cohort, and leaned on a causal-gate *fallback* that the code's own
# ablation flagged as the only significant arm. v15:
#   * scores per-class AUROC ONLY on images that were actually queried for that
#     class, and reports the parser's precision/recall against the labels as a
#     separate, explicit confound table                              [FIX 1]
#   * keeps train / val / test strictly separate; primary metrics are on a
#     single held-out split (test if produced, else val)             [FIX 3]
#   * uses a SINGLE-LABEL cohort: exactly ONE of the 5 targets positive, so a
#     class's negatives are true negatives, not other-target positives [FIX 4]
#   * ships ALLOW_FALLBACK=False as the honest primary; fallback is an ablation
#     arm only, and localization is strictly causal-only              [FIX 6]
#   * de-saturates the binary-pair softmax with a calibrated temperature and
#     prints the empirical saturation fraction                        [FIX 7]
#   * loads the checkpoint strictly (asserts the missing/unexpected budget) and
#     documents the weights_only ACE surface                          [FIX 9]
#   * Benjamini-Hochberg-corrects the ablation comparisons            [FIX 10]
#   * forces GT-box images into the localization cohort and reports Wilson
#     intervals with explicit n                                       [FIX 11]
#   * replaces the theatre self-test with tests that execute the real ensemble
#     line and 100+ report-parser gold cases                          [FIX 12]
#   * trims the over-broad uncertainty cues that silently deleted TPs  [FIX 13]
# plus the smaller correctness fixes catalogued inline and in FIXES_v15.md.
# =============================================================================

# ============================ SECTION 1.1 — installs =========================
!pip install -q open_clip_torch ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q "transformers==5.0.0"
!pip install -q timm   # [HYBRID] ViT skeleton for the frozen RadJEPA backbone

# ============================ SECTION 1.2 — imports ==========================
import os, re, gc, sys, copy, math, time, glob, json, random, pickle, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import clip
import torchvision.transforms as T
from PIL import Image, ImageDraw

from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.ndimage import label as ndimage_label, binary_dilation
# [v15 correctness] sklearn imported ONCE here; cell 8's duplicate import removed.
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

# ==================== SECTION 1.3 — device + reproducibility =================
_T_START = time.time()
USE_CUDA = torch.cuda.is_available()
Device = torch.device("cuda" if USE_CUDA else "cpu")

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if USE_CUDA:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.set_float32_matmul_precision("high")

TARGET_CLASSES = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
N_CLASSES = len(TARGET_CLASSES)

# ============ SECTION 1.4 — run-scope + BEST-HYPERPARAMETER config ===========
# ---- [v14 FIX A/B] scoring-model switches (the two biggest catastrophic-output causes)
USE_BINARY_PROMPTS  = True   # [v14 FIX A] class prob = softmax over a POS/NEG prompt
                             # PAIR per class (CheXzero method), NOT a 5-way softmax
                             # across the mutually-competing target classes.
GLOBAL_ENSEMBLE_ALPHA = 0.50 # [v14 FIX B] final class score =
                             #   alpha * whole-image CheXzero prob
                             # + (1-alpha) * patch-max prob.
                             # alpha=1.0 => pure global baseline (report it!);
                             # alpha=0.0 => pure patch-mining (old behaviour).
                             # NOTE [FIX 2]: a constant alpha*global term is shared
                             # by every patch in an image -> ties. The de-circularised
                             # metrics restrict per-class AUROC to queried images.

# [v15 FIX 7] Calibration temperature for the binary 2-way softmax. logit_scale
# (~100) saturates probabilities (cos gap 0.05 -> p~0.993). Set BINARY_PROB_TEMP
# to a smaller value (e.g. 10-25, ideally tuned on a held-out split) to
# de-saturate; None uses the raw logit_scale (original v14 behaviour). The
# saturation fraction is printed by the metrics cell.
BINARY_PROB_TEMP = 14.0

# [v15 review #12 nit] renamed: MINI_REAL_RUN = tiny run on REAL data (needs
# ckpt+data); RUN_SELFTEST (below) = the GPU/data-free unit self-test.
MINI_REAL_RUN          = False   # tiny real-data dry run (needs data + ckpt)
MINI_TRAIN_N           = 40
MINI_VAL_N             = 20

# [v15 FIX 4 + explicit instruction] Cohort selection.
#   "single_label" -> exactly 1 of 5 positive        (DEFAULT — the requested cohort)
#   "multilabel"   -> >=1 of 5 positive              (old v14 behaviour; harder task)
#   "all"          -> no cohort filter
# "single_label" makes each class's negatives TRUE negatives (not other-target
# positives). It is still NOT identical to published NIH zero-shot (no
# No-Finding rows, no multi-label rows) — every results caption must say so.
COHORT_MODE            = "single_label"
MULTICLASS_ONLY        = (COHORT_MODE != "all")   # kept for back-compat call sites

FAST_MODE              = False   # [v14 FIX C] keep False: FAST_MODE drops the 64px
                                 # scale that Atelectasis (small/subtle) needs.
MAX_TRAIN_IMAGES       = None
MAX_VAL_IMAGES         = 2000
MAX_TEST_IMAGES        = 2000
USE_LABEL_BACKUP_TRAIN = False
USE_LABEL_BACKUP_VAL   = False

WALLCLOCK_BUDGET_HOURS = 9.0
CKPT_EVERY_IMAGES      = 500
CKPT_EVERY_MIN         = 12
VIS_MAX_PER_SPLIT      = 150

IMAGE_SIZE             = 512
CLIP_EMBED_DIM         = 512
PATCH_ENCODE_BATCH_SZ  = 128
TEXT_ENCODE_BATCH_SZ   = 64
ENABLE_GRADCAM         = True
ENABLE_ZOOM_REFINEMENT = True
ENABLE_VISUALIZATION   = False
# [v15 correctness] store patch embeddings at fp16 in the checkpoint to keep the
# growing pickle small (each PatchDocument carries a 512-float vector).
CKPT_EMBED_FP16        = True

MIN_BOX_SIZE = 24
DISCARD_UNCERTAIN = True

# ============ SECTION 1.5 — scoring weights, thresholds, geometry ============
SCORE_W_SEM  = 0.45
SCORE_W_PROB = 0.35
SCORE_W_GC   = 0.20
# NOTE [FIX 8]: the three score components are correlated (fill_query_vectors
# blends the snippet 35% toward the class prompt; zeroshot_prob comes from the
# same class prompts; GradCAM uses a class-prompt objective at weight 0.35), and
# s01=(cos+1)/2 sits ~0.6 for essentially all CLIP cosines so SCORE_W_SEM adds
# almost no ranking variance. Ablation arm J:decorrelate quantifies this.

SEMANTIC_THRESHOLD = 0.16
SEMANTIC_THRESHOLD_PER_CLASS = {
    "Atelectasis": 0.16, "Cardiomegaly": 0.18, "Consolidation": 0.14,
    "Edema": 0.15, "Pleural Effusion": 0.18,
}
GRADCAM_WEAK_THR   = 0.08
SPATIAL_THRESH     = 0.18
FALLBACK_TOP_K     = 2

# [v15 FIX 6] Master switch for the causal-gate fallback. The v13/v14 ablation
# showed G:no_fallback is the ONLY statistically significant arm and that the
# Edema headline is fallback-driven. When the fallback fires, top-2 patches by
# combined_score are relabelled causal=True, BYPASSING the causal gate — which
# makes any causal claim unfalsifiable. Default is now FALSE (honest gate);
# fallback is reported only as ablation arm G. Set True to reproduce v14 AUROC.
ALLOW_FALLBACK     = False
MIN_CAUSAL_PATCHES = 2
STRIDE_BASE        = 32
GRADCAM_SNIPPET_W  = 0.65
GRADCAM_CLASS_W    = 0.35

CAUSAL_CONTAIN_MODE    = "peak_or_cover"
CAUSAL_MASK_COVER_FRAC = 0.35
CAUSAL_MASK_DILATE_PX  = 8

CAUSAL_QUERY_CLASS_BLEND = 0.35

# [v14 FIX C] per-class scale map, honoured by discover_patches_for_plan even
# when FAST_MODE is off — small findings keep 64px.
_SCALE_MAP = {
    "Atelectasis": [64, 128], "Cardiomegaly": [128, 256],
    "Consolidation": [128, 256], "Edema": [64, 128],
    "Pleural Effusion": [128, 256],
}

# ---------------------------------------------------------------------------
# [v15 correctness] FAST_MODE-derived knobs are read through helpers so an
# ablation that flips FAST_MODE at runtime actually takes effect (the v14 code
# froze these at import). Anything the ablation harness may toggle is a runtime
# lookup, never a def-time default argument.
# ---------------------------------------------------------------------------
def _max_iter():                 return 2 if FAST_MODE else 3
def _max_candidate_boxes():      return 100 if FAST_MODE else 160
def _max_zoom_candidate_boxes(): return 80 if FAST_MODE else 120
def _conf_threshold():           return 0.30 if FAST_MODE else 0.40
def _top_k_per_find():           return 3 if FAST_MODE else 5
def _patch_scales():             return [128, 256] if FAST_MODE else [64, 128, 256]
def _zoom_scales():              return [64] if FAST_MODE else [48, 64, 96]
def _spur_in_max_keep():         return _top_k_per_find()

SPUR_IN_GC_FLOOR   = GRADCAM_WEAK_THR

PERSISTENCE_N_LEVELS = 32
PERSISTENCE_TOP_K    = 2
PERSISTENCE_MIN_AREA = 16
PERSISTENCE_ENABLED  = True
# [v15 FIX — persistence birth/death bug] The v14/earlier code stored only `birth`
# and re-thresholded the reconstructed mask at `birth*0.95` (the HIGH level where a
# blob first appeared as an isolated peak) — collapsing the mask to a near-peak
# speck, so _causal_contains failed and the pipeline silently fell through to the
# crude GradCAM-threshold rule. We now store `death` (the level where the component
# merged / the sweep ended) in the persistence tuple and reconstruct just ABOVE
# death, which captures the component near its true maximum extent. A floor
# (PERSISTENCE_MIN_LEVEL_FRAC) prevents a pure survivor (death=vmin) from flooding
# the whole image. Mask areas are instrumented in _PERSIST_AREAS for the diagnostic.
PERSISTENCE_DEATH_MARGIN   = 0.05   # reconstruct at death + margin*(birth-death)
PERSISTENCE_MIN_LEVEL_FRAC = 0.15   # never threshold below vmin+frac*(vmax-vmin)
_PERSIST_AREAS = []                 # (mask_area_px, birth_minus_death) per used mask
# [v15 FIX 5] HONESTY NOTE: CLIP ViT-B/32 @224 gives a 7x7 token grid (49 real
# numbers) that we bilinearly upsample to IMAGE_SIZE before the level-set sweep.
# The "persistence" values are therefore a property of the interpolation kernel,
# NOT topology of the evidence at 512px. We keep the sweep because it is a
# robust connected-component selector, but every figure/caption calls it
# "smoothed-CAM connected-component selection", and PERSISTENCE_CAM_GRID records
# the true native CAM resolution for the methods section. A ViT-B/16 backbone
# (14x14) is the correct fix if the topological framing is load-bearing.
PERSISTENCE_CAM_GRID = 7          # native token grid of ViT-B/32 @224
PERSISTENCE_HONEST_NAME = "smoothed-CAM component selection"

OUTSIDE_ANAT_SCALES     = [64, 96, 128]
OUTSIDE_ANAT_INV_THRESH = 0.55
OUTSIDE_ANAT_MAX_CAND   = 120
OUTSIDE_ANAT_SIM_CAP    = 0.28
OUTSIDE_ANAT_MAX_KEEP   = 6
ARTIFACT_BORDER_FRAC = 0.15
ARTIFACT_MEAN_LOW    = 0.12
ARTIFACT_STD_HIGH    = 0.15
ARTIFACT_SCALES      = [64, 96]
ARTIFACT_SIM_CAP     = 0.28
ARTIFACT_MAX_KEEP    = 4

# ============ SECTION 1.5c — RadJEPA x CheXzero HYBRID causal gate ============
# [HYBRID] The v15 causal gate used a single vision-language model (CheXzero /
# CLIP) both for the "what" (semantic match) and the "where" (smoothed-CAM
# component selection). CLIP is weak at dense localization, so the "where" was
# noisy and causal/spurious patch embeddings overlapped in PCA.
#
# The hybrid design splits the two jobs across two frozen backbones:
#   * RadJEPA (self-supervised, DINOv2/I-JEPA-style ViT) — the "WHERE". We freeze
#     the backbone and train ONLY a linear probe on NIH for the 5 targets, then
#     take a probe-driven Grad-CAM. Self-supervised dense features localize
#     pathology sharply and are far more domain-invariant than CLIP's.
#   * CheXzero (CLIP) — the "WHAT". Kept 100% FROZEN / zero-shot. We NEVER train
#     its last layer: fine-tuning CheXzero on NIH would bake NIH shortcuts
#     (scanner text, borders) into the cosine scores and destroy NIH->CheXpert
#     transfer. Only the RadJEPA vision head is ever trained.
#
# A candidate patch is CAUSAL only if it passes BOTH gates (their overlap):
#   spatial gate (RadJEPA CAM mean in box, normalized 0..1) >= HYBRID_SPATIAL_THR
#   AND semantic gate (CheXzero cosine to the class/report prompt)  >= sem_thr
# The RadJEPA spatial gate REPLACES the CLIP smoothed-CAM persistence containment
# as the primary "where"; persistence is retained as the fallback when RadJEPA is
# unavailable, so the notebook still runs end-to-end exactly like v15.
ENABLE_RADJEPA_HYBRID = True    # master switch; auto-disabled if the backbone/ckpt
                               # cannot be loaded (RADJEPA_AVAILABLE set at load).
RADJEPA_AVAILABLE     = False   # set True by load_radjepa() on success.
HYBRID_SPATIAL_THR    = 0.60    # patch is spatially grounded if its normalized
                               # RadJEPA-CAM mean >= this (the ">0.60" spatial gate).
HYBRID_SEMANTIC_NORM_THR = 0.30 # informational per-image min-max-normalized semantic
                               # gate (stored for analysis). The BINDING semantic gate
                               # stays the tuned absolute SEMANTIC_THRESHOLD_PER_CLASS
                               # so recall/AUROC are not destabilized by tiny-n norms.
RADJEPA_KEEP_PERSISTENCE_AND = False  # if True, require RadJEPA spatial AND the CLIP
                               # persistence containment (stricter). Default False:
                               # RadJEPA spatial replaces persistence as the "where".

# ---- RadJEPA backbone + probe configuration ----
# RADJEPA_CKPT: a self-supervised chest-x-ray ViT checkpoint (I-JEPA / DINOv2 /
# RAD-DINO-style). Point this at your Kaggle dataset. If it is missing OR timm is
# unavailable, load_radjepa() falls back gracefully and the hybrid disables itself.
RADJEPA_CKPT          = os.environ.get(
    "RADJEPA_CKPT",
    "/kaggle/input/radjepa/pytorch/default/1/radjepa_vitb16.pth")
RADJEPA_ARCH          = "vit_base_patch16_224"  # timm arch used as the ViT skeleton
RADJEPA_IMG_SIZE      = 224
RADJEPA_EMBED_DIM     = 768     # ViT-B; overwritten from the loaded backbone.
RADJEPA_CKPT_MAX_MISSING    = 40
RADJEPA_CKPT_MAX_UNEXPECTED = 40
# Linear-probe training (ONLY the linear head is trained; backbone stays frozen).
RADJEPA_PROBE_EPOCHS     = 6
RADJEPA_PROBE_LR         = 1e-3
RADJEPA_PROBE_WD         = 1e-4
RADJEPA_PROBE_BATCH      = 64
RADJEPA_PROBE_MAX_IMAGES = 8000   # cap the training images used for the probe (speed)
RADJEPA_FEATURE_POOL     = "mean" # "mean" (patch-token mean) or "cls" pooling for probe
RADJEPA_CAM_ABS          = True   # use |grad*act| (class-agnostic magnitude) fallback
                                  # when the probe is degenerate; keeps CAM non-empty.

# ============ SECTION 1.6 — zero-shot prompts (POS + NEG pairs) ==============
ZS_PROMPTS = {
    "Atelectasis": [
        "atelectasis on chest x-ray", "lung collapse on chest radiograph",
        "plate-like atelectasis in the lung", "subsegmental atelectasis chest x-ray"],
    "Cardiomegaly": [
        "cardiomegaly on chest x-ray", "enlarged cardiac silhouette radiograph",
        "cardiac enlargement chest radiograph", "increased cardiothoracic ratio on x-ray"],
    "Consolidation": [
        "consolidation on chest x-ray", "airspace opacity in the lung",
        "lobar consolidation on radiograph", "air bronchogram in consolidated lung"],
    "Edema": [
        "pulmonary edema on chest x-ray", "bilateral interstitial edema radiograph",
        "vascular congestion in both lungs", "perihilar edema on chest radiograph"],
    "Pleural Effusion": [
        "pleural effusion on chest x-ray", "blunting of costophrenic angle",
        "pleural fluid on chest radiograph", "layering pleural effusion x-ray"],
}
# [v14 FIX A] NEGATIVE prompts — the second half of each CheXzero binary pair.
ZS_PROMPTS_NEG = {
    "Atelectasis": [
        "no atelectasis on chest x-ray", "no lung collapse",
        "lungs are fully expanded", "no volume loss in the lung"],
    "Cardiomegaly": [
        "normal heart size on chest x-ray", "no cardiomegaly",
        "normal cardiac silhouette", "normal cardiothoracic ratio"],
    "Consolidation": [
        "no consolidation on chest x-ray", "clear lungs without airspace opacity",
        "no lobar consolidation", "no air bronchogram"],
    "Edema": [
        "no pulmonary edema on chest x-ray", "no interstitial edema",
        "no vascular congestion", "clear lungs without edema"],
    "Pleural Effusion": [
        "no pleural effusion on chest x-ray", "sharp costophrenic angles",
        "no pleural fluid", "no layering effusion"],
}

# ============ SECTION 1.7 — dataset-specific configuration ===================
# >>> EDIT ME <<<
DATASET = "NIH"   # "NIH" or "MIMIC"
if DATASET == "NIH":
    VERSION = "v15"
    DATASET_NAME = "NIH-ChestXray"
    CSV_DIR       = "/kaggle/input/combinedreportimgpath"
    TRAIN_CSV     = os.path.join(CSV_DIR, "train_pairs_labeled.txt")
    VAL_CSV       = os.path.join(CSV_DIR, "val_pairs_labeled.txt")
    TEST_CSV      = os.path.join(CSV_DIR, "test_pairs_labeled.txt")
    CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                     "best_64_5e-05_original_22000_0.864.pt")
    NIH_IMAGE_DIR = "/kaggle/input/datasets/organizations/nih-chest-xrays/data"
    REPORT_DIR       = "/kaggle/input/datasets/anikazarin/nih-reports/reports"
    TRAIN_REPORT_DIR = os.path.join(REPORT_DIR, "train")
    VAL_REPORT_DIR   = os.path.join(REPORT_DIR, "val")
    TEST_REPORT_DIR  = os.path.join(REPORT_DIR, "test")
    OUT_DIR = "/kaggle/working/output"
    BBOX_CSV = "/kaggle/input/datasets/organizations/nih-chest-xrays/data/BBox_List_2017.csv"
    WALLCLOCK_BUDGET_HOURS = 9.0
    CKPT_EVERY_IMAGES = 500; CKPT_EVERY_MIN = 12
    VIEW_FROM_REPORT = False; COMBINED_CSV = None; VAL_CARVE_FRAC = 0.08
else:
    VERSION = "v15_mimic"
    DATASET_NAME = "MIMIC-CXR"
    COMBINED_CSV  = "/kaggle/input/datasets/anikataf/mimic-cxr/mimic_cxr_combined_full.csv"
    CHEXZERO_CKPT = ("/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
                     "best_64_5e-05_original_22000_0.864.pt")
    TRAIN_CSV = VAL_CSV = TEST_CSV = None
    REPORT_DIR = TRAIN_REPORT_DIR = VAL_REPORT_DIR = TEST_REPORT_DIR = None
    OUT_DIR = "/kaggle/working"
    BBOX_CSV = None   # MIMIC ships no bounding boxes
    WALLCLOCK_BUDGET_HOURS = 9.5
    CKPT_EVERY_IMAGES = 400; CKPT_EVERY_MIN = 10
    VIEW_FROM_REPORT = True; VAL_CARVE_FRAC = 0.08

VIS_DIR = f"{OUT_DIR}/vis"
os.makedirs(OUT_DIR, exist_ok=True)
for _sp in ("train", "val", "test"):
    os.makedirs(f"{VIS_DIR}/{_sp}", exist_ok=True)
STAGE2_OUT       = f"{OUT_DIR}/stage2_outputs.pkl"
STAGE3_RESULTS   = f"{OUT_DIR}/stage3_outputs.pkl"
STAGE3_CAUSAL_PT = f"{OUT_DIR}/stage3_causal.pt"
STAGE3_SPIN_PT   = f"{OUT_DIR}/stage3_spur_in.pt"
STAGE3_SPOUT_PT  = f"{OUT_DIR}/stage3_spur_out.pt"
STAGE3_META      = f"{OUT_DIR}/stage3_meta.pkl"
# [HYBRID / next-round] the ONE artifact that carries EVERYTHING needed to retrain
# on full NIH and to transfer to CheXpert: one record per image with image path,
# report path, split, all metadata (sex/age/view/patient-id/...), the 5 labels +
# full finding labels, queried classes, and — for every causal / spurious-in /
# spurious-out patch — box, scale, pathology, all scores (semantic, zeroshot_prob,
# gradcam, radjepa_spatial, combined + normalized), gate, and the fp16 visual
# embedding. Saved as a pickle AND flattened .pt tensors below.
STAGE3_FULL_RECORDS = f"{OUT_DIR}/stage3_full_records.pkl"
STAGE3_FULL_PT      = f"{OUT_DIR}/stage3_full_patch_bank.pt"
RADJEPA_PROBE_PT    = f"{OUT_DIR}/radjepa_linear_probe.pt"
_DEADLINE = _T_START + WALLCLOCK_BUDGET_HOURS * 3600.0

# ============ SECTION 1.8 — counters + helpers ==============================
_COUNTERS = defaultdict(int)

def _budget_left_h(deadline=None):
    dl = deadline if deadline is not None else _DEADLINE
    return max(0.0, (dl - time.time()) / 3600.0)

def _atomic_pickle(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    os.replace(tmp, path)

def _wilson_ci(k, n, z=1.96):
    """[v15 FIX 11] Wilson score interval for a binomial rate k/n. Returns
    (point, lo, hi). Honest small-n reporting for pointing-game / IoU."""
    if n <= 0:
        return (float("nan"), float("nan"), float("nan"))
    p = k / n
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return (p, max(0.0, centre - half), min(1.0, centre + half))

def _benjamini_hochberg(pvals, alpha=0.05):
    """[v15 FIX 10] BH-FDR. Returns (reject_mask, qvals) aligned to input order."""
    p = np.asarray(pvals, dtype=float)
    ok = ~np.isnan(p)
    m = int(ok.sum())
    reject = np.zeros_like(p, dtype=bool)
    q = np.full_like(p, np.nan)
    if m == 0:
        return reject, q
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[idx])]
    ranked = p[order]
    qv = ranked * m / (np.arange(1, m + 1))
    qv = np.minimum.accumulate(qv[::-1])[::-1]        # enforce monotonicity
    q[order] = np.clip(qv, 0, 1)
    thresh = ranked <= (np.arange(1, m + 1) / m) * alpha
    if thresh.any():
        kmax = np.max(np.where(thresh)[0])
        reject[order[:kmax + 1]] = True
    return reject, q

print(f"[cell 1] DATASET={DATASET_NAME} ({VERSION}) | device={Device} | "
      f"cohort={COHORT_MODE} | binary_prompts={USE_BINARY_PROMPTS} | "
      f"ensemble_alpha={GLOBAL_ENSEMBLE_ALPHA} | prob_temp={BINARY_PROB_TEMP} | "
      f"fallback={ALLOW_FALLBACK} | fast={FAST_MODE}")
print(f"[cell 1] HYBRID: radjepa_hybrid={ENABLE_RADJEPA_HYBRID} "
      f"spatial_thr={HYBRID_SPATIAL_THR} sem_norm_thr={HYBRID_SEMANTIC_NORM_THR} "
      f"| RadJEPA ckpt={RADJEPA_CKPT} | CheXzero stays FROZEN (zero-shot).")


## Cell 2 — Data structures, anatomy priors, report parser

Dataclasses (`QueryPlan`/`PatchDocument`/`ImagePatchResult`, now carrying `queried_classes`), the anatomical Gaussian-prior heatmaps, and the report parser. `_UNC_CUES` is trimmed (FIX 13) so boilerplate like *correlate clinically* / *versus* no longer deletes true positives; `sentence_status` is covered by 102 gold cases in Cell 7.

In [ ]:
# =============================================================================
# CELL 2 / 8  —  DATA STRUCTURES, ANATOMY PRIORS, REPORT PARSING
# =============================================================================

# ==================== SECTION 2.1 — data structures =========================
@dataclass
class AnatomicalPrior:
    region_name: str
    center_x: float; center_y: float; sigma_x: float; sigma_y: float

@dataclass
class PathologyQueryItem:
    pathology: str
    text_snippet: str
    query_vector: Optional[torch.Tensor] = None
    anatomical_prior: Optional[AnatomicalPrior] = None
    confidence: float = 1.0
    negated: bool = False
    source: str = "rule"

@dataclass
class QueryPlan:
    image_name: str; image_path: str; report_path: str
    view_position: str; findings_text: str; impression_text: str
    query_items: List[PathologyQueryItem] = field(default_factory=list)
    suppressed_regions: List[str] = field(default_factory=list)
    # [v15 FIX 1] classes the parser DECIDED to query, recorded so the metrics
    # can (a) restrict AUROC to queried images and (b) score the parser itself.
    queried_classes: List[str] = field(default_factory=list)
    # [HYBRID / next-round] ALL raw metadata for this image (sex, age, view,
    # patient id, follow-up, original size, full finding labels, ...) and the
    # per-class GT label vector, carried so the saved outputs miss nothing.
    meta: Dict = field(default_factory=dict)
    labels: Dict = field(default_factory=dict)

@dataclass
class PatchDocument:
    image_name: str; pathology: str; scale: int
    box: Tuple[int, int, int, int]
    visual_embedding: torch.Tensor
    semantic_score: float; zeroshot_prob: float; gradcam_score: float
    combined_score: float; causal: bool; anatomical_region: str
    confidence: float; text_snippet: str
    zoom_level: int = 1
    spurious_source: str = "in_anatomy"
    selection_source: str = "threshold"
    # [HYBRID] RadJEPA "where" gate + normalized scores, stored per patch so the
    # overlap decision is fully auditable and reusable for the next round.
    radjepa_spatial_score: float = 0.0   # mean RadJEPA-CAM intensity inside the box
    radjepa_spatial_norm: float = 0.0    # same, normalized 0..1 across the class boxes
    semantic_norm: float = 0.0           # CheXzero cosine, min-max normed across boxes
    causal_gate: str = "none"            # which gate fired: hybrid / persistence / fallback / none

@dataclass
class ImagePatchResult:
    image_name: str; image_path: str; split: str
    causal_patches: List[PatchDocument] = field(default_factory=list)
    spurious_patches: List[PatchDocument] = field(default_factory=list)
    refined: bool = False; n_iterations: int = 1; used_fallback: bool = False
    queried_classes: List[str] = field(default_factory=list)   # [v15 FIX 1]
    # [HYBRID / next-round] image-level metadata + labels + report path threaded
    # through so nothing is lost when we persist the full per-image record.
    report_path: str = ""
    view_position: str = ""
    meta: Dict = field(default_factory=dict)
    labels: Dict = field(default_factory=dict)

# ==================== SECTION 2.2 — anatomy + heatmaps ======================
ANATOMY_COORDS_PA = {
    "right lung": (0.28, 0.42, 0.14, 0.22), "left lung": (0.72, 0.42, 0.14, 0.22),
    "bilateral lungs": (0.50, 0.42, 0.38, 0.22), "lungs": (0.50, 0.42, 0.38, 0.22),
    "lung": (0.50, 0.42, 0.38, 0.22),
    "right upper lobe": (0.28, 0.18, 0.10, 0.10), "right middle lobe": (0.28, 0.38, 0.10, 0.09),
    "right lower lobe": (0.28, 0.60, 0.10, 0.11), "left upper lobe": (0.72, 0.18, 0.10, 0.10),
    "left lower lobe": (0.72, 0.60, 0.10, 0.11), "lower lobes": (0.50, 0.62, 0.32, 0.10),
    "upper lobes": (0.50, 0.16, 0.32, 0.09), "lower lobe": (0.50, 0.62, 0.32, 0.10),
    "right costophrenic angle": (0.22, 0.81, 0.07, 0.06),
    "left costophrenic angle": (0.78, 0.81, 0.07, 0.06),
    "costophrenic angles": (0.50, 0.81, 0.36, 0.06), "costophrenic angle": (0.50, 0.81, 0.36, 0.06),
    "costophrenic": (0.50, 0.81, 0.36, 0.06),
    "cardiac silhouette": (0.53, 0.48, 0.12, 0.14), "heart": (0.53, 0.48, 0.12, 0.14),
    "cardiomegaly": (0.53, 0.48, 0.12, 0.14), "mediastinum": (0.50, 0.34, 0.08, 0.18),
    "mediastinal": (0.50, 0.34, 0.08, 0.18), "hila": (0.50, 0.42, 0.08, 0.08),
    "right hilum": (0.39, 0.42, 0.05, 0.06), "left hilum": (0.60, 0.42, 0.05, 0.06),
    "hilar": (0.50, 0.42, 0.08, 0.08), "apices": (0.50, 0.07, 0.26, 0.05),
    "right apex": (0.28, 0.06, 0.08, 0.04), "left apex": (0.72, 0.06, 0.08, 0.04),
    "apex": (0.50, 0.07, 0.26, 0.05), "right pleura": (0.17, 0.44, 0.05, 0.26),
    "left pleura": (0.83, 0.44, 0.05, 0.26), "pleural space": (0.50, 0.44, 0.42, 0.26),
    "pleural": (0.50, 0.44, 0.42, 0.26), "right hemidiaphragm": (0.30, 0.76, 0.12, 0.04),
    "left hemidiaphragm": (0.70, 0.78, 0.12, 0.04), "diaphragm": (0.50, 0.77, 0.32, 0.04),
    "trachea": (0.50, 0.20, 0.04, 0.08), "carina": (0.50, 0.30, 0.04, 0.03),
    "ribs": (0.50, 0.45, 0.40, 0.26), "clavicles": (0.50, 0.09, 0.30, 0.03),
    "spine": (0.50, 0.45, 0.04, 0.30),
}
ANATOMY_COORDS_AP = {
    **ANATOMY_COORDS_PA,
    "cardiac silhouette": (0.53, 0.52, 0.16, 0.17), "heart": (0.53, 0.52, 0.16, 0.17),
    "cardiomegaly": (0.53, 0.52, 0.16, 0.17),
    "right upper lobe": (0.29, 0.20, 0.12, 0.12), "left upper lobe": (0.71, 0.20, 0.12, 0.12),
    "right lower lobe": (0.29, 0.62, 0.11, 0.12), "left lower lobe": (0.71, 0.62, 0.11, 0.12),
}
_ANATOMY_SORTED = sorted(ANATOMY_COORDS_PA.keys(), key=len, reverse=True)

def get_anatomy_coords(region, view="PA"):
    lut = ANATOMY_COORDS_AP if str(view).upper() == "AP" else ANATOMY_COORDS_PA
    key = str(region).lower().strip()
    if key in lut:
        return lut[key]
    for k in _ANATOMY_SORTED:
        if k in key:
            return lut[k]
    return (0.50, 0.42, 0.36, 0.24)

def make_gaussian_heatmap_gpu(cx, cy, sx, sy, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    xs = torch.linspace(0, 1, W, device=device)
    ys = torch.linspace(0, 1, H, device=device)
    yg, xg = torch.meshgrid(ys, xs, indexing="ij")
    return torch.exp(-((xg - cx) ** 2 / (2 * sx ** 2) + (yg - cy) ** 2 / (2 * sy ** 2))).float()

def get_composite_heatmap_gpu(plan, H=IMAGE_SIZE, W=IMAGE_SIZE, device=None):
    if device is None:
        device = Device
    composite = torch.zeros(H, W, device=device)
    suppressed = torch.zeros(H, W, device=device)
    for item in plan.query_items:
        if item.negated or item.anatomical_prior is None:
            continue
        p = item.anatomical_prior
        composite += item.confidence * make_gaussian_heatmap_gpu(
            p.center_x, p.center_y, p.sigma_x, p.sigma_y, H, W, device)
    for region in plan.suppressed_regions:
        rx, ry, rsx, rsy = get_anatomy_coords(region, plan.view_position)
        suppressed += make_gaussian_heatmap_gpu(rx, ry, rsx, rsy, H, W, device)
    suppressed = suppressed.clamp(0, 1)
    maxc = composite.max()
    if maxc > 0:
        composite = composite / maxc
    return (composite * (1.0 - suppressed)).float()

# ==================== SECTION 2.3 — report parsing ==========================
_SECTION_HDR_PAT = re.compile(r"^\s*\d+\s*\)\s*", re.M)
_FINDINGS_PAT    = re.compile(r"^\s*(?:\d+\s*[\).:]\s*)?FINDINGS\b",   re.I | re.M)
_IMPRESSION_PAT  = re.compile(r"^\s*(?:\d+\s*[\).:]\s*)?IMPRESSION\b", re.I | re.M)
_NONCLINICAL_PAT = re.compile(
    r"^\s*(?:\d+\s*[\).:]\s*)?"
    r"(EXAM\s*/?\s*TECHNIQUES?|IMAGE\s*QUAL[A-Z]*\s*/?\s*TECHNICAL|"
    r"DEVICES\s*/?\s*LINES|LIMITATIONS?\s*/?\s*UNCERTAINTY|DOMAIN\s*VECTOR)", re.I | re.M)
_NEG_CUES = re.compile(
    r"\b(no\b|not\b|none\b|without|absent|clear\b|unremarkable|normal\b|"
    r"no evidence of|no acute|free of|negative for|unlikely|no significant|"
    r"no definite|is not seen|are not seen|resolved|ruled out)\b", re.I)
# [v15 FIX 13] Removed over-broad / boilerplate cues that silently deleted true
# positives when DISCARD_UNCERTAIN=True: bare "appears", "versus/vs", "further
# evaluation", and "correlate clinically" (boilerplate in a large fraction of
# reports). Kept genuine epistemic hedges. "appears to be" is retained.
_UNC_CUES = re.compile(
    r"\b(possible|possibly|probable|probably|may\b|might|suspected|"
    r"cannot exclude|cannot rule out|question of|questionable|"
    r"differential|borderline|likely|equivocal|indeterminate|"
    r"suggestive of|concerning for|could represent|may represent|"
    r"appears?\s+to\s+be|worrisome for)\b", re.I)
_VIS_LIMIT_CUES = re.compile(
    r"\bnot\s+(?:fully|well|completely|adequately|entirely)?\s*"
    r"(?:visualized|evaluated|assessed|included|imaged|seen on)", re.I)
_SCOPE_BREAK = re.compile(
    r"\b(but|however|although|though|except|aside from|other than|whereas|while)\b|[,;]", re.I)

_PROJECTION_FIELD = re.compile(
    r"projection\s*[:\(]\s*([A-Za-z]+(?:\s+(?:supine|erect|portable|upright))*)", re.I)
_PORTABLE_FIELD = re.compile(r"portable[-\s]?likelihood\s*[:\(]\s*(low|medium|moderate|high)", re.I)
_POSITION_FIELD = re.compile(r"patient\s+position\s*[:\(]\s*([A-Za-z\- ]+)", re.I)
_VIEW_AP_CUE = re.compile(r"\b(AP|SUPINE|SEMI[- ]?ERECT|SEMI[- ]?UPRIGHT|DECUBITUS)\b", re.I)
_VIEW_PA_CUE = re.compile(r"\bPA\b", re.I)

SEED_KEYWORDS = {
    "Atelectasis": ["atelectasis", "atelectatic", "volume loss", "plate-like", "discoid",
        "linear opacity", "linear opacities", "subsegmental", "crowding of",
        "elevated hemidiaphragm", "collapse"],
    "Cardiomegaly": ["cardiomegaly", "enlarged cardiac", "cardiac enlargement",
        "cardiothoracic ratio", "cardiac silhouette is enlarged", "heart size",
        "heart is enlarged", "heart appears enlarged", "enlarged heart",
        "cardiac silhouette", "cardiomediastinal silhouette", "cardiac contour"],
    "Consolidation": ["consolidation", "airspace opacity", "airspace opacities",
        "lobar opacity", "air bronchogram", "airspace disease", "patchy opacity",
        "patchy opacities", "focal opacity", "focal opacities", "increased opacity",
        "increased density", "ill-defined opacity", "ill-defined opacities",
        "infiltrate", "infiltrates", "dense opacity"],
    "Edema": ["edema", "oedema", "pulmonary congestion", "vascular congestion", "kerley",
        "perihilar", "interstitial markings", "vascular redistribution",
        "interstitial opacity", "interstitial opacities", "hazy opacity",
        "hazy opacities", "reticular opacity", "reticular opacities",
        "increased interstitial", "cephalization"],
    "Pleural Effusion": ["effusion", "pleural fluid", "blunting", "meniscus", "costophrenic",
        "pleural collection", "layering", "blunted costophrenic", "fluid in the pleural"],
}
DEFAULT_ANATOMY = {
    "Atelectasis": "lower lobes", "Cardiomegaly": "cardiac silhouette",
    "Consolidation": "right lower lobe", "Edema": "bilateral lungs",
    "Pleural Effusion": "right costophrenic angle",
}

def detect_view_from_report(raw, default="PA"):
    text = str(raw); head = text[:1200]
    m = _PROJECTION_FIELD.search(head)
    proj = m.group(1).strip().upper() if m else ""
    if proj.startswith("LATERAL"): return "LATERAL"
    if proj.startswith("AP"):      return "AP"
    if proj.startswith("PA"):      return "PA"
    mp = _PORTABLE_FIELD.search(head)
    if mp and mp.group(1).lower() in ("medium", "moderate", "high"): return "AP"
    mpos = _POSITION_FIELD.search(head)
    if mpos and re.search(r"supine|semi[- ]?erect|semi[- ]?upright|decubitus", mpos.group(1), re.I):
        return "AP"
    scan = re.sub(r"portable[-\s]?likelihood[^\n]*", " ", head, flags=re.I).upper()
    has_ap = bool(_VIEW_AP_CUE.search(scan)); has_pa = bool(_VIEW_PA_CUE.search(scan))
    if has_pa and not has_ap: return "PA"
    if has_ap and not has_pa: return "AP"
    return default

def parse_report_sections(raw):
    lines = str(raw).split("\n")
    sections = {"findings": "", "impression": ""}
    current, buf = None, []
    def _flush():
        if current and buf:
            prev = sections.get(current, ""); joined = "\n".join(buf).strip()
            sections[current] = (prev + "\n" + joined).strip() if prev else joined
    for line in lines:
        if _FINDINGS_PAT.search(line):     _flush(); current, buf = "findings", []
        elif _IMPRESSION_PAT.search(line): _flush(); current, buf = "impression", []
        elif _NONCLINICAL_PAT.search(line):_flush(); current, buf = None, []
        elif _SECTION_HDR_PAT.match(line): _flush(); current, buf = None, []
        elif current is not None:          buf.append(line)
    _flush()
    return sections

def _scoped(window, forward):
    m = _SCOPE_BREAK.search(window)
    if not m: return window
    return window[m.end():] if not forward else window[:m.start()]

def sentence_status(sentence, keyword):
    """Return one of {absent, negated, uncertain, positive} for `keyword` in
    `sentence`, scoped to the nearest clause boundary. This is the single
    highest-risk component in the pipeline; it is covered by 100+ gold cases in
    the self-test (SECTION 7.2)."""
    sl = str(sentence).lower(); kw = str(keyword).lower(); pos = sl.find(kw)
    if pos < 0: return "absent"
    prefix = _scoped(sl[:pos], forward=False)
    suffix = _scoped(sl[pos + len(kw):], forward=True)
    if _VIS_LIMIT_CUES.search(prefix) or _VIS_LIMIT_CUES.search(suffix): return "uncertain"
    if _NEG_CUES.search(prefix) or _NEG_CUES.search(suffix): return "negated"
    if _UNC_CUES.search(prefix) or _UNC_CUES.search(suffix): return "uncertain"
    return "positive"

def extract_keyword_sentences(text, keywords):
    ordered = sorted({str(k).lower() for k in keywords}, key=len, reverse=True)
    results = []
    for sent in re.split(r"[.\n;]", str(text)):
        sent = re.sub(r"\s+", " ", sent).strip()
        if len(sent) < 8: continue
        low = sent.lower()
        for kw in ordered:
            if kw in low:
                results.append((sent, sentence_status(sent, kw))); break
    return results

def extract_anatomy_mentions(text):
    tl = str(text).lower()
    return [k for k in _ANATOMY_SORTED if k in tl]

def _read_report(path):
    try:
        if isinstance(path, str) and os.path.exists(path):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
    except Exception:
        _COUNTERS["reports_read_error"] += 1
    return ""

def _strip_nonclinical(raw):
    keep, drop = [], False
    for line in str(raw).split("\n"):
        if _FINDINGS_PAT.search(line) or _IMPRESSION_PAT.search(line):
            drop = False; continue
        if _NONCLINICAL_PAT.search(line) or _SECTION_HDR_PAT.match(line):
            drop = True; continue
        if not drop: keep.append(line)
    return "\n".join(keep).strip()

def _clinical_text_from_raw(raw):
    secs = parse_report_sections(raw)
    findings = secs.get("findings", ""); impression = secs.get("impression", "")
    clinical = (findings + "\n" + impression).strip()
    if len(clinical) < 20: clinical = _strip_nonclinical(raw)
    if len(clinical) < 20:
        _COUNTERS["reports_no_clinical_text"] += 1; clinical = str(raw)
    return findings, impression, clinical

print("[cell 2] data structures, anatomy priors, report parser ready.")


## Cell 3 — Vocab mining, cohort filters, Stage-2 query plans

Mines per-disease query phrases, resolves label columns, and builds the **single-label cohort** (`filter_single_label_rows`). `sample_keeping_gt` implements **FIX 11** — GT-box images are forced into the cohort during size-sampling so localization `n` is never sampled away. Stage-2 builds one `QueryPlan` per image and records which classes were queried.

In [ ]:
# =============================================================================
# CELL 3 / 8  —  VOCAB MINING  +  COHORT FILTERS  +  STAGE-2 QUERY-PLAN BUILDER
# =============================================================================

# ==================== SECTION 3.1 — vocab mining ============================
def mine_vocab_from_reports(train_df, top_n=8, min_count=3, max_phrase_len=120):
    if "report_path" not in train_df.columns:
        print("report_path missing - seed vocab only.")
        return {d: {"query_phrases": [f"{d.lower()} on chest x-ray"],
                    "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
                for d in TARGET_CLASSES}
    report_paths = train_df["report_path"].dropna().astype(str).unique().tolist()
    print(f"\n{'='*70}\nVOCAB MINER: {len(report_paths):,} reports\n{'='*70}")
    disease_sentences = {d: [] for d in TARGET_CLASSES}; skipped = 0
    for path in tqdm(report_paths, desc="Mining vocab"):
        raw = _read_report(path)
        if not raw.strip():
            skipped += 1; continue
        _, _, clinical = _clinical_text_from_raw(raw)
        for disease in TARGET_CLASSES:
            for sent_raw in re.split(r"[.\n;]", clinical):
                sent = re.sub(r"\s+", " ", sent_raw.strip())
                kw = next((k for k in SEED_KEYWORDS[disease] if k.lower() in sent.lower()), None)
                if kw and sentence_status(sent, kw) == "positive" and 15 <= len(sent) <= max_phrase_len:
                    disease_sentences[disease].append(sent.lower())
    print(f"Skipped: {skipped:,}")
    mined = {}
    for d in TARGET_CLASSES:
        sents = disease_sentences[d]
        print(f"  {d:<20}: {len(sents):,}")
        if not sents:
            mined[d] = {"query_phrases": [f"{d.lower()} on chest x-ray"],
                        "keywords": SEED_KEYWORDS[d], "default_anatomy": DEFAULT_ANATOMY[d]}
            continue
        ctr = Counter(sents)
        top = [s for s, _ in sorted(
            [(s, c) for s, c in ctr.items() if c >= min_count and len(s) >= 20],
            key=lambda x: (-x[1], -len(x[0])))[:top_n]]
        if len(top) < 2:
            top = [s for s, _ in ctr.most_common(top_n)]
        mined[d] = {"query_phrases": top[:top_n], "keywords": SEED_KEYWORDS[d],
                    "default_anatomy": DEFAULT_ANATOMY[d]}
    return mined

# ==================== SECTION 3.2 — labels + CSV ============================
_LABEL_ALIASES = {
    "Atelectasis": ["Atelectasis"], "Cardiomegaly": ["Cardiomegaly"],
    "Consolidation": ["Consolidation"], "Edema": ["Edema"],
    "Pleural Effusion": ["Pleural Effusion", "Pleural_Effusion", "Effusion"],
}
def _detect_csv_sep(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        line = f.readline()
    return "\t" if "\t" in line and line.count("\t") >= line.count(",") else ","

def read_pairs_csv(path):
    return pd.read_csv(path, sep=_detect_csv_sep(path))

def _find_label_cols(df):
    cols = {c: c for c in df.columns}; lower = {c.lower(): c for c in df.columns}; found = {}
    for cls, aliases in _LABEL_ALIASES.items():
        hit = None
        for a in aliases:
            for cand in (a, a.replace(" ", "_"), a.lower(), a.replace(" ", "_").lower()):
                if cand in cols: hit = cols[cand]; break
                if cand in lower: hit = lower[cand]; break
            if hit: break
        if hit: found[cls] = hit
    return found

def filter_multiclass_rows(df, name):
    """>=1 of 5 target labels positive (old v14 behaviour; harder/different task)."""
    n0 = len(df); lcols = _find_label_cols(df)
    if lcols:
        mask = pd.Series(False, index=df.index)
        for _cls, col in lcols.items():
            v = pd.to_numeric(df[col], errors="coerce"); mask = mask | (v == 1.0)
        out = df[mask].reset_index(drop=True)
        print(f"  {name}: multi-label filter kept {len(out):,}/{n0:,} (label cols: {list(lcols.values())})")
        return out
    aliases = set()
    for al in _LABEL_ALIASES.values(): aliases |= set(al)
    def _has(s): return len(set(str(s).split("|")) & aliases) > 0
    for lc in ["Finding Labels", "finding_labels", "labels", "Finding_Labels", "Labels"]:
        if lc in df.columns:
            out = df[df[lc].apply(_has)].reset_index(drop=True)
            print(f"  {name}: multi-label filter via pipe column '{lc}' kept {len(out):,}/{n0:,}")
            return out
    print(f"  {name}: no label columns found - skipping filter.")
    return df

def filter_single_label_rows(df, name):
    """[v15 FIX 4 / explicit request] Keep only images with EXACTLY ONE of the 5
    target labels positive. This makes each class's negatives true negatives
    instead of other-target positives, so per-class AUROC is not the harder /
    different multi-label task. NOTE: this cohort is still not identical to
    published NIH zero-shot (no No-Finding rows, no multi-label rows) — state it
    in every caption."""
    lcols = _find_label_cols(df)
    if not lcols:
        print(f"  {name}: no label columns found - single-label filter skipped.")
        return df
    n0 = len(df)
    M = np.zeros((len(df), len(lcols)), dtype=float)
    for j, col in enumerate(lcols.values()):
        M[:, j] = (pd.to_numeric(df[col], errors="coerce").values == 1.0).astype(float)
    npos = M.sum(axis=1)
    out = df[npos == 1.0].reset_index(drop=True)
    kept_per = {cls: int((pd.to_numeric(out[col], errors="coerce") == 1.0).sum())
                for cls, col in _find_label_cols(out).items()} if len(out) else {}
    print(f"  {name}: SINGLE-LABEL filter kept {len(out):,}/{n0:,} "
          f"(exactly-1-of-{len(lcols)} positive) | per-class: {kept_per}")
    return out

def apply_cohort_filter(df, name):
    """Dispatch on COHORT_MODE: single_label / multilabel / all."""
    if COHORT_MODE == "single_label":
        return filter_single_label_rows(df, name)
    if COHORT_MODE == "multilabel":
        return filter_multiclass_rows(df, name)
    return df

def _row_basenames(df):
    """Basename of the image index/path for every row (aligned to df.index)."""
    name_col = next((c for c in ["image_name", "Image Index", "image", "filename"]
                     if c in df.columns), None)
    if name_col is not None:
        return df[name_col].astype(str).map(os.path.basename)
    return df.apply(lambda r: os.path.basename(_resolve_image_name(r)), axis=1)

def sample_keeping_gt(df, max_n, gt_names, name, seed=SEED):
    """[v15 FIX 11 — actually implemented, not deferred] Size-sample `df` to
    `max_n` rows while GUARANTEEING every image that has a GT localization box is
    kept. Without this, MAX_VAL_IMAGES on NIH (few GT-box images) can randomly
    sample away most of the localization eval set, leaving evaluate_localization
    with a tiny, uncontrolled n. Returns the sampled frame."""
    if max_n is None or len(df) <= max_n:
        return df.reset_index(drop=True)
    bn = _row_basenames(df)
    must_mask = bn.isin(set(gt_names)) if gt_names else pd.Series(False, index=df.index)
    must = df[must_mask]
    rest = df[~must_mask]
    n_rest = max(0, max_n - len(must))
    if len(rest) > n_rest:
        rest = rest.sample(n_rest, random_state=seed)
    out = pd.concat([must, rest]).reset_index(drop=True)
    if len(must):
        print(f"  {name}: kept {len(must):,} GT-box image(s) + {len(rest):,} sampled "
              f"= {len(out):,} (target {max_n:,}) [FIX 11]")
    return out

def _get_col(row, cands, default=""):
    for c in cands:
        if c in row.index: return row[c]
    return default

def _get_label_val(row, label):
    for c in [label, label.replace(" ", "_"), label.lower(), label.replace(" ", "_").lower()]:
        if c in row.index: return row[c]
    return None

def _resolve_image_name(row):
    name = str(_get_col(row, ["image_name", "Image Index", "image", "filename", "study_id"], ""))
    if not name or name.lower() == "nan":
        p = str(_get_col(row, ["image_path", "path"], ""))
        name = os.path.basename(p) if p else ""
    return name

# [HYBRID / next-round] canonical metadata columns we want to keep for EVERY image
# (sex/age/view/patient/etc.). We capture these named fields under normalized keys
# AND dump every remaining raw column verbatim, so nothing the CSV carries is lost.
_META_FIELD_MAP = {
    "patient_age":  ["Patient Age", "Patient_Age", "age", "Age"],
    "patient_sex":  ["Patient Gender", "Patient_Gender", "sex", "gender", "Sex", "Gender"],
    "view_position":["View Position", "View_Position", "view_position", "view"],
    "patient_id":   ["Patient ID", "Patient_ID", "patient_id", "subject_id"],
    "follow_up":    ["Follow-up #", "Follow_up", "follow_up"],
    "orig_width":   ["OriginalImage[Width", "OriginalImageWidth", "width"],
    "orig_height":  ["Height]", "OriginalImageHeight", "height"],
    "finding_labels":["Finding Labels", "Finding_Labels", "finding_labels", "labels", "Labels"],
    "study_id":     ["study_id", "Study ID", "StudyID"],
}

def _extract_row_meta(row):
    """Return a dict with normalized key metadata fields PLUS every raw column of
    the row (JSON-friendly scalars), so the saved record misses no CSV column."""
    meta = {}
    for key, cands in _META_FIELD_MAP.items():
        v = _get_col(row, cands, None)
        if v is not None and not (isinstance(v, float) and pd.isna(v)):
            meta[key] = v.item() if hasattr(v, "item") else v
    raw = {}
    try:
        for col in row.index:
            v = row[col]
            if isinstance(v, float) and pd.isna(v):
                raw[str(col)] = None
            else:
                raw[str(col)] = v.item() if hasattr(v, "item") else v
    except Exception:
        pass
    meta["raw_columns"] = raw
    return meta

def _extract_row_labels(row):
    """Per-class GT label vector (0/1/NaN) for the 5 targets, from label columns."""
    out = {}
    for c in TARGET_CLASSES:
        v = _get_label_val(row, c)
        try:
            fv = float(v) if (v is not None and not pd.isna(v)) else float("nan")
            out[c] = fv if fv in (0.0, 1.0) else float("nan")
        except Exception:
            out[c] = float("nan")
    return out

# ==================== SECTION 3.3 — query plans ============================
def _rule_based_items(clinical, vocab, view, positive_labels, use_label_backup):
    disease_hits = {}
    for disease, entry in vocab.items():
        keywords = entry.get("keywords", SEED_KEYWORDS.get(disease, []))
        disease_hits[disease] = extract_keyword_sentences(clinical, keywords)
    suppressed = []
    for disease, hits in disease_hits.items():
        for sent, status in hits:
            if status == "negated":
                for region in extract_anatomy_mentions(sent):
                    if region not in suppressed: suppressed.append(region)
    report_detected = {d for d, hits in disease_hits.items()
                       if any(s == "positive" for _, s in hits)}
    candidates = set(report_detected)
    if use_label_backup: candidates = candidates | set(positive_labels)
    items = []
    for disease in sorted(candidates):
        if disease not in vocab: continue
        entry = vocab[disease]; hits = disease_hits.get(disease, [])
        pos_hits = [s for s, st in hits if st == "positive"]
        unc_hits = [s for s, st in hits if st == "uncertain"]
        if pos_hits:
            snippet = max(pos_hits, key=len)
            confidence = 1.00 if (use_label_backup and disease in positive_labels) else 0.80
            negated = False
        elif unc_hits and not DISCARD_UNCERTAIN:
            snippet = max(unc_hits, key=len); confidence = 0.55; negated = False
        elif disease in positive_labels and use_label_backup:
            snippet = entry.get("query_phrases", [f"{disease.lower()} on chest x-ray"])[0]
            confidence = 0.70; negated = False
        else:
            continue
        anatomy_hits = extract_anatomy_mentions(snippet)
        primary = anatomy_hits[0] if anatomy_hits else entry.get(
            "default_anatomy", DEFAULT_ANATOMY.get(disease, "lungs"))
        cx, cy, sx, sy = get_anatomy_coords(primary, view)
        items.append(PathologyQueryItem(
            pathology=disease, text_snippet=snippet, query_vector=None,
            anatomical_prior=AnatomicalPrior(primary, cx, cy, sx, sy),
            confidence=confidence, negated=negated, source="rule"))
    return items, suppressed

def build_query_plan_for_row(row, vocab, view, use_label_backup, view_from_report=False):
    img_name = _resolve_image_name(row)
    img_path = str(_get_col(row, ["image_path", "path"], ""))
    rpt_path = str(_get_col(row, ["report_path", "report", "text_path"], ""))
    raw = _read_report(rpt_path)
    if view not in ("PA", "AP"):
        view = detect_view_from_report(raw) if view_from_report else "PA"
    findings, impression, clinical = _clinical_text_from_raw(raw)
    plan = QueryPlan(image_name=img_name, image_path=img_path, report_path=rpt_path,
                     view_position=view, findings_text=findings, impression_text=impression)
    pos_labels = set()
    for d in TARGET_CLASSES:
        val = _get_label_val(row, d)
        try:
            if val is not None and not pd.isna(val) and float(val) == 1.0: pos_labels.add(d)
        except Exception:
            pass
    items, suppressed = _rule_based_items(clinical, vocab, view, pos_labels, use_label_backup)
    plan.query_items = items; plan.suppressed_regions = suppressed
    # [v15 FIX 1] record which classes the parser queried (non-negated), so the
    # metrics can restrict AUROC to queried images and score the parser itself.
    plan.queried_classes = sorted({qi.pathology for qi in items
                                   if not qi.negated and qi.pathology in TARGET_CLASSES})
    # [HYBRID / next-round] attach ALL image metadata + the GT label vector so the
    # persisted outputs carry sex/age/view/labels/etc. for the CheXpert round.
    plan.meta = _extract_row_meta(row)
    plan.labels = _extract_row_labels(row)
    return plan

def run_stage2(df, vocab, desc, use_label_backup, view_default="PA", view_from_report=False):
    """[v15 correctness] `df` is already the cohort- and size-sampled frame — no
    internal .iloc[:max_imgs] double-truncation (that both re-truncated an
    already-sampled frame and used an ordered slice inconsistent with the val
    path's random sample). Sampling happens once, in the driver."""
    plans = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        view = str(_get_col(row, ["View_Position", "View Position", "view_position", "view"],
                            view_default)).upper()
        if view not in ("PA", "AP"): view = "AUTO" if view_from_report else "PA"
        plans.append(build_query_plan_for_row(row, vocab, view, use_label_backup,
                                              view_from_report=view_from_report))
    return plans

def split_stats(plans, name):
    ni = sum(len(p.query_items) for p in plans)
    ne = sum(1 for p in plans if p.query_items)
    n_ap = sum(1 for p in plans if p.view_position == "AP")
    print(f"{name:<6}: {len(plans):,} | {ne:,} non-empty | {ni:,} items | "
          f"avg={ni/max(len(plans),1):.2f} | AP-view={n_ap:,}")

print("[cell 3] vocab miner + cohort filters + Stage-2 query-plan builder ready.")


## Cell 4 — CheXzero, binary prompts, GradCAM, scoring

Strict checkpoint loader (**FIX 9** — asserts the missing/unexpected-key budget, documents the `weights_only` risk), the POS/NEG binary prompt matrices, the temperature-calibrated `binary_class_probs_gpu` (**FIX 7**), ViT-GradCAM, and the whole-image global class-prob used by the ensemble (computed from the **native** image).

In [ ]:
# =============================================================================
# CELL 4 / 8  —  CHEXZERO  +  BINARY PROMPTS  +  GRAD-CAM  +  SCORING
# =============================================================================

# ==================== SECTION 4.1 — load CheXzero ==========================
# [v15 FIX 9] The v14 loader printed a warning and proceeded no matter how many
# keys were missing/unexpected, and applied .replace("model.", "") to EVERY key
# (which can silently corrupt/collide names). This loader:
#   * strips only leading "module."/"model." prefixes,
#   * asserts the missing/unexpected key counts are within an explicit budget,
#   * logs the loaded key count,
#   * documents that torch.load(weights_only=False) is an arbitrary-code path
#     — only run it on a checkpoint you trust.
# [v15] Key-mismatch budget. A small non-zero default tolerates benign drift
# (a buffer key, a logit_scale rename) while iterating, instead of hard-failing
# the whole pipeline; TIGHTEN TO 0 for the final reproducible run.
CKPT_MAX_MISSING    = 25
CKPT_MAX_UNEXPECTED = 25

def _load_state_from_ckpt(ckpt_path, device):
    assert os.path.exists(ckpt_path), f"Not found: {ckpt_path}"
    # SECURITY: weights_only=False unpickles arbitrary objects. Trusted ckpt only.
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict):
        for k in ("state_dict", "model", "model_state_dict", "net"):
            if k in state and isinstance(state[k], dict):
                state = state[k]; break
    def _strip(k):
        for pre in ("module.", "model."):
            if k.startswith(pre):
                return k[len(pre):]
        return k
    return {_strip(k): v for k, v in state.items()}

def load_chexzero(ckpt_path, device, strict_budget=True):
    print("\n" + "=" * 70 + "\nLOADING CHEXZERO\n" + "=" * 70)
    model, preprocess = clip.load("ViT-B/32", device=device, jit=False)
    clean = _load_state_from_ckpt(ckpt_path, device)
    miss, unex = model.load_state_dict(clean, strict=False)
    n_loaded = len(clean) - len(unex)
    print(f"Loaded {n_loaded} keys | Missing: {len(miss)} | Unexpected: {len(unex)}")
    if len(miss) > CKPT_MAX_MISSING or len(unex) > CKPT_MAX_UNEXPECTED:
        msg = (f"Checkpoint key mismatch beyond budget "
               f"(missing={len(miss)}>{CKPT_MAX_MISSING} or unexpected={len(unex)}>"
               f"{CKPT_MAX_UNEXPECTED}). First missing: {list(miss)[:8]}")
        if strict_budget:
            raise RuntimeError(msg)
        print("WARNING (fail-open disabled by default): " + msg)
    model.eval().to(device)
    for p in model.parameters(): p.requires_grad_(False)
    # last block stays trainable ONLY so GradCAM can backprop through it.
    for p in model.visual.transformer.resblocks[-1].parameters(): p.requires_grad_(True)
    return model, preprocess

# ==================== SECTION 4.2 — text matrices ==========================
@torch.no_grad()
def _encode_prompt_group(model, prompts, device):
    """Mean L2-normalised text embedding for a list of prompts -> [D]."""
    toks = clip.tokenize(prompts, truncate=True).to(device)
    e = model.encode_text(toks).float()
    e = e / e.norm(dim=-1, keepdim=True)
    m = e.mean(0); m = m / m.norm()
    return m

@torch.no_grad()
def build_class_text_matrix(model, device):
    """5-way positive-only CTM. Retained for the GradCAM class objective only."""
    rows = [_encode_prompt_group(model, ZS_PROMPTS[c], device) for c in TARGET_CLASSES]
    ctm = torch.stack(rows).float()
    ls = model.logit_scale.exp().detach().float()
    print(f"CTM: {tuple(ctm.shape)} | logit_scale={ls.item():.2f}")
    return ctm, ls

@torch.no_grad()
def build_binary_prompt_matrices(model, device):
    """[v14 FIX A] Per-class POS and NEG prompt matrices -> (pos[C,D], neg[C,D])."""
    pos = torch.stack([_encode_prompt_group(model, ZS_PROMPTS[c], device) for c in TARGET_CLASSES])
    neg = torch.stack([_encode_prompt_group(model, ZS_PROMPTS_NEG[c], device) for c in TARGET_CLASSES])
    print(f"Binary prompt matrices: pos{tuple(pos.shape)} neg{tuple(neg.shape)}")
    return pos.float(), neg.float()

# ==================== SECTION 4.3 — fill query vectors =====================
@torch.no_grad()
def fill_query_vectors(plans, model, device, ctm=None, bs=None):
    if bs is None: bs = TEXT_ENCODE_BATCH_SZ
    refs, texts = [], []
    for p in plans:
        for qi in p.query_items:
            refs.append(qi); texts.append(str(qi.text_snippet).strip() or qi.pathology)
    if not texts: return plans
    model.eval(); all_e = []
    for i in tqdm(range(0, len(texts), bs), desc="Text encode"):
        toks = clip.tokenize(texts[i:i + bs], truncate=True).to(device)
        e = model.encode_text(toks).float(); e = e / e.norm(dim=-1, keepdim=True)
        all_e.append(e.detach().cpu())
    all_e = torch.cat(all_e)
    ctm_cpu = ctm.detach().cpu().float() if ctm is not None else None
    a = float(CAUSAL_QUERY_CLASS_BLEND); n_blend = 0
    # NOTE [FIX 8]: this class-prompt blend is what correlates semantic_score
    # with zeroshot_prob. Ablation arm J:decorrelate sets the blend to 0.
    for qi, e in zip(refs, all_e):
        e = e.float()
        if ctm_cpu is not None and a > 0.0 and qi.pathology in TARGET_CLASSES:
            cprompt = ctm_cpu[TARGET_CLASSES.index(qi.pathology)]
            blended = (1.0 - a) * e + a * cprompt; nrm = blended.norm()
            if nrm > 0: e = (blended / nrm).float(); n_blend += 1
        qi.query_vector = e.float()
    if ctm_cpu is not None and a > 0.0:
        print(f"  causal query blend: {n_blend:,} query vectors enriched (weight={a:.2f})")
    return plans

# ==================== SECTION 4.4 — ViT Grad-CAM ==========================
class ViTGradCAM:
    def __init__(self, model, preprocess, device, ctm, logit_scale):
        self.model = model; self.preprocess = preprocess; self.device = device
        self.ctm = ctm; self.logit_scale = logit_scale
        self._acts = self._grads = None; self._active = False
        blk = model.visual.transformer.resblocks[-1]
        self._fh = blk.ln_1.register_forward_hook(self._save_acts)
        self._bh = blk.ln_1.register_full_backward_hook(self._save_grads)
    def _save_acts(self, _m, _i, o):
        if self._active: self._acts = o
    def _save_grads(self, _m, _gi, go):
        if self._active: self._grads = go[0]
    def _patch_tokens(self, x):
        if x is None or x.dim() != 3: return None
        if x.shape[1] == 1:   return x[1:, 0, :]
        elif x.shape[0] == 1: return x[0, 1:, :]
        return None
    def _cam(self):
        a = self._patch_tokens(self._acts); g = self._patch_tokens(self._grads)
        if a is None or g is None: return None
        a = a.float(); g = g.float()
        c = F.relu((a * g.mean(0).unsqueeze(0)).sum(-1))
        if c.max() <= 1e-8: return None
        c = c / c.max(); s = int(math.sqrt(c.numel()))
        if s * s != c.numel(): return None
        # [v15 FIX 5 note] s is the native token grid (7 for ViT-B/32@224). The
        # interpolation below is cosmetic upsampling; downstream "persistence" is
        # over these s*s real values, hence "smoothed-CAM component selection".
        return F.interpolate(c.reshape(s, s).unsqueeze(0).unsqueeze(0),
                             size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear",
                             align_corners=False).squeeze().detach()
    @torch.enable_grad()
    def _forward_backward(self, pil, backward_fn):
        self._acts = self._grads = None; self._active = True
        t = self.preprocess(pil).unsqueeze(0).to(self.device)
        self.model.zero_grad(set_to_none=True)
        f = self.model.encode_image(t).float(); f = f / f.norm(dim=-1, keepdim=True)
        backward_fn(f).backward(); self._active = False
        return self._cam()
    def compute_combined(self, pil, qv, tidx):
        qv_d = qv.to(self.device).float(); qv_d = qv_d / qv_d.norm()
        sc = self._forward_backward(pil, lambda f: (f * qv_d.unsqueeze(0)).sum(-1).squeeze())
        cc = self._forward_backward(pil, lambda f: (
            self.logit_scale.float() * (f @ self.ctm.float().T)).softmax(-1)[0, tidx])
        vs = sc is not None and sc.max().item() > 1e-6
        vc = cc is not None and cc.max().item() > 1e-6
        if vs and vc:
            r = GRADCAM_SNIPPET_W * (sc / sc.max()) + GRADCAM_CLASS_W * (cc / cc.max())
            return r / r.max() if r.max().item() >= 0.05 else None
        return (sc / sc.max()) if vs else ((cc / cc.max()) if vc else None)
    def remove_hooks(self):
        self._fh.remove(); self._bh.remove()

# ==================== SECTION 4.5 — scoring ==============================
@torch.no_grad()
def zeroshot_probs_gpu(embs, ctm, ls):
    """Legacy 5-way softmax across the target classes. Reachable ONLY via the
    H:5way_softmax ablation arm; couples co-occurring findings. Not for scoring."""
    return (ls.float() * (embs.float() @ ctm.float().T)).softmax(-1)

def _prob_temperature(ls):
    """[v15 FIX 7] Effective temperature for the binary softmax. Uses
    BINARY_PROB_TEMP (a small, held-out-tunable scale) when set, else the raw
    logit_scale (~100), which saturates probabilities."""
    if BINARY_PROB_TEMP is not None:
        return torch.tensor(float(BINARY_PROB_TEMP), dtype=torch.float32, device=ls.device)
    return ls.float()

@torch.no_grad()
def binary_class_probs_gpu(embs, pos_mat, neg_mat, ls):
    """[v14 FIX A + v15 FIX 7] Independent per-class P(present) -> [N, C].
    For each class c: softmax( temp * [sim(emb,pos_c), sim(emb,neg_c)] )[...,0].
    Decoupled across classes; temperature de-saturates the probabilities."""
    e = embs.float()
    sp = e @ pos_mat.float().T          # [N, C]
    sn = e @ neg_mat.float().T          # [N, C]
    temp = _prob_temperature(ls)
    z = torch.stack([sp, sn], dim=-1) * temp          # [N, C, 2]
    return z.softmax(dim=-1)[..., 0]                  # [N, C] P(present)

@torch.no_grad()
def class_probs_gpu(embs, ctm, ls, pos_mat=None, neg_mat=None):
    """Dispatcher used everywhere downstream. Binary pairs when available."""
    if USE_BINARY_PROMPTS and pos_mat is not None and neg_mat is not None:
        return binary_class_probs_gpu(embs, pos_mat, neg_mat, ls)
    return zeroshot_probs_gpu(embs, ctm, ls)

@torch.no_grad()
def global_image_class_probs(pil, model, preprocess, device, ctm, ls, pos_mat=None, neg_mat=None):
    """[v14 FIX B + v15 correctness] Whole-image CheXzero class probs -> [C].
    Computed from the NATIVE-resolution PIL passed in (the v14 code re-resized an
    already-512 image, double-resampling the 'global baseline'). Callers pass the
    native image."""
    t = preprocess(pil).unsqueeze(0).to(device)
    with torch.amp.autocast("cuda", enabled=USE_CUDA):
        f = model.encode_image(t).float(); f = f / f.norm(dim=-1, keepdim=True)
    return class_probs_gpu(f, ctm, ls, pos_mat, neg_mat)[0]   # [C]

print("[cell 4] CheXzero strict-loader, calibrated binary prompts, GradCAM, scoring ready.")


## Cell 4b — RadJEPA: frozen SSL backbone + trained linear probe + probe-driven Grad-CAM (the hybrid "where")


In [ ]:
# =============================================================================
# CELL 4b / 9  —  RadJEPA  (frozen SSL backbone + trained linear probe + GradCAM)
# =============================================================================
# The "WHERE" half of the hybrid. RadJEPA is a self-supervised (I-JEPA / DINOv2
# style) chest-x-ray ViT. We:
#   1. load it and FREEZE the whole backbone,
#   2. train ONLY a linear probe (one nn.Linear) on the NIH 5-target labels,
#   3. take a probe-driven Grad-CAM => a sharp, anatomically-precise, largely
#      domain-invariant "where" map.
# CheXzero (the "WHAT") is never touched here and stays 100% frozen / zero-shot.
#
# ROBUSTNESS: everything here is optional. If timm is missing, or the RadJEPA
# checkpoint cannot be found/loaded within budget and no fallback backbone is
# available, RADJEPA_AVAILABLE stays False, ENABLE_RADJEPA_HYBRID is forced off,
# and the pipeline runs EXACTLY like v15 (CheXzero smoothed-CAM persistence as the
# "where"). Nothing downstream hard-depends on RadJEPA.

try:
    import timm
    _HAS_TIMM = True
except Exception as _e:
    _HAS_TIMM = False
    print(f"[radjepa] timm unavailable ({_e}); hybrid will fall back to CheXzero-only.")

# ---- imagenet-style preprocess for the RadJEPA ViT (grayscale CXR loaded RGB) ----
_RADJEPA_MEAN = (0.485, 0.456, 0.406)
_RADJEPA_STD  = (0.229, 0.224, 0.225)
def _build_radjepa_preprocess(img_size):
    return T.Compose([
        T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(_RADJEPA_MEAN, _RADJEPA_STD),
    ])

def _strip_ssl_prefixes(state):
    """I-JEPA / DINO / MAE checkpoints wrap the encoder under various prefixes.
    Prefer the (target_)encoder sub-dict when present, then strip module/backbone."""
    if isinstance(state, dict):
        for k in ("target_encoder", "encoder", "teacher", "student", "backbone",
                  "model", "state_dict", "model_state_dict", "net"):
            if k in state and isinstance(state[k], dict):
                state = state[k]; break
    clean = {}
    for k, v in state.items():
        nk = k
        for pre in ("module.", "backbone.", "encoder.", "target_encoder.",
                    "context_encoder.", "base_encoder.", "model."):
            if nk.startswith(pre):
                nk = nk[len(pre):]
        clean[nk] = v
    return clean

def load_radjepa(ckpt_path, device, arch=None, img_size=None):
    """Return (backbone, preprocess, embed_dim, num_prefix_tokens) or a disabled
    tuple (None, None, 0, 0). Sets the global RADJEPA_AVAILABLE flag."""
    global RADJEPA_AVAILABLE, RADJEPA_EMBED_DIM, ENABLE_RADJEPA_HYBRID
    arch = arch or RADJEPA_ARCH; img_size = img_size or RADJEPA_IMG_SIZE
    if not _HAS_TIMM:
        RADJEPA_AVAILABLE = False; ENABLE_RADJEPA_HYBRID = False
        return None, None, 0, 0
    print("\n" + "=" * 70 + "\nLOADING RadJEPA (self-supervised 'where' backbone)\n" + "=" * 70)
    backbone = timm.create_model(arch, pretrained=False, num_classes=0, img_size=img_size)
    loaded_ok = False
    if ckpt_path and os.path.exists(ckpt_path):
        try:
            # SECURITY: weights_only=False unpickles arbitrary objects; trusted ckpt only.
            raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            clean = _strip_ssl_prefixes(raw)
            miss, unex = backbone.load_state_dict(clean, strict=False)
            n_loaded = len(clean) - len(unex)
            print(f"  RadJEPA ckpt: loaded {n_loaded} keys | missing {len(miss)} | unexpected {len(unex)}")
            if len(miss) <= RADJEPA_CKPT_MAX_MISSING and n_loaded >= 20:
                loaded_ok = True
            else:
                print(f"  WARNING: key mismatch beyond budget "
                      f"(missing={len(miss)}>{RADJEPA_CKPT_MAX_MISSING}); trying timm-pretrained fallback.")
        except Exception as e:
            print(f"  RadJEPA ckpt load failed ({e}); trying timm-pretrained fallback.")
    else:
        print(f"  RadJEPA ckpt not found at {ckpt_path}; trying timm-pretrained fallback.")
    if not loaded_ok:
        try:
            backbone = timm.create_model(arch, pretrained=True, num_classes=0, img_size=img_size)
            loaded_ok = True
            print("  Using timm-pretrained ImageNet ViT as the RadJEPA fallback backbone "
                  "(SSL ckpt preferred; supply RADJEPA_CKPT for the real weights).")
        except Exception as e:
            print(f"  timm-pretrained fallback unavailable ({e}).")
    if not loaded_ok:
        RADJEPA_AVAILABLE = False; ENABLE_RADJEPA_HYBRID = False
        print("  RadJEPA DISABLED — hybrid falls back to CheXzero smoothed-CAM (v15 behaviour).")
        return None, None, 0, 0
    backbone.eval().to(device)
    for p in backbone.parameters():
        p.requires_grad_(False)
    # The backbone is FROZEN for training (only the linear probe is ever optimized).
    # We re-enable autograd on the LAST transformer block ONLY so the probe-driven
    # Grad-CAM can backprop to the patch tokens — these params are never handed to
    # an optimizer, so the backbone weights never change (same trick as ViTGradCAM).
    try:
        for p in backbone.blocks[-1].parameters():
            p.requires_grad_(True)
    except Exception as e:
        print(f"  [radjepa] could not enable last-block grad for CAM ({e}).")
    embed_dim = int(getattr(backbone, "num_features", RADJEPA_EMBED_DIM))
    n_prefix = int(getattr(backbone, "num_prefix_tokens", 1))
    RADJEPA_EMBED_DIM = embed_dim; RADJEPA_AVAILABLE = True
    preprocess = _build_radjepa_preprocess(img_size)
    print(f"  RadJEPA ready | embed_dim={embed_dim} | prefix_tokens={n_prefix} | img={img_size}")
    return backbone, preprocess, embed_dim, n_prefix


class RadJEPALinearProbe(nn.Module):
    """One trainable linear layer on top of the FROZEN RadJEPA pooled features."""
    def __init__(self, embed_dim, n_classes=N_CLASSES):
        super().__init__()
        self.fc = nn.Linear(embed_dim, n_classes)
    def forward(self, pooled):
        return self.fc(pooled)


@torch.no_grad()
def _radjepa_pool_tokens(backbone, x, n_prefix):
    """forward_features -> pooled feature [B, D] using RADJEPA_FEATURE_POOL."""
    feats = backbone.forward_features(x)          # [B, N(+prefix), D] or [B, D]
    if feats.dim() == 2:
        return feats
    if RADJEPA_FEATURE_POOL == "cls" and n_prefix >= 1:
        return feats[:, 0]
    return feats[:, n_prefix:, :].mean(dim=1)


def train_radjepa_probe(backbone, preprocess, n_prefix, train_df, device,
                        max_images=None, epochs=None, save_path=None):
    """Freeze backbone, cache pooled features for a sample of NIH train images,
    then train ONLY the linear probe (multi-label BCE, NaN-masked). Returns the
    trained probe (or None if RadJEPA is unavailable / no usable data)."""
    if not RADJEPA_AVAILABLE or backbone is None:
        return None
    max_images = max_images or RADJEPA_PROBE_MAX_IMAGES
    epochs = epochs or RADJEPA_PROBE_EPOCHS
    probe = RadJEPALinearProbe(RADJEPA_EMBED_DIM, N_CLASSES).to(device)
    # resume a previously trained probe if present
    if save_path and os.path.exists(save_path):
        try:
            probe.load_state_dict(torch.load(save_path, map_location=device))
            probe.eval()
            print(f"[radjepa-probe] loaded existing probe from {save_path}")
            return probe
        except Exception as e:
            print(f"[radjepa-probe] could not reuse {save_path} ({e}); retraining.")

    df = train_df
    if max_images is not None and len(df) > max_images:
        df = df.sample(max_images, random_state=SEED).reset_index(drop=True)
    print(f"\n[radjepa-probe] caching frozen features for {len(df):,} train images "
          f"(pool={RADJEPA_FEATURE_POOL}) ...")
    feats_list, lab_list = [], []
    batch_imgs, batch_labs = [], []
    def _flush():
        if not batch_imgs: return
        x = torch.stack(batch_imgs).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_CUDA):
            f = _radjepa_pool_tokens(backbone, x, n_prefix).float()
        feats_list.append(f.detach().cpu())
        lab_list.append(torch.stack(batch_labs))
        batch_imgs.clear(); batch_labs.clear()
    for _, row in tqdm(df.iterrows(), total=len(df), desc="RadJEPA feature cache"):
        ipath = str(_get_col(row, ["image_path", "path"], ""))
        if not ipath or not os.path.isfile(ipath):
            continue
        try:
            pil = Image.open(ipath).convert("RGB")
        except Exception:
            continue
        labs = _extract_row_labels(row)
        lab_vec = torch.tensor([labs.get(c, float("nan")) for c in TARGET_CLASSES], dtype=torch.float32)
        batch_imgs.append(preprocess(pil)); batch_labs.append(lab_vec)
        if len(batch_imgs) >= RADJEPA_PROBE_BATCH:
            _flush()
    _flush()
    if not feats_list:
        print("[radjepa-probe] no usable training images; probe skipped (hybrid uses CAM magnitude).")
        return probe
    X = torch.cat(feats_list); Y = torch.cat(lab_list)
    mask = ~torch.isnan(Y)
    Yf = torch.nan_to_num(Y, nan=0.0)
    pos = ((Yf == 1) & mask).sum(0).clamp(min=1).float()
    neg = ((Yf == 0) & mask).sum(0).clamp(min=1).float()
    pos_weight = (neg / pos).to(device)
    print(f"[radjepa-probe] training linear head: X={tuple(X.shape)} "
          f"epochs={epochs} lr={RADJEPA_PROBE_LR} (backbone stays FROZEN)")
    opt = torch.optim.AdamW(probe.parameters(), lr=RADJEPA_PROBE_LR, weight_decay=RADJEPA_PROBE_WD)
    lossf = nn.BCEWithLogitsLoss(reduction="none", pos_weight=pos_weight)
    Xd = X.to(device); Yd = Yf.to(device); Md = mask.float().to(device)
    n = X.shape[0]; probe.train()
    for ep in range(epochs):
        perm = torch.randperm(n, device=device); tot = 0.0; nb = 0
        for i in range(0, n, RADJEPA_PROBE_BATCH):
            idx = perm[i:i + RADJEPA_PROBE_BATCH]
            logits = probe(Xd[idx])
            l = lossf(logits, Yd[idx]) * Md[idx]
            loss = l.sum() / Md[idx].sum().clamp(min=1)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += float(loss.item()); nb += 1
        print(f"  epoch {ep+1}/{epochs}  bce={tot/max(nb,1):.4f}")
    probe.eval()
    if save_path:
        try:
            torch.save(probe.state_dict(), save_path)
            print(f"[radjepa-probe] saved -> {save_path}")
        except Exception as e:
            print(f"[radjepa-probe] save failed ({e})")
    return probe


class RadJEPAGradCAM:
    """Probe-driven Grad-CAM over the RadJEPA patch tokens => sharp 'where' map,
    upsampled to IMAGE_SIZE and normalized to [0,1]. Mirrors the CheXzero
    ViTGradCAM hook pattern but backprops the linear-probe class logit."""
    def __init__(self, backbone, preprocess, probe, device, n_prefix):
        self.backbone = backbone; self.preprocess = preprocess
        self.probe = probe; self.device = device; self.n_prefix = n_prefix
        self._acts = self._grads = None; self._active = False
        self._fh = self._bh = None; self._hooked = False
        self._register()
    def _register(self):
        if self.backbone is None or self._hooked:
            return
        blk = self.backbone.blocks[-1]
        self._fh = blk.norm1.register_forward_hook(self._save_acts)
        self._bh = blk.norm1.register_full_backward_hook(self._save_grads)
        self._hooked = True
    def _save_acts(self, _m, _i, o):
        if self._active: self._acts = o
    def _save_grads(self, _m, _gi, go):
        if self._active: self._grads = go[0]
    def _patch_grid(self, x):
        # x: [B, N(+prefix), D] -> patch tokens [Npatch, D]
        if x is None or x.dim() != 3: return None
        t = x[0, self.n_prefix:, :]
        return t
    @torch.enable_grad()
    def compute(self, pil, class_idx):
        """Return a [IMAGE_SIZE, IMAGE_SIZE] normalized CAM tensor on device, or None."""
        if self.backbone is None or self.probe is None:
            return None
        self._register()   # re-arm if hooks were removed (e.g. reused across cells)
        self._acts = self._grads = None; self._active = True
        try:
            x = self.preprocess(pil).unsqueeze(0).to(self.device)
            self.backbone.zero_grad(set_to_none=True); self.probe.zero_grad(set_to_none=True)
            feats = self.backbone.forward_features(x)
            if feats.dim() == 2:
                pooled = feats
            elif RADJEPA_FEATURE_POOL == "cls" and self.n_prefix >= 1:
                pooled = feats[:, 0]
            else:
                pooled = feats[:, self.n_prefix:, :].mean(dim=1)
            logits = self.probe(pooled.float())
            score = logits[0, class_idx]
            score.backward()
        except Exception:
            self._active = False; return None
        self._active = False
        a = self._patch_grid(self._acts); g = self._patch_grid(self._grads)
        if a is None or g is None:
            return None
        a = a.float(); g = g.float()
        cam = F.relu((a * g.mean(0, keepdim=True)).sum(-1))          # [Npatch]
        if cam.max() <= 1e-8:
            if RADJEPA_CAM_ABS:
                cam = (a * g).abs().sum(-1)                          # class-agnostic magnitude
            if cam.max() <= 1e-8:
                return None
        cam = cam / cam.max()
        s = int(math.sqrt(cam.numel()))
        if s * s != cam.numel():
            return None
        cam2d = F.interpolate(cam.reshape(s, s).unsqueeze(0).unsqueeze(0),
                              size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear",
                              align_corners=False).squeeze().detach()
        mx = cam2d.max()
        return (cam2d / mx) if mx > 0 else None
    def remove_hooks(self):
        if self._fh is not None: self._fh.remove(); self._fh = None
        if self._bh is not None: self._bh.remove(); self._bh = None
        self._hooked = False

print("[cell 4b] RadJEPA loader + linear-probe trainer + probe-driven GradCAM ready "
      f"(timm={_HAS_TIMM}).")


## Cell 5 — Patch utilities & the persistence mask

Box geometry, patch encoding, and the **persistence causal mask**. This cell carries the **critical fix**: the level-set sweep now stores each component's `death` level and reconstructs the mask *just above death* (near the component's true extent) instead of near `birth` — the old code collapsed the mask to a near-peak speck. Mask areas are instrumented in `_PERSIST_AREAS`.

In [ ]:
# =============================================================================
# CELL 5 / 8  —  PATCH UTILITIES (geometry, encode, score, classify, persistence)
# =============================================================================

# ==================== SECTION 5.1 — box geometry ==========================
def _clamp_box_square(x1, y1, side, W=IMAGE_SIZE, H=IMAGE_SIZE, min_side=MIN_BOX_SIZE):
    side = int(round(side))
    if side < min_side: side = min_side
    if side > min(W, H): side = min(W, H)
    x1 = int(max(0, min(int(round(x1)), W - side)))
    y1 = int(max(0, min(int(round(y1)), H - side)))
    x2, y2 = x1 + side, y1 + side
    if x2 <= x1 or y2 <= y1: return None
    return (x1, y1, x2, y2)

def extract_candidate_boxes_gpu(hmap, scale, stride, thresh):
    p = F.avg_pool2d(hmap.unsqueeze(0).unsqueeze(0).float(),
                     kernel_size=scale, stride=stride, padding=0).squeeze()
    v = (p >= thresh).nonzero(as_tuple=False)
    if v.shape[0] == 0: return []
    y1s = (v[:, 0] * stride).cpu().tolist(); x1s = (v[:, 1] * stride).cpu().tolist()
    out = []
    for x, y in zip(x1s, y1s):
        b = _clamp_box_square(x, y, scale)
        if b is not None: out.append(b)
    return out

def deduplicate_boxes(boxes):
    seen, u = set(), []
    for b in boxes:
        k = tuple(map(int, b))
        if k not in seen: seen.add(k); u.append(b)
    return u

def keep_top_boxes_gpu(boxes, hmap, mx):
    """[v15 correctness] Keep the mx highest-scoring boxes. The v14 version
    (a) assumed all boxes share one scale (sc = boxes[0][2]-boxes[0][0]) and
    (b) sampled the pooled map at the box's TOP-LEFT corner clamped by min(),
    silently mis-scoring edge boxes. This version scores each box by the MEAN of
    the heatmap inside the actual box rectangle, which is scale-agnostic and
    correct at the borders."""
    if len(boxes) <= mx: return boxes
    hm = hmap.float()
    H, W = hm.shape[-2], hm.shape[-1]
    scores = []
    for (x1, y1, x2, y2) in boxes:
        xa, ya = max(0, int(x1)), max(0, int(y1))
        xb, yb = min(W, int(x2)), min(H, int(y2))
        if xb <= xa or yb <= ya:
            scores.append(0.0)
        else:
            scores.append(float(hm[ya:yb, xa:xb].mean().item()))
    idx = torch.tensor(scores).topk(mx).indices.tolist()
    return [boxes[i] for i in sorted(idx)]

# ==================== SECTION 5.2 — score + encode ========================
# [v15 factorial ablation] which of the 3 score components are active. The full
# 2^3-1 = 7 non-empty subsets are ablated by arm group M (sem/gc/prob only, each
# pair, and all three) — a complete factorial over the scoring components.
SCORE_ACTIVE = ("sem", "prob", "gc")

def combined_score_gpu(sem, prob, gc):
    """Weighted geometric mean of the ACTIVE score components (the three are
    correlated — see FIX 8). Inactive components are dropped so arm group M can
    test every subset; within a config the ranking is what AUROC compares."""
    eps = 1e-6
    out = torch.ones_like(prob.clamp(eps, 1.0))
    if "sem" in SCORE_ACTIVE:
        out = out * ((sem + 1.0) * 0.5).clamp(eps, 1.0) ** SCORE_W_SEM
    if "prob" in SCORE_ACTIVE:
        out = out * prob.clamp(eps, 1.0) ** SCORE_W_PROB
    if "gc" in SCORE_ACTIVE:
        out = out * gc.clamp(eps, 1.0) ** SCORE_W_GC
    return out.float()

@torch.no_grad()
def encode_patch_batch_gpu(crops, model, preprocess, device, bs=None):
    # [v15 correctness] runtime lookup, not def-time binding, so config changes
    # (e.g. an ablation of PATCH_ENCODE_BATCH_SZ) take effect.
    if bs is None: bs = PATCH_ENCODE_BATCH_SZ
    if not crops: return torch.zeros(0, CLIP_EMBED_DIM, device=device)
    outs = []
    for i in range(0, len(crops), bs):
        batch = torch.stack([preprocess(c) for c in crops[i:i + bs]]).to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_CUDA):
            f = model.encode_image(batch).float(); f = f / f.norm(dim=-1, keepdim=True)
        outs.append(f)
    return torch.cat(outs)

# ==================== SECTION 5.3 — GradCAM in box ========================
def _gc_scores_batch(boxes, gcam_gpu, device):
    if gcam_gpu is None:
        return torch.full((len(boxes),), 0.0, device=device)
    H, W = gcam_gpu.shape[-2], gcam_gpu.shape[-1]
    scores = torch.empty(len(boxes), device=device)
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        xa, ya = max(0, int(x1)), max(0, int(y1))
        xb, yb = min(W, int(x2)), min(H, int(y2))
        if xb <= xa or yb <= ya:
            scores[i] = float(GRADCAM_WEAK_THR); _COUNTERS["gc_degenerate_box"] += 1; continue
        scores[i] = gcam_gpu[ya:yb, xa:xb].mean()
    return torch.nan_to_num(scores, nan=float(GRADCAM_WEAK_THR))

# ==================== SECTION 5.4 — causal containment ====================
def _causal_contains(box, cx_p, cy_p, mask, peaks):
    x1, y1, x2, y2 = box
    center_in = bool(mask[min(cy_p, IMAGE_SIZE - 1), min(cx_p, IMAGE_SIZE - 1)].item() > 0.5)
    if CAUSAL_CONTAIN_MODE == "center":
        return center_in
    peak_in = any(x1 <= px < x2 and y1 <= py < y2 for (py, px) in peaks) if peaks else False
    if CAUSAL_CONTAIN_MODE == "peak":
        return center_in or peak_in
    sub = mask[y1:y2, x1:x2]
    cover = (float(sub.sum().item()) / max(float(mask.sum().item()), 1.0)) if sub.numel() else 0.0
    if CAUSAL_CONTAIN_MODE == "cover":
        return center_in or cover >= CAUSAL_MASK_COVER_FRAC
    return center_in or peak_in or cover >= CAUSAL_MASK_COVER_FRAC

# ==================== SECTION 5.5 — classify patches ======================
def _minmax_np(arr):
    a = np.asarray(arr, dtype=float)
    if a.size == 0: return a
    lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def classify_patches_gpu(boxes, embs_gpu, probs_gpu, gcam_gpu, qi, target_idx,
                         zoom_level, image_name, device,
                         persistent_mask=None, persistent_peaks=None, global_prob=None,
                         radjepa_cam=None):
    """Label each in-anatomy candidate causal / not.
    probs_gpu: [N, C] independent class probs (binary pairs) from class_probs_gpu.
    global_prob: [C] whole-image probs; [v14 FIX B] the stored/used class score is
      ens = GLOBAL_ENSEMBLE_ALPHA*global + (1-alpha)*patch  for the target class.
    radjepa_cam: [HYBRID] normalized (0..1) RadJEPA probe-CAM for this (image,class).
      When the hybrid is active a patch is CAUSAL only if it passes BOTH the RadJEPA
      spatial gate (mean CAM in box >= HYBRID_SPATIAL_THR) AND the CheXzero semantic
      gate (cosine >= sem_thr) — the 'overlap' of the two frozen models."""
    if not boxes or qi.query_vector is None:
        return [], []
    qv = qi.query_vector.to(device).float(); qv = qv / qv.norm()
    sims = (embs_gpu.float() * qv.unsqueeze(0)).sum(-1)
    patch_prob = probs_gpu[:, target_idx]
    # [v14 FIX B] ensemble the patch class prob with the whole-image class prob
    if global_prob is not None and GLOBAL_ENSEMBLE_ALPHA > 0.0:
        g = float(global_prob[target_idx].item())
        cls_prob = GLOBAL_ENSEMBLE_ALPHA * g + (1.0 - GLOBAL_ENSEMBLE_ALPHA) * patch_prob
    else:
        cls_prob = patch_prob
    gc_vals = _gc_scores_batch(boxes, gcam_gpu, device)
    combo = combined_score_gpu(sims, cls_prob, gc_vals)
    sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)
    region = qi.anatomical_prior.region_name if qi.anatomical_prior else "unknown"
    peaks = persistent_peaks or []
    sims_cpu = sims.cpu().numpy(); probs_cpu = cls_prob.cpu().numpy()
    gc_cpu = gc_vals.cpu().numpy(); combo_cpu = combo.cpu().numpy()
    # [HYBRID] RadJEPA spatial score per box (mean of the 0..1 CAM inside the box).
    hybrid_active = bool(ENABLE_RADJEPA_HYBRID and RADJEPA_AVAILABLE and radjepa_cam is not None)
    if radjepa_cam is not None:
        radj_cpu = _gc_scores_batch(boxes, radjepa_cam, device).cpu().numpy()
    else:
        radj_cpu = np.zeros(len(boxes), dtype=float)
    radj_norm = _minmax_np(radj_cpu); sem_norm_arr = _minmax_np(sims_cpu)
    causal, all_scored = [], []
    for i, box in enumerate(boxes):
        sim = float(sims_cpu[i]); prob = float(probs_cpu[i])
        gc_v = float(gc_cpu[i]); cv = float(combo_cpu[i])
        rj = float(radj_cpu[i]); rj_n = float(radj_norm[i]); sm_n = float(sem_norm_arr[i])
        cx_p = (box[0] + box[2]) // 2; cy_p = (box[1] + box[3]) // 2
        has_sem = sim >= sem_thr
        if hybrid_active:
            # spatial gate: box sits in a high RadJEPA-CAM region (0..1 CAM mean).
            spatial_pass = rj >= HYBRID_SPATIAL_THR
            if RADJEPA_KEEP_PERSISTENCE_AND and persistent_mask is not None:
                spatial_pass = spatial_pass and _causal_contains(box, cx_p, cy_p, persistent_mask, peaks)
            is_causal = bool(has_sem and spatial_pass)
            gate = "hybrid"
        elif persistent_mask is not None:
            is_causal = has_sem and _causal_contains(box, cx_p, cy_p, persistent_mask, peaks)
            gate = "persistence"
        else:
            is_causal = has_sem and gc_v >= GRADCAM_WEAK_THR
            gate = "gradcam_weak"
        # [v15 correctness] store fp16 embeddings to halve the growing checkpoint
        # pickle (each doc carries a 512-vector); split_embeddings up-casts.
        _emb = embs_gpu[i].detach().cpu()
        _emb = _emb.half() if CKPT_EMBED_FP16 else _emb.float()
        doc = PatchDocument(
            image_name=image_name, pathology=qi.pathology, scale=int(box[2] - box[0]),
            box=tuple(map(int, box)), visual_embedding=_emb,
            semantic_score=sim, zeroshot_prob=prob, gradcam_score=gc_v, combined_score=cv,
            causal=is_causal, anatomical_region=region, confidence=float(qi.confidence),
            text_snippet=str(qi.text_snippet), zoom_level=zoom_level,
            spurious_source="in_anatomy", selection_source="threshold",
            radjepa_spatial_score=rj, radjepa_spatial_norm=rj_n,
            semantic_norm=sm_n, causal_gate=(gate if is_causal else "none"))
        all_scored.append(doc)
        if is_causal: causal.append(doc)
    return causal, all_scored

def select_spurious_in_anatomy(all_scored, sem_thr, max_keep=None):
    if max_keep is None: max_keep = _spur_in_max_keep()   # [v15] runtime lookup
    noncausal = [d for d in all_scored if not d.causal]
    if not noncausal: return []
    seen, nc = set(), []
    for d in sorted(noncausal, key=lambda x: x.semantic_score):
        if d.box not in seen: seen.add(d.box); nc.append(d)
    gated = [d for d in nc if d.semantic_score < sem_thr and d.gradcam_score >= SPUR_IN_GC_FLOOR]
    if not gated:
        gated = nc[:max_keep]; _COUNTERS["spur_in_safetynet"] += 1
    spin = gated[:max_keep]
    for d in spin:
        d.causal = False; d.spurious_source = "in_anatomy"; d.selection_source = "threshold"
    return spin

# ==================== SECTION 5.6/5.7 — persistence mask ==================
def _persistent_mask_scipy(gcam_np, top_k=None, n_levels=None):
    # [v15 correctness] runtime lookup so ablation of PERSISTENCE_TOP_K / _N_LEVELS
    # applies (v14 bound these at def time). [v15 FIX 5] this is level-set
    # connected-component selection on the (upsampled, low-native-res) CAM — see
    # PERSISTENCE_HONEST_NAME; not topology of the 512px evidence.
    if top_k is None: top_k = PERSISTENCE_TOP_K
    if n_levels is None: n_levels = PERSISTENCE_N_LEVELS
    H, W = gcam_np.shape
    vmax = float(gcam_np.max()); vmin = float(max(gcam_np.min(), 0.0))
    if vmax - vmin < 0.01: return None, []
    thresholds = np.linspace(vmax * 0.98, vmin + 1e-4, n_levels)
    alive = {}; dead = []; nxt = 0
    prev_labels = np.zeros((H, W), dtype=np.int32); prev_map = {}
    for t in thresholds:
        binary = (gcam_np >= t); labels, n = ndimage_label(binary); curr_map = {}
        for c in range(1, n + 1):
            cmask = (labels == c); overlap = prev_labels[cmask]; parents = set()
            for pl in np.unique(overlap):
                if pl > 0 and pl in prev_map: parents.add(prev_map[pl])
            if len(parents) == 0:
                vals = gcam_np.copy(); vals[~cmask] = -1
                peak = np.unravel_index(vals.argmax(), (H, W))
                alive[nxt] = {"birth": float(t), "peak": peak}; curr_map[c] = nxt; nxt += 1
            elif len(parents) == 1:
                curr_map[c] = list(parents)[0]
            else:
                sp = sorted(parents, key=lambda i: alive[i]["birth"], reverse=True); elder = sp[0]
                for younger in sp[1:]:
                    if younger in alive:
                        # store (persistence, birth, DEATH=t, peak) — death is the
                        # merge level; it was thrown away in the buggy version.
                        dead.append((alive[younger]["birth"] - t, alive[younger]["birth"],
                                     float(t), alive[younger]["peak"])); del alive[younger]
                curr_map[c] = elder
        prev_labels = labels; prev_map = curr_map
    for cid, info in alive.items():
        # a surviving (never-merged) component "dies" at vmin.
        dead.append((info["birth"] - vmin, info["birth"], vmin, info["peak"]))
    if not dead: return None, []
    dead.sort(key=lambda x: -x[0])
    level_floor = vmin + PERSISTENCE_MIN_LEVEL_FRAC * (vmax - vmin)
    mask = np.zeros((H, W), dtype=bool); peaks = []
    for k in range(min(top_k, len(dead))):
        pers, birth, death, peak_yx = dead[k]
        if pers < 0.01: continue
        # [FIX] reconstruct just ABOVE death (component near max extent, before it
        # merged) — NOT below birth. Floor guards against a survivor flooding the map.
        thr = death + PERSISTENCE_DEATH_MARGIN * (birth - death)
        thr = max(thr, level_floor)
        candidate = (gcam_np >= thr); labels, n = ndimage_label(candidate)
        if n == 0: continue
        py, px = peak_yx; cid = labels[py, px]
        if cid > 0:
            comp = (labels == cid)
            if comp.sum() >= PERSISTENCE_MIN_AREA:
                mask |= comp; peaks.append((int(py), int(px)))
                _PERSIST_AREAS.append((int(comp.sum()), float(birth - death)))
    if mask.sum() < PERSISTENCE_MIN_AREA: return None, []
    return mask, peaks

def compute_persistent_causal_mask(gcam_gpu):
    # Single definition (v14 cell 8 shadowed this to work around the default-arg
    # binding bug; that shadow is gone — the base function now respects ablations).
    if gcam_gpu is None or not PERSISTENCE_ENABLED: return None, []
    gcam_np = gcam_gpu.cpu().numpy().astype(np.float64)
    if gcam_np.max() - gcam_np.min() < 0.01: return None, []
    mask, peaks = _persistent_mask_scipy(gcam_np)
    if mask is None: return None, []
    if CAUSAL_MASK_DILATE_PX > 0:
        mask = binary_dilation(mask, iterations=int(CAUSAL_MASK_DILATE_PX))
    _COUNTERS["persistence_used"] += 1
    return torch.from_numpy(mask.astype(np.float32)).to(gcam_gpu.device), peaks

# ==================== SECTION 5.8 — outside-anatomy scan ==================
def scan_all_outside_anatomy(pil_img, plan, model, preprocess, device, ctm_gpu, ls_gpu, composite_hmap):
    inv_hmap = (1.0 - composite_hmap.clamp(0, 1)).clamp(0, 1)
    qvs = [qi.query_vector for qi in plan.query_items
           if qi.query_vector is not None and not qi.negated]
    qmat = (torch.stack([q.to(device).float() / q.to(device).float().norm() for q in qvs])
            if qvs else None)
    gray = np.asarray(pil_img.convert("L"), dtype=np.float32) / 255.0
    gH, gW = gray.shape[0], gray.shape[1]
    def _score_boxes(boxes):
        if not boxes: return []
        crops = [pil_img.crop(b) for b in boxes]
        embs = encode_patch_batch_gpu(crops, model, preprocess, device)
        if qmat is not None:
            sims = (embs.float() @ qmat.T).max(dim=1).values
        else:
            sims = torch.zeros(embs.shape[0], device=device)
        sims_cpu = sims.cpu().numpy(); out = []
        for i, b in enumerate(boxes):
            x1, y1, x2, y2 = b
            xa, ya = max(0, int(x1)), max(0, int(y1))
            xb, yb = min(gW, int(x2)), min(gH, int(y2))
            patch = gray[ya:yb, xa:xb]
            if patch.size == 0:
                _COUNTERS["outside_degenerate_box"] += 1; continue
            out.append({"box": b, "emb": embs[i].detach().cpu().float(),
                        "sim": float(sims_cpu[i]), "mean": float(patch.mean()),
                        "std": float(patch.std())})
        return out
    geo_boxes = []
    for sc in OUTSIDE_ANAT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        geo_boxes += extract_candidate_boxes_gpu(inv_hmap, sc, stride, OUTSIDE_ANAT_INV_THRESH)
    geo_boxes = deduplicate_boxes(geo_boxes)[:OUTSIDE_ANAT_MAX_CAND]
    border = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, device=device)
    bw = int(IMAGE_SIZE * ARTIFACT_BORDER_FRAC)
    border[:bw, :] = 1.0; border[-bw:, :] = 1.0; border[:, :bw] = 1.0; border[:, -bw:] = 1.0
    ap = (border * inv_hmap).clamp(0, 1)
    int_boxes = []
    for sc in ARTIFACT_SCALES:
        stride = max(16, STRIDE_BASE * sc // 128)
        int_boxes += extract_candidate_boxes_gpu(ap, sc, stride, 0.35)
    int_boxes = deduplicate_boxes(int_boxes)[:OUTSIDE_ANAT_MAX_CAND]
    geo_scored = _score_boxes(geo_boxes); int_scored = _score_boxes(int_boxes)
    def _select(scored, sim_cap, max_keep, sel_src):
        if not scored: return []
        gated = [s for s in scored if s["sim"] < sim_cap]
        if not gated:
            gated = sorted(scored, key=lambda s: s["sim"])[:max_keep]
            _COUNTERS[f"outside_{sel_src}_safetynet"] += 1
        gated = sorted(gated, key=lambda s: s["std"], reverse=True)[:max_keep]
        docs = []
        for s in gated:
            b = s["box"]
            docs.append(PatchDocument(
                image_name=plan.image_name, pathology="domain_artifact",
                scale=int(b[2] - b[0]), box=tuple(map(int, b)), visual_embedding=s["emb"],
                semantic_score=s["sim"], zeroshot_prob=0.0, gradcam_score=s["std"],
                combined_score=s["mean"], causal=False, anatomical_region="outside_anatomy",
                confidence=1.0, text_snippet=f"{sel_src}_m={s['mean']:.3f}_s={s['std']:.3f}",
                zoom_level=1, spurious_source="outside_anatomy", selection_source=sel_src))
        return docs
    geo_docs = _select(geo_scored, OUTSIDE_ANAT_SIM_CAP, OUTSIDE_ANAT_MAX_KEEP, "geometric_scan")
    int_docs = _select(int_scored, ARTIFACT_SIM_CAP, ARTIFACT_MAX_KEEP, "intensity_scan")
    seen = set(d.box for d in geo_docs); merged = list(geo_docs)
    for d in int_docs:
        if d.box not in seen: seen.add(d.box); merged.append(d)
    if merged:
        _COUNTERS["outside_anatomy_patches"] += len(merged)
        _COUNTERS["images_with_outside_anatomy"] += 1
    return merged[:OUTSIDE_ANAT_MAX_KEEP + ARTIFACT_MAX_KEEP]

print("[cell 5] patch geometry, binary+ensemble classifier, persistence, outside-scan ready.")


## Cell 6 — Stage-3 discovery, metrics & plot suite

Patch discovery (fallback off by default), the resumable Stage-3 driver, and the honest metrics: held-out-split AUROC restricted to queried images (**FIX 1**, with the `zeropad_frac` residual tie-mass reported), the explicit parser-vs-labels confound table, the ungated whole-image baseline, strict causal-only Wilson-CI localization, and the full **plot suite** (ROC/PR/calibration/saturation/per-class-CI, PCA embedding separation, Wilson-vs-n, persistence diagnostic, subgroup fairness).

In [ ]:
# =============================================================================
# CELL 6 / 8  —  STAGE-3 DISCOVERY  +  A* METRICS  +  GRADCAM / BBOX VISUALS
# =============================================================================

# ==================== SECTION 6.1 — discover ==============================
def discover_patches_for_plan(plan, gradcam, model, preprocess, device, ctm, ls,
                              split_name="", pos_mat=None, neg_mat=None, radjepa_gc=None):
    result = ImagePatchResult(image_name=plan.image_name, image_path=plan.image_path,
                              split=split_name, queried_classes=list(plan.queried_classes),
                              report_path=getattr(plan, "report_path", ""),
                              view_position=getattr(plan, "view_position", ""),
                              meta=dict(getattr(plan, "meta", {}) or {}),
                              labels=dict(getattr(plan, "labels", {}) or {}))
    if not plan.image_path or not os.path.isfile(plan.image_path):
        _COUNTERS["images_missing"] += 1; return result
    if not plan.query_items:
        return result
    try:
        pil_native = Image.open(plan.image_path).convert("RGB")               # native res
        pil = pil_native.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)       # working res
    except Exception:
        _COUNTERS["images_failed"] += 1; return result

    cmap = {c.lower(): c for c in TARGET_CLASSES}
    comp_hmap = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, device)
    # [v14 FIX B / v15 correctness] one whole-image class-prob vector from the
    # NATIVE image (resampled once, not twice), ensembled into every patch.
    global_prob = global_image_class_probs(pil_native, model, preprocess, device,
                                           ctm, ls, pos_mat, neg_mat)
    rc, rs = [], []
    max_iter = _max_iter(); conf_thr = _conf_threshold()               # [v15] runtime
    max_cand = _max_candidate_boxes(); max_zoom = _max_zoom_candidate_boxes()
    topk_find = _top_k_per_find()

    for qi in plan.query_items:
        if qi.negated or qi.query_vector is None: continue
        key = qi.pathology.lower().strip()
        if key not in cmap: continue
        qi.pathology = cmap[key]; tidx = TARGET_CLASSES.index(qi.pathology)
        sem_thr = SEMANTIC_THRESHOLD_PER_CLASS.get(qi.pathology, SEMANTIC_THRESHOLD)

        if qi.anatomical_prior:
            p = qi.anatomical_prior
            ihmap = make_gaussian_heatmap_gpu(p.center_x, p.center_y, p.sigma_x, p.sigma_y,
                                              IMAGE_SIZE, IMAGE_SIZE, device)
        else:
            ihmap = comp_hmap.clone()
        mx = ihmap.max()
        if mx > 0: ihmap = ihmap / mx

        gcam = gradcam.compute_combined(pil, qi.query_vector, tidx) if ENABLE_GRADCAM else None
        pmask, ppeaks = compute_persistent_causal_mask(gcam)
        if pmask is None: _COUNTERS["persistence_fallback"] += 1
        # [HYBRID] RadJEPA probe-driven "where" CAM for this class (0..1) — the
        # spatial gate. None => classify falls back to the CLIP persistence gate.
        rcam = None
        if radjepa_gc is not None and ENABLE_RADJEPA_HYBRID and RADJEPA_AVAILABLE:
            rcam = radjepa_gc.compute(pil, tidx)
            _COUNTERS["radjepa_cam_used" if rcam is not None else "radjepa_cam_empty"] += 1

        ac, asc = [], []
        scales = _SCALE_MAP.get(qi.pathology, _patch_scales())   # [v14 FIX C] per-class scales
        for it in range(1, max_iter + 1):
            sp = SPATIAL_THRESH * (0.70 ** (it - 1)); ih = ihmap.clone()
            if it > 1:
                ih = (ih * 1.30).clamp(0, 1); result.refined = True
            for sc in scales:
                stride = max(16, STRIDE_BASE * sc // 128)
                boxes = keep_top_boxes_gpu(
                    deduplicate_boxes(extract_candidate_boxes_gpu(ih, sc, stride, sp)),
                    ih, max_cand)
                if not boxes: continue
                embs = encode_patch_batch_gpu([pil.crop(b) for b in boxes], model, preprocess, device)
                probs = class_probs_gpu(embs, ctm, ls, pos_mat, neg_mat)  # [v14 FIX A]
                c, a = classify_patches_gpu(boxes, embs, probs, gcam, qi, tidx, it,
                                            plan.image_name, device,
                                            persistent_mask=pmask, persistent_peaks=ppeaks,
                                            global_prob=global_prob, radjepa_cam=rcam)
                ac.extend(c); asc.extend(a)
            if len(ac) >= MIN_CAUSAL_PATCHES:
                if max((d.zeroshot_prob for d in ac), default=0) >= conf_thr or it == max_iter:
                    break

        if 0 < len(ac) < MIN_CAUSAL_PATCHES and ENABLE_ZOOM_REFINEMENT:
            top = sorted(ac, key=lambda d: d.combined_score, reverse=True)[0]
            x1, y1, x2, y2 = top.box
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2; w, h = x2 - x1, y2 - y1
            ex1 = max(0, int(cx - w * 0.75)); ey1 = max(0, int(cy - h * 0.75))
            ex2 = min(IMAGE_SIZE, int(cx + w * 0.75)); ey2 = min(IMAGE_SIZE, int(cy + h * 0.75))
            if ex2 > ex1 and ey2 > ey1:
                sxr = (ex2 - ex1) / IMAGE_SIZE; syr = (ey2 - ey1) / IMAGE_SIZE
                iso = 0.5 * (sxr + syr)
                zhmap = torch.ones(IMAGE_SIZE, IMAGE_SIZE, device=device); remap_boxes = []
                for sc in _zoom_scales():
                    stride = max(12, STRIDE_BASE * sc // 128)
                    zb = deduplicate_boxes(extract_candidate_boxes_gpu(zhmap, sc, stride, 0.0))[:max_zoom]
                    for b in zb:
                        zcx = 0.5 * (b[0] + b[2]); zcy = 0.5 * (b[1] + b[3])
                        ocx = ex1 + zcx * sxr; ocy = ey1 + zcy * syr
                        ob = _clamp_box_square(ocx - sc * iso / 2, ocy - sc * iso / 2, sc * iso)
                        if ob is not None: remap_boxes.append(ob)
                remap_boxes = deduplicate_boxes(remap_boxes)
                if remap_boxes:
                    ze = encode_patch_batch_gpu([pil.crop(b) for b in remap_boxes], model, preprocess, device)
                    zp = class_probs_gpu(ze, ctm, ls, pos_mat, neg_mat)
                    zc, za = classify_patches_gpu(remap_boxes, ze, zp, gcam, qi, tidx, 2,
                                                  plan.image_name, device,
                                                  persistent_mask=pmask, persistent_peaks=ppeaks,
                                                  global_prob=global_prob, radjepa_cam=rcam)
                    ac.extend(zc); asc.extend(za)
                result.refined = True

        # [v15 FIX 6] fallback OFF by default (honest "gate may abstain"). When
        # ALLOW_FALLBACK is set (ablation arm G only), top-K patches are relabelled
        # causal, which BYPASSES the causal gate — never the primary configuration.
        if ALLOW_FALLBACK and not ac and asc:
            pool = [d for d in asc if d.semantic_score >= sem_thr * 0.6]
            if pool:
                fb = sorted(pool, key=lambda d: d.combined_score, reverse=True)[:FALLBACK_TOP_K]
                for d in fb: d.causal = True; d.selection_source = "fallback"
                ac.extend(fb); result.used_fallback = True

        spin = select_spurious_in_anatomy(asc, sem_thr, max_keep=_spur_in_max_keep())
        rs.extend(spin)
        rc.extend(sorted(ac, key=lambda d: d.combined_score, reverse=True)[:topk_find])

    rs.extend(scan_all_outside_anatomy(pil, plan, model, preprocess, device, ctm, ls, comp_hmap))
    result.causal_patches = rc; result.spurious_patches = rs
    result.n_iterations = max_iter if result.refined else 1
    return result

# ==================== SECTION 6.2 — run_stage3 driver =====================
def run_stage3(plans, split_name, gradcam, model, preprocess, device, ctm, ls,
               deadline=None, out_dir=".", pos_mat=None, neg_mat=None, radjepa_gc=None):
    ckpt = f"{out_dir}/stage3_{split_name}_ckpt.pkl"; start = 0; results = []
    if os.path.exists(ckpt):
        with open(ckpt, "rb") as f: results = pickle.load(f)
        start = len(results)
        print(f"  Resuming {split_name} from checkpoint: {start:,}/{len(plans):,}")
    # [v15 correctness] recompute BOTH n_empty and n_fail on resume (v14 only
    # recovered n_empty, so a fully-broken run could print "complete").
    n_empty = sum(1 for r in results if not r.causal_patches)
    n_fail = sum(1 for r in results
                 if (not r.causal_patches and not r.spurious_patches and not r.queried_classes))
    t0 = time.time(); last_ckpt_t = t0; stopped_early = False
    pbar = tqdm(range(start, len(plans)), desc=f"Stage 3 - {split_name}", initial=start, total=len(plans))
    for idx in pbar:
        if deadline is not None and time.time() > deadline:
            _atomic_pickle(results, ckpt); stopped_early = True
            print(f"\n  [{split_name}] Budget reached at {idx:,}/{len(plans):,}. Checkpoint saved."); break
        try:
            res = discover_patches_for_plan(plans[idx], gradcam, model, preprocess, device, ctm, ls,
                                            split_name, pos_mat=pos_mat, neg_mat=neg_mat,
                                            radjepa_gc=radjepa_gc)
        except Exception as e:
            print(f"  {plans[idx].image_name}: {e}")
            res = ImagePatchResult(image_name=plans[idx].image_name,
                                   image_path=plans[idx].image_path, split=split_name)
            n_fail += 1
        results.append(res)
        if not res.causal_patches: n_empty += 1
        now = time.time(); done_now = len(results) - start
        if (done_now % CKPT_EVERY_IMAGES == 0) or ((now - last_ckpt_t) > CKPT_EVERY_MIN * 60):
            if USE_CUDA: torch.cuda.empty_cache()
            gc.collect(); _atomic_pickle(results, ckpt); last_ckpt_t = now
            rate = done_now / max(now - t0, 1e-6); remaining = len(plans) - len(results)
            eta_h = remaining / max(rate, 1e-9) / 3600.0
            pbar.set_postfix_str(f"{rate*3600:,.0f} img/h | ETA {eta_h:.2f}h | budget {_budget_left_h(deadline):.2f}h")
    _atomic_pickle(results, ckpt); completed = (len(results) >= len(plans))
    if completed:
        print(f"  {split_name}: {n_fail} failed | {n_empty:,} empty | {len(results)-n_empty:,} w/causal | complete")
        if os.path.exists(ckpt): os.remove(ckpt)
    else:
        print(f"  {split_name}: partial {len(results):,}/{len(plans):,} ({len(results)-n_empty:,} w/causal). Checkpoint kept.")
    return results, completed, stopped_early

# ==================== SECTION 6.3 — split + summarize =====================
# [HYBRID / next-round] every patch field we persist alongside the embedding, so a
# .pt row is self-describing for the CheXpert transfer round (nothing dropped).
_MK = ["image_name", "pathology", "box", "scale", "semantic_score", "zeroshot_prob",
       "gradcam_score", "combined_score", "anatomical_region", "confidence",
       "text_snippet", "zoom_level", "causal", "causal_gate",
       "radjepa_spatial_score", "radjepa_spatial_norm", "semantic_norm",
       "spurious_source", "selection_source"]

def _image_meta_dict(r, sp):
    """[HYBRID / next-round] image-level metadata attached to EVERY patch row:
    image path, report path, split, view, sex, age, patient id, labels, ..."""
    return {"split": sp, "image_path": getattr(r, "image_path", ""),
            "report_path": getattr(r, "report_path", ""),
            "view_position": getattr(r, "view_position", ""),
            "queried_classes": list(getattr(r, "queried_classes", []) or []),
            "labels": dict(getattr(r, "labels", {}) or {}),
            "meta": dict(getattr(r, "meta", {}) or {})}

def _md(p, sp, ek, img_meta=None):
    d = {}
    for k in _MK:
        d[k] = getattr(p, k, None)
    for k in ek:
        d[k] = getattr(p, k, None)
    if img_meta is not None:
        d.update(img_meta)          # image path / sex / age / labels / ...
    else:
        d["split"] = sp
    return d

def split_embeddings_three_ways(rd):
    ce, cm = [], []; se, sm = [], []; oe, om = [], []
    for sp, results in rd.items():
        for r in results:
            im = _image_meta_dict(r, sp)
            for p in r.causal_patches:
                ce.append(p.visual_embedding.float()); cm.append(_md(p, sp, ["selection_source"], im))
            for p in r.spurious_patches:
                if p.spurious_source == "outside_anatomy":
                    oe.append(p.visual_embedding.float()); om.append(_md(p, sp, ["spurious_source", "selection_source"], im))
                else:
                    se.append(p.visual_embedding.float()); sm.append(_md(p, sp, ["spurious_source", "selection_source"], im))
    def pk(e, m):
        return {"embeddings": torch.stack(e) if e else torch.zeros(0, CLIP_EMBED_DIM), "meta": m, "n": len(e)}
    return pk(ce, cm), pk(se, sm), pk(oe, om)

# [HYBRID / next-round] THE deliverable that carries EVERYTHING for retraining on
# full NIH and transferring to CheXpert. One record per image, nothing omitted.
def _patch_to_full_dict(p):
    return {
        "pathology": p.pathology, "box": tuple(map(int, p.box)), "scale": int(p.scale),
        "causal": bool(p.causal), "causal_gate": getattr(p, "causal_gate", "none"),
        "semantic_score": float(p.semantic_score), "semantic_norm": float(getattr(p, "semantic_norm", 0.0)),
        "zeroshot_prob": float(p.zeroshot_prob), "gradcam_score": float(p.gradcam_score),
        "radjepa_spatial_score": float(getattr(p, "radjepa_spatial_score", 0.0)),
        "radjepa_spatial_norm": float(getattr(p, "radjepa_spatial_norm", 0.0)),
        "combined_score": float(p.combined_score), "anatomical_region": p.anatomical_region,
        "confidence": float(p.confidence), "text_snippet": str(p.text_snippet),
        "zoom_level": int(p.zoom_level), "spurious_source": p.spurious_source,
        "selection_source": p.selection_source,
        # fp16 CLIP/CheXzero visual embedding — the vector the next round consumes.
        "visual_embedding": p.visual_embedding.half().cpu(),
    }

def build_full_records(rd):
    """List of per-image dicts: image/report path, split, all metadata (sex, age,
    view, patient id, raw CSV columns), the 5 GT labels, queried classes, and every
    causal / spurious-in / spurious-out patch with all scores + fp16 embedding."""
    records = []
    for sp, results in rd.items():
        for r in results:
            causal = [_patch_to_full_dict(p) for p in r.causal_patches]
            spin   = [_patch_to_full_dict(p) for p in r.spurious_patches if p.spurious_source != "outside_anatomy"]
            spout  = [_patch_to_full_dict(p) for p in r.spurious_patches if p.spurious_source == "outside_anatomy"]
            records.append({
                "image_name": r.image_name, "image_path": getattr(r, "image_path", ""),
                "report_path": getattr(r, "report_path", ""), "split": sp,
                "view_position": getattr(r, "view_position", ""),
                "labels": dict(getattr(r, "labels", {}) or {}),
                "queried_classes": list(getattr(r, "queried_classes", []) or []),
                "metadata": dict(getattr(r, "meta", {}) or {}),   # sex/age/patient id/raw cols
                "n_causal": len(causal), "n_spur_in": len(spin), "n_spur_out": len(spout),
                "causal_patches": causal, "spurious_in_patches": spin,
                "spurious_out_patches": spout,
            })
    return records

def save_full_records(rd, records_path=STAGE3_FULL_RECORDS, bank_path=STAGE3_FULL_PT):
    """Persist the full per-image records (pickle) AND a flattened patch bank (.pt)
    of stacked embeddings + aligned per-patch+image metadata for fast loading."""
    records = build_full_records(rd)
    _atomic_pickle(records, records_path)
    embs, meta = [], []
    for rec in records:
        base = {k: rec[k] for k in ("image_name", "image_path", "report_path", "split",
                                    "view_position", "labels", "queried_classes", "metadata")}
        for bucket in ("causal_patches", "spurious_in_patches", "spurious_out_patches"):
            btype = {"causal_patches": "causal", "spurious_in_patches": "spurious_in",
                     "spurious_out_patches": "spurious_out"}[bucket]
            for pd_ in rec[bucket]:
                embs.append(pd_["visual_embedding"].float())
                row = {kk: vv for kk, vv in pd_.items() if kk != "visual_embedding"}
                row.update(base); row["bucket"] = btype
                meta.append(row)
    bank = {"embeddings": torch.stack(embs) if embs else torch.zeros(0, CLIP_EMBED_DIM),
            "meta": meta, "n": len(embs)}
    torch.save(bank, bank_path)
    n_img = len(records); n_p = len(embs)
    print(f"[full-records] saved {n_img:,} image records -> {records_path}")
    print(f"[full-records] saved flattened patch bank ({n_p:,} patches) -> {bank_path}")
    return records, bank

def summarize_stage3(results, split):
    nc = sum(len(r.causal_patches) for r in results)
    ne = sum(1 for r in results if r.causal_patches)
    nfb = sum(sum(1 for p in r.causal_patches if p.selection_source == "fallback") for r in results)
    nfi = sum(1 for r in results if r.used_fallback); n_true = nc - nfb
    nsi = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "in_anatomy") for r in results)
    nso = sum(sum(1 for p in r.spurious_patches if p.spurious_source == "outside_anatomy") for r in results)
    fb_pct = 100.0 * nfb / max(nc, 1)
    print(f"{split:<6}: {len(results):,} imgs | {ne:,} w/causal | {nc:,} causal "
          f"({n_true} true / {nfb} fb = {fb_pct:.1f}% fb / {nfi} imgs) | spur_in={nsi:,} spur_out={nso:,}")

# ==================== SECTION 6.4 — A* localization metrics ===============
_BBOX_LABEL_MAP = {"Atelectasis": "Atelectasis", "Cardiomegaly": "Cardiomegaly",
                   "Pleural Effusion": "Effusion"}   # Consolidation/Edema: no GT boxes

def load_bbox_gt(bbox_csv, orig_size=1024):
    """[v15 correctness] Robust column detection for NIH BBox_List_2017.csv.
    Returns {basename: [(class, (x1,y1,x2,y2) in IMAGE_SIZE space), ...]}. The v14
    version guessed columns ("h]" hardcoded) and swallowed every parse error. This
    resolves x/y/w/h by fuzzy name and warns (rather than silently dropping) on the
    first unparseable row; orig_size is documented (NIH frontals are 1024px)."""
    if not bbox_csv or not os.path.exists(bbox_csv):
        print(f"  [loc] BBox CSV not found ({bbox_csv}) — localization eval skipped."); return {}
    df = pd.read_csv(bbox_csv)
    norm = {re.sub(r"[^a-z]", "", c.lower()): c for c in df.columns}
    def _pick(*keys, required=True):
        for k in keys:
            if k in norm: return norm[k]
        for k in keys:
            for nk, orig in norm.items():
                if nk.startswith(k): return orig
        if required:
            raise KeyError(f"BBox CSV missing a column for {keys}; have {list(df.columns)}")
        return None
    ci = _pick("imageindex", "image")
    cl = _pick("findinglabel", "label")
    bx = _pick("bboxx", "x")
    by = _pick("bboxy", "y")
    bw = _pick("bboxw", "w")
    bh = _pick("bboxh", "h")
    inv = {v: k for k, v in _BBOX_LABEL_MAP.items()}
    s = IMAGE_SIZE / float(orig_size); gt = defaultdict(list); n_bad = 0
    for _, r in df.iterrows():
        lab = str(r[cl]).strip()
        if lab not in inv: continue
        try:
            x = float(r[bx]) * s; y = float(r[by]) * s; w = float(r[bw]) * s; h = float(r[bh]) * s
        except Exception:
            n_bad += 1; continue
        gt[os.path.basename(str(r[ci]))].append((inv[lab], (x, y, x + w, y + h)))
    if n_bad:
        print(f"  [loc] WARNING: {n_bad} GT rows unparseable and skipped.")
    print(f"  [loc] loaded {sum(len(v) for v in gt.values())} GT boxes over {len(gt)} images "
          f"(classes: {sorted(set(inv.values()))})")
    return gt

def _iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1); ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1); inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0

def _center_in(box, gt):
    cx, cy = (box[0] + box[2]) / 2.0, (box[1] + box[3]) / 2.0
    return gt[0] <= cx <= gt[2] and gt[1] <= cy <= gt[3]

def evaluate_localization(rd, bbox_gt, splits=("val",), iou_thrs=(0.1, 0.25), causal_only=True):
    """[v15 FIX 6/11] A* patch-quality metric on a single held-out split, STRICTLY
    causal-only (fallback patches never counted — v14 re-admitted them "if nothing
    else", making causal_only inert). Reports n and Wilson 95% CI so a
    pointing-game printed at n=4 is not mistaken for a real number."""
    if not bbox_gt:
        return pd.DataFrame()
    results = [r for sp in splits for r in rd.get(sp, [])]
    per = {c: {"n": 0, "hit": 0, **{f"iou@{t}": 0 for t in iou_thrs}} for c in _BBOX_LABEL_MAP}
    for r in results:
        gts = bbox_gt.get(os.path.basename(str(r.image_name)))
        if not gts: continue
        for cls in _BBOX_LABEL_MAP:
            cls_gts = [g[1] for g in gts if g[0] == cls]
            if not cls_gts: continue
            cand = [p for p in r.causal_patches if p.pathology == cls
                    and (not causal_only or p.selection_source != "fallback")]
            if not cand: continue          # STRICT: no fallback re-admission
            best = max(cand, key=lambda p: p.combined_score); pb = best.box
            per[cls]["n"] += 1
            if any(_center_in(pb, g) for g in cls_gts): per[cls]["hit"] += 1
            miou = max(_iou(pb, g) for g in cls_gts)
            for t in iou_thrs:
                if miou >= t: per[cls][f"iou@{t}"] += 1
    rows = []
    for c, d in per.items():
        n = d["n"]; pg, lo, hi = _wilson_ci(d["hit"], n)
        row = {"class": c, "n_eval": n, "pointing_game": pg, "pg_ci_lo": lo, "pg_ci_hi": hi}
        for t in iou_thrs:
            row[f"iou@{t}"] = (d[f"iou@{t}"] / n) if n else float("nan")
        rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty:
        print("\n" + "=" * 78 + "\nA* LOCALIZATION (top CAUSAL patch vs NIH GT box, held-out)\n" + "=" * 78)
        print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        print("pointing_game=center-in-GT (Wilson 95% CI). Trust nothing with n_eval<~20.")
    return df

# ==================== SECTION 6.5 — A* image-level metrics ================
def build_gt_lookup(*dfs):
    """[v15 correctness] `*dfs` signature so any number of splits can be folded in
    without the fragile two-arg-plus-concat call the v14 code used for NIH's
    three splits."""
    lut = {}
    for df in dfs:
        if df is None: continue
        lcols = _find_label_cols(df)
        name_col = next((c for c in ["image_name", "Image Index", "image"] if c in df.columns), None)
        for _, row in df.iterrows():
            nm = os.path.basename(str(row[name_col])) if name_col else os.path.basename(_resolve_image_name(row))
            vec = {}
            for c in TARGET_CLASSES:
                col = lcols.get(c)
                v = row[col] if col is not None else _get_label_val(row, c)
                try:
                    fv = float(v) if (v is not None and not pd.isna(v)) else float("nan")
                    vec[c] = fv if fv in (0.0, 1.0) else float("nan")
                except Exception:
                    vec[c] = float("nan")
            lut[nm] = vec
    return lut

def report_parser_agreement(plans, gt_lut):
    """[v15 FIX 1] Make the circularity confound EXPLICIT. The parser decides which
    classes to query by keyword-matching the report; the NIH labels are themselves
    NLP-derived from those reports. This table scores the parser's queried_classes
    against the labels — high precision/recall here means the downstream 'AUROC'
    is largely re-measuring the parser, not CheXzero or patch-mining."""
    tp = {c: 0 for c in TARGET_CLASSES}; fp = {c: 0 for c in TARGET_CLASSES}
    fn = {c: 0 for c in TARGET_CLASSES}; tn = {c: 0 for c in TARGET_CLASSES}
    for p in plans:
        gt = gt_lut.get(os.path.basename(str(p.image_name)))
        if gt is None: continue
        q = set(p.queried_classes)
        for c in TARGET_CLASSES:
            lv = gt.get(c, float("nan"))
            if lv != lv: continue
            pos_lab = (lv == 1.0); queried = (c in q)
            if queried and pos_lab: tp[c] += 1
            elif queried and not pos_lab: fp[c] += 1
            elif not queried and pos_lab: fn[c] += 1
            else: tn[c] += 1
    rows = []
    for c in TARGET_CLASSES:
        prec = tp[c] / max(tp[c] + fp[c], 1); rec = tp[c] / max(tp[c] + fn[c], 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        rows.append({"class": c, "queried_precision": prec, "queried_recall": rec,
                     "f1": f1, "tp": tp[c], "fp": fp[c], "fn": fn[c], "tn": tn[c]})
    df = pd.DataFrame(rows)
    print("\n" + "=" * 82 + "\n[FIX 1] REPORT-PARSER vs LABELS (the circularity confound, explicit)\n" + "=" * 82)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print("High precision/recall => downstream image AUROC largely re-measures the parser.")
    return df

def _saturation_fraction(scores):
    a = np.asarray([s for s in scores if s == s], dtype=float)
    if a.size == 0: return float("nan")
    return float(np.mean((a > 0.99) | (a < 0.01)))

def image_level_metrics(rd, gt_lut, plans_by_name=None, splits=("val",), topk=3, restrict_to_queried=True):
    """[v15 FIX 1/2/3/7] Per-class AUROC/AUPRC on a SINGLE held-out split.
      * FIX 3: no train+val pooling — `splits` selects one held-out split.
      * FIX 1/2: with restrict_to_queried, an image contributes to class c ONLY if
        c was queried for it (removes the 0.0 tie mass from plan-less images that
        made AUROC 'measure the parser'). The unrestricted column is printed too.
      * FIX 7: prints the empirical saturation fraction of the score."""
    results = [r for sp in splits for r in rd.get(sp, [])]
    q_lookup = {}
    if plans_by_name is not None:
        q_lookup = {os.path.basename(str(n)): set(p.queried_classes)
                    for n, p in plans_by_name.items()}
    def _collect(restrict):
        scores = {c: [] for c in TARGET_CLASSES}; gts = {c: [] for c in TARGET_CLASSES}
        zeropad = {c: 0 for c in TARGET_CLASSES}   # queried-but-no-causal-patch -> 0.0
        for r in results:
            nm = os.path.basename(str(r.image_name))
            gt = gt_lut.get(nm)
            if gt is None: continue
            qset = set(r.queried_classes) or q_lookup.get(nm, set())
            by_c = {c: [] for c in TARGET_CLASSES}
            for p in r.causal_patches:
                if p.pathology in by_c: by_c[p.pathology].append(float(p.zeroshot_prob))
            for c in TARGET_CLASSES:
                lv = gt.get(c, float("nan"))
                if lv != lv: continue
                if restrict and c not in qset: continue
                pr = sorted(by_c[c], reverse=True)
                if not pr: zeropad[c] += 1
                scores[c].append(float(np.mean(pr[:topk])) if pr else 0.0); gts[c].append(lv)
        return scores, gts, zeropad
    def _auc_table(scores, gts, zeropad, tag):
        rows = []
        for c in TARGET_CLASSES:
            yt = np.asarray(gts[c]); ys = np.asarray(scores[c])
            n = len(ys); zp = zeropad[c]
            base = {"class": c, "n": n, "n_zeropad": zp,
                    "zeropad_frac": (zp / n) if n else float("nan"),
                    "n_pos": int((yt == 1).sum()), "n_neg": int((yt == 0).sum()),
                    "sat_frac": _saturation_fraction(ys)}
            if len(np.unique(yt)) != 2:
                rows.append({**base, "auroc": float("nan"), "auprc": float("nan"),
                             "prevalence": float(yt.mean()) if len(yt) else float("nan")}); continue
            rows.append({**base, "auroc": float(roc_auc_score(yt, ys)),
                         "auprc": float(average_precision_score(yt, ys)),
                         "prevalence": float(yt.mean())})
        df = pd.DataFrame(rows)
        df["auprc_gain"] = df["auprc"] - df["prevalence"]
        print("\n" + "=" * 96 + f"\nA* IMAGE-LEVEL METRICS [{tag}] split={splits[0]} cohort={COHORT_MODE}\n" + "=" * 96)
        print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        print(f"macro AUROC={np.nanmean(df['auroc']):.3f} | macro AUPRC={np.nanmean(df['auprc']):.3f} "
              f"| mean sat_frac={np.nanmean(df['sat_frac']):.3f}")
        # [v15 honesty note — reviewer follow-up] Restricting to queried images
        # removes the plan-less 0.0 mass, but a queried image with ZERO causal
        # patches for class c STILL contributes a 0.0 (zeropad_frac). That residual
        # tie mass is now reported per-class rather than hidden — high zeropad_frac
        # means the AUROC is still partly a "did any patch survive" detector.
        print("NOTE: zeropad_frac = queried images with no causal patch (score=0.0); "
              "high values mean residual tie mass — read AUROC with this in mind.")
        return df
    s_r, g_r, z_r = _collect(restrict_to_queried)
    df_r = _auc_table(s_r, g_r, z_r, "queried-only [FIX 1]")
    s_a, g_a, z_a = _collect(False)
    df_a = _auc_table(s_a, g_a, z_a, "all-images (0.0-padded, for reference)")
    # raw per-class (yt, ys) for the queried set -> ROC/PR/calibration/CI plots.
    raw = {c: (np.asarray(g_r[c]), np.asarray(s_r[c])) for c in TARGET_CLASSES}
    return df_r, df_a, raw

def global_baseline_metrics(plans, gt_lut, model, preprocess, device, ctm, ls,
                            pos_mat=None, neg_mat=None, splits_label="held-out"):
    """[v15 — the one missing experiment] Standalone whole-image CheXzero baseline
    on the EXACT SAME cohort, ungated (every image scored, no plan/patch
    dependence). Until patch-mining beats THIS on the same rows there is no
    evidence it contributes anything."""
    scores = {c: [] for c in TARGET_CLASSES}; gts = {c: [] for c in TARGET_CLASSES}
    name_to_score = {}   # per-image {class: prob} for the fairness plot
    for p in tqdm(plans, desc="Global baseline"):
        nm = os.path.basename(str(p.image_name)); gt = gt_lut.get(nm)
        if gt is None or not p.image_path or not os.path.isfile(p.image_path): continue
        try:
            pil = Image.open(p.image_path).convert("RGB")
        except Exception:
            continue
        gp = global_image_class_probs(pil, model, preprocess, device, ctm, ls, pos_mat, neg_mat)
        name_to_score[nm] = {c: float(gp[TARGET_CLASSES.index(c)].item()) for c in TARGET_CLASSES}
        for c in TARGET_CLASSES:
            lv = gt.get(c, float("nan"))
            if lv == lv:
                scores[c].append(float(gp[TARGET_CLASSES.index(c)].item())); gts[c].append(lv)
    rows = []
    for c in TARGET_CLASSES:
        yt = np.asarray(gts[c]); ys = np.asarray(scores[c])
        if len(np.unique(yt)) != 2:
            rows.append({"class": c, "auroc": float("nan"), "auprc": float("nan"),
                         "n_pos": int((yt == 1).sum()), "n_neg": int((yt == 0).sum()),
                         "sat_frac": _saturation_fraction(ys)}); continue
        rows.append({"class": c, "auroc": float(roc_auc_score(yt, ys)),
                     "auprc": float(average_precision_score(yt, ys)),
                     "n_pos": int((yt == 1).sum()), "n_neg": int((yt == 0).sum()),
                     "sat_frac": _saturation_fraction(ys)})
    df = pd.DataFrame(rows)
    print("\n" + "=" * 88 + f"\nWHOLE-IMAGE CHEXZERO BASELINE (ungated, {splits_label}, cohort={COHORT_MODE})\n" + "=" * 88)
    print(df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print(f"macro AUROC={np.nanmean(df['auroc']):.3f} | macro AUPRC={np.nanmean(df['auprc']):.3f} "
          f"— patch-mining must beat THIS on the same rows to justify itself.")
    return df, name_to_score

# ==================== SECTION 6.6 — GradCAM grid for causal patches =======
def plot_causal_gradcam_grid(results, plans_by_name, gradcam, save_path=None, seed=42):
    """Must-have figure. [v15 correctness] samples a RANDOM qualifying image per
    class with a fixed seed (v14 took the arbitrary first match = cherry-pick),
    strictly non-fallback causal patches."""
    import matplotlib.cm as cm
    rng = random.Random(seed)
    fig, axes = plt.subplots(N_CLASSES, 3, figsize=(9, 3 * N_CLASSES))
    if N_CLASSES == 1: axes = np.array([axes])
    col_titles = ["(a) chest X-ray", "(b) GradCAM (causal class)", "(c) overlay + causal box"]
    for ridx, cls in enumerate(TARGET_CLASSES):
        cands = []
        for r in results:
            cps = [p for p in r.causal_patches if p.pathology == cls and p.selection_source != "fallback"]
            if cps and r.image_path and os.path.isfile(r.image_path):
                cands.append((r, max(cps, key=lambda p: p.combined_score)))
        pick = rng.choice(cands) if cands else None
        for c in range(3): axes[ridx, c].axis("off")
        if pick is None:
            axes[ridx, 0].set_title(f"{cls}: no causal patch"); continue
        r, patch = pick
        plan = plans_by_name.get(r.image_name)
        qi = None
        if plan is not None:
            qi = next((q for q in plan.query_items if q.pathology == cls and q.query_vector is not None), None)
        pil = Image.open(r.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        img = np.asarray(pil).astype(np.float32) / 255.0
        gcam = None
        if qi is not None and gradcam is not None:
            gcam = gradcam.compute_combined(pil, qi.query_vector, TARGET_CLASSES.index(cls))
        heat = gcam.cpu().numpy() if gcam is not None else np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.float32)
        heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
        heat_rgb = cm.jet(heat)[..., :3]
        overlay = 0.55 * img + 0.45 * heat_rgb
        axes[ridx, 0].imshow(img); axes[ridx, 1].imshow(heat_rgb); axes[ridx, 2].imshow(np.clip(overlay, 0, 1))
        x1, y1, x2, y2 = patch.box
        axes[ridx, 2].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                fill=False, edgecolor="lime", lw=2.2))
        axes[ridx, 0].set_title(cls, fontsize=11, loc="left")
        if ridx == 0:
            axes[ridx, 0].set_title(f"{cls}\n{col_titles[0]}", fontsize=10)
            axes[ridx, 1].set_title(col_titles[1], fontsize=10); axes[ridx, 2].set_title(col_titles[2], fontsize=10)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

# ==================== SECTION 6.7 — bboxes separated by bucket ============
def plot_bboxes_by_bucket(results, n_examples=4, save_path=None, seed=42):
    picks = [r for r in results if r.image_path and os.path.isfile(r.image_path)
             and (r.causal_patches or r.spurious_patches)]
    rng = random.Random(seed); rng.shuffle(picks); picks = picks[:n_examples]
    if not picks:
        print("  [bbox] no drawable results."); return
    fig, axes = plt.subplots(len(picks), 3, figsize=(11, 3.6 * len(picks)))
    if len(picks) == 1: axes = np.array([axes])
    titles = ["causal", "spurious in-anatomy", "spurious outside-anatomy"]
    for ridx, r in enumerate(picks):
        pil = Image.open(r.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        for c in range(3):
            axes[ridx, c].imshow(pil); axes[ridx, c].axis("off")
            if ridx == 0: axes[ridx, c].set_title(titles[c], fontsize=11)
        for p in r.causal_patches:
            col = "#1e8cff" if p.selection_source == "fallback" else "#00dc3c"
            x1, y1, x2, y2 = p.box
            axes[ridx, 0].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                    fill=False, edgecolor=col, lw=2,
                                    linestyle="--" if p.selection_source == "fallback" else "-"))
        for p in r.spurious_patches:
            x1, y1, x2, y2 = p.box
            if p.spurious_source == "in_anatomy":
                axes[ridx, 1].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor="#dc1e1e", lw=2))
            else:
                axes[ridx, 2].add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                        fill=False, edgecolor="#ffa500", lw=2, linestyle=":"))
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

# ==================== SECTION 6.8 — publication plot suite ================
# Reviewer's plot checklist. Each returns a saved PNG; all use Agg + plt.close.
try:
    from sklearn.metrics import roc_curve, precision_recall_curve
    from sklearn.calibration import calibration_curve
    from sklearn.decomposition import PCA
    _HAS_SK_EXTRA = True
except Exception:
    _HAS_SK_EXTRA = False

def _bootstrap_auroc_ci(yt, ys, n_boot=1000, seed=0):
    yt = np.asarray(yt); ys = np.asarray(ys)
    if len(np.unique(yt)) != 2: return (float("nan"), float("nan"))
    r = np.random.default_rng(seed); vals = []; n = len(yt)
    for _ in range(n_boot):
        i = r.integers(0, n, n)
        if len(np.unique(yt[i])) == 2:
            try: vals.append(roc_auc_score(yt[i], ys[i]))
            except Exception: pass
    if not vals: return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

def plot_per_class_curves(raw, save_path=None, ctrl_auroc=None):
    """[plots #6/#7/#8/#9/#10] ROC, PR, reliability (calibration), per-class AUROC
    with bootstrap CI, and the score-saturation histogram — one figure, 5 panels.
    `raw` = {class: (yt, ys)} from image_level_metrics."""
    fig, ax = plt.subplots(2, 3, figsize=(16, 9))
    colors = plt.cm.tab10(np.linspace(0, 1, len(TARGET_CLASSES)))
    # (0,0) ROC
    for c, col in zip(TARGET_CLASSES, colors):
        yt, ys = raw.get(c, (np.array([]), np.array([])))
        if len(np.unique(yt)) == 2 and _HAS_SK_EXTRA:
            fpr, tpr, _ = roc_curve(yt, ys)
            ax[0, 0].plot(fpr, tpr, color=col, lw=1.6,
                          label=f"{c} ({roc_auc_score(yt, ys):.2f})")
    ax[0, 0].plot([0, 1], [0, 1], "k:", lw=1); ax[0, 0].set_title("(a) ROC (held-out)")
    ax[0, 0].set_xlabel("FPR"); ax[0, 0].set_ylabel("TPR"); ax[0, 0].legend(fontsize=7)
    # (0,1) PR
    for c, col in zip(TARGET_CLASSES, colors):
        yt, ys = raw.get(c, (np.array([]), np.array([])))
        if len(np.unique(yt)) == 2 and _HAS_SK_EXTRA:
            pr, rc, _ = precision_recall_curve(yt, ys)
            ax[0, 1].plot(rc, pr, color=col, lw=1.6,
                          label=f"{c} ({average_precision_score(yt, ys):.2f})")
    ax[0, 1].set_title("(b) Precision-Recall"); ax[0, 1].set_xlabel("recall")
    ax[0, 1].set_ylabel("precision"); ax[0, 1].legend(fontsize=7)
    # (0,2) calibration
    for c, col in zip(TARGET_CLASSES, colors):
        yt, ys = raw.get(c, (np.array([]), np.array([])))
        if len(np.unique(yt)) == 2 and _HAS_SK_EXTRA and len(yt) >= 10:
            try:
                fr, mp = calibration_curve(yt, ys, n_bins=min(8, len(yt) // 2), strategy="quantile")
                ax[0, 2].plot(mp, fr, "o-", color=col, ms=3, lw=1.3, label=c)
            except Exception: pass
    ax[0, 2].plot([0, 1], [0, 1], "k:", lw=1); ax[0, 2].set_title("(c) Reliability")
    ax[0, 2].set_xlabel("mean predicted"); ax[0, 2].set_ylabel("empirical frac pos"); ax[0, 2].legend(fontsize=7)
    # (1,0) per-class AUROC with bootstrap CI
    labs, aus, los, his = [], [], [], []
    for c in TARGET_CLASSES:
        yt, ys = raw.get(c, (np.array([]), np.array([])))
        if len(np.unique(yt)) == 2:
            au = roc_auc_score(yt, ys); lo, hi = _bootstrap_auroc_ci(yt, ys)
            labs.append(c); aus.append(au); los.append(au - lo); his.append(hi - au)
    if labs:
        xx = np.arange(len(labs))
        ax[1, 0].bar(xx, aus, yerr=[los, his], capsize=4, color="#2ca02c")
        ax[1, 0].axhline(0.5, color="#888", ls=":")
        if ctrl_auroc == ctrl_auroc and ctrl_auroc is not None:
            ax[1, 0].axhline(ctrl_auroc, color="#d62728", ls="--", lw=1.2, label=f"CTRL={ctrl_auroc:.2f}")
            ax[1, 0].legend(fontsize=7)
        ax[1, 0].set_xticks(xx); ax[1, 0].set_xticklabels(labs, rotation=40, ha="right", fontsize=7)
    ax[1, 0].set_ylim(0, 1); ax[1, 0].set_title("(d) per-class AUROC + 95% CI")
    # (1,1) saturation histogram
    allsc = np.concatenate([raw[c][1] for c in TARGET_CLASSES if len(raw[c][1])]) if raw else np.array([])
    if allsc.size:
        ax[1, 1].hist(allsc, bins=30, color="#1f77b4", alpha=0.85)
        ax[1, 1].axvline(0.99, color="#d62728", ls=":"); ax[1, 1].axvline(0.01, color="#d62728", ls=":")
        ax[1, 1].set_title(f"(e) score histogram (sat_frac={_saturation_fraction(allsc):.2f})")
        ax[1, 1].set_xlabel("ensembled class prob")
    # (1,2) prevalence vs AUPRC
    for c, col in zip(TARGET_CLASSES, colors):
        yt, ys = raw.get(c, (np.array([]), np.array([])))
        if len(np.unique(yt)) == 2:
            ax[1, 2].scatter(yt.mean(), average_precision_score(yt, ys), color=col, s=40, label=c)
    ax[1, 2].plot([0, 1], [0, 1], "k:", lw=1); ax[1, 2].set_xlabel("prevalence")
    ax[1, 2].set_ylabel("AUPRC"); ax[1, 2].set_title("(f) AUPRC vs prevalence"); ax[1, 2].legend(fontsize=7)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

def plot_embedding_separation(cp, sp, op, save_path=None, seed=42, max_pts=1500):
    """[plot #11 — load-bearing] Do causal / spurious-in / spurious-out patch
    embeddings actually separate? 2D PCA of the three populations from the .pt
    files. If they don't separate, the 'causal vs spurious' framing is unsupported."""
    if not _HAS_SK_EXTRA:
        print("  [emb] sklearn PCA unavailable — skipped."); return
    groups = [("causal", cp, "#00a000"), ("spurious-in", sp, "#d62728"), ("spurious-out", op, "#ff9900")]
    X, y = [], []
    for i, (name, pk, _) in enumerate(groups):
        e = pk.get("embeddings")
        if e is None or (hasattr(e, "shape") and e.shape[0] == 0): continue
        e = e.float().cpu().numpy()
        X.append(e); y += [i] * len(e)
    if not X:
        print("  [emb] no embeddings to plot — skipped."); return
    X = np.concatenate(X); y = np.asarray(y)
    if len(X) > max_pts:
        r = np.random.default_rng(seed); idx = r.choice(len(X), max_pts, replace=False); X, y = X[idx], y[idx]
    Z = PCA(n_components=2, random_state=seed).fit_transform(X)
    fig, axp = plt.subplots(figsize=(7, 6))
    for i, (name, _, col) in enumerate(groups):
        m = y == i
        if m.any(): axp.scatter(Z[m, 0], Z[m, 1], s=8, alpha=0.5, color=col, label=f"{name} (n={int(m.sum())})")
    axp.legend(); axp.set_title("Patch-embedding separation (2D PCA)")
    axp.set_xlabel("PC1"); axp.set_ylabel("PC2")
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

def plot_localization_ci(loc_df, save_path=None):
    """[plot #13] Wilson-CI width vs n_eval per class — makes the 'trust nothing
    with n_eval<~20' warning visual, and confirms FIX 11 raised n."""
    if loc_df is None or loc_df.empty: return
    fig, axl = plt.subplots(figsize=(7, 5))
    for _, r in loc_df.iterrows():
        n = r.get("n_eval", 0); pg = r.get("pointing_game", float("nan"))
        lo = r.get("pg_ci_lo", float("nan")); hi = r.get("pg_ci_hi", float("nan"))
        if n and pg == pg:
            axl.errorbar(n, pg, yerr=[[pg - lo], [hi - pg]], fmt="o", capsize=4, label=str(r["class"]))
    axl.axvline(20, color="#888", ls=":", label="n=20 trust floor")
    axl.set_xlabel("n_eval (images with GT box + causal patch)"); axl.set_ylabel("pointing game")
    axl.set_ylim(0, 1); axl.set_title("Localization: pointing-game vs n (Wilson 95% CI)"); axl.legend(fontsize=8)
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

def plot_persistence_diag(save_path=None):
    """[plot #14] Distribution of reconstructed persistence-mask AREAS and the
    birth-death gap — validates the death-threshold fix (masks should be real
    basins, not near-peak specks) and is a methods-section figure."""
    if not _PERSIST_AREAS:
        print("  [persist-diag] no masks recorded — skipped."); return
    areas = np.array([a for a, _ in _PERSIST_AREAS], dtype=float)
    gaps = np.array([g for _, g in _PERSIST_AREAS], dtype=float)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    a1.hist(areas, bins=40, color="#1f77b4"); a1.axvline(PERSISTENCE_MIN_AREA, color="#d62728", ls=":")
    a1.set_title(f"(a) persistence mask area (median={np.median(areas):.0f}px)")
    a1.set_xlabel("mask area (px)"); a1.set_ylabel("count")
    a2.hist(gaps, bins=40, color="#2ca02c"); a2.set_title("(b) birth-death gap (persistence)")
    a2.set_xlabel("birth - death"); a2.set_ylabel("count")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

def plot_fairness_subgroups(eval_df, name_to_score, gt_lut, save_path=None):
    """[plot #12 — best effort] Subgroup-stratified macro-AUROC by View Position /
    Patient Gender / Patient Age band, IF those columns exist in the cohort frame.
    Uses the ungated global-baseline score per image (name_to_score). Honestly
    skips (with a printed note) any subgroup column that is absent."""
    subgroup_cols = {"view": ["View Position", "View_Position", "view_position"],
                     "sex":  ["Patient Gender", "Patient_Gender", "sex", "gender"],
                     "age":  ["Patient Age", "Patient_Age", "age"]}
    present = {}
    for key, cands in subgroup_cols.items():
        col = next((c for c in cands if c in eval_df.columns), None)
        if col is not None: present[key] = col
    if not present:
        print("  [fairness] no subgroup columns (view/sex/age) in cohort — skipped honestly."); return
    bn = _row_basenames(eval_df)
    def _macro_auroc(mask):
        aus = []
        for c in TARGET_CLASSES:
            yt, ys = [], []
            for i in np.where(mask)[0]:
                nm = bn.iloc[i]; gt = gt_lut.get(nm); sc = name_to_score.get(nm)
                if gt is None or sc is None: continue
                lv = gt.get(c, float("nan"))
                if lv == lv: yt.append(lv); ys.append(sc.get(c, 0.0))
            if len(set(yt)) == 2:
                try: aus.append(roc_auc_score(yt, ys))
                except Exception: pass
        return float(np.nanmean(aus)) if aus else float("nan")
    fig, axes = plt.subplots(1, len(present), figsize=(5 * len(present), 4.2), squeeze=False)
    for j, (key, col) in enumerate(present.items()):
        vals = eval_df[col].astype(str)
        if key == "age":
            num = pd.to_numeric(eval_df[col], errors="coerce")
            vals = pd.cut(num, [0, 40, 60, 200], labels=["<40", "40-60", ">60"]).astype(str)
        cats = [v for v in vals.dropna().unique() if v != "nan"][:6]
        scores = [_macro_auroc((vals == cat).values) for cat in cats]
        axes[0, j].bar(range(len(cats)), scores, color="#1f77b4")
        axes[0, j].axhline(0.5, color="#888", ls=":")
        axes[0, j].set_xticks(range(len(cats))); axes[0, j].set_xticklabels(cats, rotation=30, ha="right", fontsize=8)
        axes[0, j].set_ylim(0, 1); axes[0, j].set_title(f"macroAUROC by {key}"); axes[0, j].set_ylabel("AUROC")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=130, bbox_inches="tight"); print(f"Saved: {save_path}")
    plt.close(fig)

print("[cell 6] discovery, de-circularised/held-out metrics, parser agreement, "
      "global baseline, Wilson localization, seeded figures + plot suite ready.")


## Cell 7 — Flags, self-test (+ visualization), Stage 2+3 driver

`RUN_SELFTEST` runs the data-free unit test that **executes the real ensemble line**, checks the temperature de-saturates, runs 102 parser gold cases, and asserts the persistence mask is a real basin (area + peak-on-blob) — the test that would have caught the persistence bug. `RUN_SELFTEST_VIZ=True` renders synthetic-data figures so you can **see visualizations without any dataset**. The driver keeps train/val/test strictly separate and reports on one held-out split.

In [ ]:
# =============================================================================
# CELL 7 / 8  —  RUN FLAGS  +  REAL UNIT SELF-TEST  +  STAGE 2+3 DRIVER
# =============================================================================

# ==================== SECTION 7.1 — flags =================================
RUN_SELFTEST     = True     # [renamed] GPU/data-free unit self-test (safe anywhere; ~2s)
RUN_SELFTEST_VIZ = True     # [NEW] render synthetic-data VISUALIZATIONS from the self-test
                            # so you can SEE the persistence mask + saturation plots WITHOUT
                            # any dataset or checkpoint. Set False to skip the figures.
RUN_STAGE23    = True      # Stage 2 (query plans) + Stage 3 (patch discovery) — needs data+ckpt
RUN_TEST_EVAL  = True       # [FIX 3] also run Stage 3 on the held-out TEST split (NIH; ~+1x cost)
RUN_ABLATIONS  = True      # cell 8 — needs RUN_STAGE23 objects in memory

# ==================== SECTION 7.2 — internal unit self-test ===============
# [v15 FIX 12] The v14 self-test was theatre: it re-derived arithmetic by hand,
# asserted structural decoupling, and (worst) asserted np.random.RandomState
# LACKS .integers so the suite failed the day numpy added it. This version
# EXECUTES the real classify_patches_gpu ensemble line and runs 100+ report-parser
# gold cases (the highest-risk component). No CUDA / CheXzero / data required.
def _parser_gold_cases():
    """>=100 hand-labelled (sentence, keyword, expected_status) cases covering
    negation, uncertainty, visualization-limits, and scope breaks."""
    G = []
    # --- clear positives (definite findings) ---
    pos = [
        ("there is a large pleural effusion", "effusion"),
        ("cardiomegaly is present", "cardiomegaly"),
        ("dense consolidation in the right lower lobe", "consolidation"),
        ("bibasilar atelectasis", "atelectasis"),
        ("pulmonary edema with vascular congestion", "edema"),
        ("increased opacity in the right base", "opacity"),
        ("the heart is enlarged", "heart is enlarged"),
        ("layering pleural effusion on the left", "effusion"),
        ("moderate cardiomegaly", "cardiomegaly"),
        ("patchy airspace opacity at the left base", "airspace opacity"),
        ("blunting of the costophrenic angle", "blunting"),
        ("plate-like atelectasis at the lung base", "atelectasis"),
        ("interstitial edema", "edema"),
        ("focal consolidation", "consolidation"),
        ("the cardiac silhouette is enlarged", "cardiac silhouette is enlarged"),
        ("there is volume loss in the left lung", "volume loss"),
        ("kerley b lines consistent with edema", "edema"),
        ("air bronchogram within the consolidation", "air bronchogram"),
        ("effusion is seen at the right base", "effusion"),
        ("small pleural effusion", "effusion"),
    ]
    G += [(s, k, "positive") for s, k in pos]
    # --- negations ---
    neg = [
        ("no evidence of pneumonia", "pneumonia"),
        ("no pleural effusion", "effusion"),
        ("no consolidation", "consolidation"),
        ("no pneumothorax or effusion", "effusion"),
        ("the lungs are clear without consolidation", "consolidation"),
        ("heart size is normal", "heart size"),
        ("no cardiomegaly", "cardiomegaly"),
        ("without pulmonary edema", "edema"),
        ("no focal airspace opacity", "airspace opacity"),
        ("no blunting of the costophrenic angle", "blunting"),
        ("no atelectasis", "atelectasis"),
        ("effusion has resolved", "effusion"),
        ("no significant cardiomegaly", "cardiomegaly"),
        ("free of consolidation", "consolidation"),
        ("negative for effusion", "effusion"),
        ("no definite consolidation", "consolidation"),
        ("edema is not seen", "edema"),
        ("atelectasis has resolved", "atelectasis"),
        ("no acute cardiopulmonary consolidation", "consolidation"),
        ("unremarkable heart size", "heart size"),
    ]
    G += [(s, k, "negated") for s, k in neg]
    # --- uncertainty (genuine epistemic hedges kept in _UNC_CUES) ---
    unc = [
        ("possible consolidation at the base", "consolidation"),
        ("probable atelectasis", "atelectasis"),
        ("cardiomegaly is likely", "cardiomegaly"),
        ("findings suggestive of edema", "edema"),
        ("concerning for effusion", "effusion"),
        ("may represent atelectasis", "atelectasis"),
        ("could represent consolidation", "consolidation"),
        ("questionable effusion", "effusion"),
        ("suspected pulmonary edema", "edema"),
        ("borderline cardiomegaly", "cardiomegaly"),
        ("cannot exclude consolidation", "consolidation"),
        ("indeterminate opacity", "opacity"),
        ("equivocal atelectasis", "atelectasis"),
        ("worrisome for effusion", "effusion"),
        ("differential includes edema", "edema"),
    ]
    G += [(s, k, "uncertain") for s, k in unc]
    # --- visualization limits -> uncertain ---
    vis = [
        ("the left base is not well visualized for consolidation", "consolidation"),
        ("effusion not fully evaluated on this projection", "effusion"),
        ("atelectasis not adequately assessed", "atelectasis"),
    ]
    G += [(s, k, "uncertain") for s, k in vis]
    # --- scope-break: negation in a different clause must NOT negate finding ---
    scope = [
        ("increased opacity in the right base, no pleural effusion", "opacity", "positive"),
        ("cardiomegaly is present, but no effusion", "cardiomegaly", "positive"),
        ("consolidation in the left lung; heart size normal", "consolidation", "positive"),
        ("no effusion, however there is dense consolidation", "consolidation", "positive"),
        ("effusion on the right, lungs otherwise clear", "effusion", "positive"),
    ]
    G += scope
    # --- absent (keyword not present) ---
    G += [("the study is unremarkable", "effusion", "absent"),
          ("no acute findings", "consolidation", "absent")]
    # [FIX 13 regression] these boilerplate/broad cues must NOT flip a positive:
    G += [("dense consolidation, correlate clinically", "consolidation", "positive"),
          ("effusion versus atelectasis at the base", "effusion", "positive"),
          ("further evaluation of the opacity is recommended", "opacity", "positive")]
    # --- extra positives (finding present, no cue in scope) ---
    pos2 = [
        ("bilateral pleural effusions", "effusion"),
        ("right lower lobe consolidation", "consolidation"),
        ("mild cardiomegaly", "cardiomegaly"),
        ("basilar atelectasis", "atelectasis"),
        ("mild pulmonary edema", "edema"),
        ("large left pleural effusion", "effusion"),
        ("consolidation is noted in the right upper lobe", "consolidation"),
        ("stable cardiomegaly", "cardiomegaly"),
        ("persistent right basilar atelectasis", "atelectasis"),
        ("diffuse interstitial opacities", "interstitial opacities"),
        ("new consolidation in the lingula", "consolidation"),
        ("worsening pulmonary edema", "edema"),
        ("gross cardiomegaly", "cardiomegaly"),
        ("layering effusion", "effusion"),
        ("dependent atelectasis at both bases", "atelectasis"),
    ]
    G += [(s, k, "positive") for s, k in pos2]
    # --- extra negations ---
    neg2 = [
        ("no pleural effusions", "effusion"),
        ("without evidence of consolidation", "consolidation"),
        ("cardiac silhouette is normal", "cardiac silhouette"),
        ("no interval development of edema", "edema"),
        ("lungs remain clear of consolidation", "consolidation"),
        ("no residual effusion", "effusion"),
        ("effusion is no longer seen", "effusion"),
        ("normal cardiomediastinal silhouette", "cardiomediastinal silhouette"),
        ("no new consolidation", "consolidation"),
    ]
    G += [(s, k, "negated") for s, k in neg2]
    # --- extra uncertainty ---
    unc2 = [
        ("possible small effusion", "effusion"),
        ("likely atelectasis versus scarring", "atelectasis"),
        ("may reflect edema", "edema"),
        ("probable consolidation", "consolidation"),
        ("questionable opacity", "opacity"),
        ("cannot rule out effusion", "effusion"),
        ("findings could represent edema", "edema"),
    ]
    G += [(s, k, "uncertain") for s, k in unc2]
    # --- a few more to clear 100 gold cases ---
    G += [("moderate pulmonary edema", "edema", "positive"),
          ("trace pleural effusion", "effusion", "positive"),
          ("no cardiopulmonary abnormality", "consolidation", "absent")]
    return G

def run_internal_smoke_test():
    print("\n" + "=" * 70 + "\nINTERNAL UNIT SELF-TEST (v15)\n" + "=" * 70)
    dev = torch.device("cpu"); ok = True
    ls = torch.tensor(100.0)

    # -- FIX A: binary prompts decouple classes --------------------------------
    torch.manual_seed(0)
    pos = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    neg = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    emb = F.normalize(torch.randn(4, CLIP_EMBED_DIM), dim=-1)
    bp = binary_class_probs_gpu(emb, pos, neg, ls)
    assert bp.shape == (4, N_CLASSES) and torch.all((bp >= 0) & (bp <= 1)), "binary prob shape/range"
    pos2 = pos.clone(); pos2[1:] = F.normalize(torch.randn(N_CLASSES - 1, CLIP_EMBED_DIM), dim=-1)
    decoupled = torch.allclose(bp[:, 0], binary_class_probs_gpu(emb, pos2, neg, ls)[:, 0], atol=1e-6)
    print(f"  [FIX A] binary decoupled across classes: {decoupled}")
    ok &= bool(decoupled)

    # -- FIX 7: temperature actually de-saturates ------------------------------
    global BINARY_PROB_TEMP
    _save_temp = BINARY_PROB_TEMP
    close = F.normalize(torch.tensor([[1.0] + [0.0]*(CLIP_EMBED_DIM-1)]), dim=-1)
    p_hot = binary_class_probs_gpu(close, pos, neg, ls)
    BINARY_PROB_TEMP = 10.0
    p_cool = binary_class_probs_gpu(close, pos, neg, ls)
    BINARY_PROB_TEMP = _save_temp
    desat = _saturation_fraction(p_cool.flatten().tolist()) <= _saturation_fraction(p_hot.flatten().tolist())
    print(f"  [FIX 7] cooler temperature does not increase saturation: {desat}")
    ok &= bool(desat)

    # -- FIX B live: EXECUTE the real classify_patches_gpu ensemble line -------
    qi = PathologyQueryItem(pathology="Atelectasis", text_snippet="x",
                            query_vector=F.normalize(torch.randn(CLIP_EMBED_DIM), dim=0))
    boxes = [(0, 0, 128, 128), (100, 100, 228, 228)]
    embs = F.normalize(torch.randn(2, CLIP_EMBED_DIM), dim=-1)
    probs = torch.tensor([[0.2, 0.1, 0.1, 0.1, 0.1], [0.6, 0.1, 0.1, 0.1, 0.1]])
    gp = torch.tensor([0.9, 0.0, 0.0, 0.0, 0.0])
    _, allsc = classify_patches_gpu(boxes, embs, probs, None, qi, 0, 1, "img", dev,
                                    persistent_mask=None, persistent_peaks=None, global_prob=gp)
    a = GLOBAL_ENSEMBLE_ALPHA; exp0 = a * 0.9 + (1 - a) * 0.2
    ens_live = abs(allsc[0].zeroshot_prob - exp0) < 1e-4
    print(f"  [FIX B live] ensemble applied by real code: {ens_live} "
          f"(stored {allsc[0].zeroshot_prob:.3f} vs expected {exp0:.3f})")
    ok &= ens_live

    # -- FIX 12: 100+ report-parser gold cases (the highest-risk component) -----
    gold = _parser_gold_cases()
    bad = [(s, kw, exp, sentence_status(s, kw)) for s, kw, exp in gold if sentence_status(s, kw) != exp]
    print(f"  [parser] sentence_status gold: {len(gold)-len(bad)}/{len(gold)} correct"
          + ("" if not bad else f"  MISMATCH(first5): {bad[:5]}"))
    ok &= (len(bad) == 0)

    # -- FIX C: per-class scale map keeps 64px for small findings --------------
    keeps64 = all(64 in _SCALE_MAP[c] for c in ("Atelectasis", "Edema"))
    print(f"  [FIX C] Atelectasis/Edema keep 64px scale: {keeps64}"); ok &= keeps64

    # -- Box geometry + persistence on a synthetic 2-blob CAM ------------------
    for (x, y, s) in [(500, 500, 128), (-10, -10, 8), (0, 0, 9999)]:
        b = _clamp_box_square(x, y, s)
        if b is not None:
            w = b[2]-b[0]; h = b[3]-b[1]
            assert w == h and w >= MIN_BOX_SIZE and 0 <= b[0] and b[2] <= IMAGE_SIZE, f"geom {b}"
    yy, xx = torch.meshgrid(torch.arange(IMAGE_SIZE), torch.arange(IMAGE_SIZE), indexing="ij")
    g = torch.exp(-(((xx-150.)**2 + (yy-150.)**2)/(2*40.**2))) \
        + 0.8*torch.exp(-(((xx-380.)**2 + (yy-360.)**2)/(2*30.**2)))
    m, peaks = compute_persistent_causal_mask(g.float())
    # [v15 FIX regression] The v14 self-test only checked `m is not None` — so the
    # birth-vs-death collapse bug (mask reconstructed as a near-peak speck) sailed
    # straight through it. Assert the mask is a REAL basin (not a speck, not the
    # whole image) and that its peak sits on a known synthetic blob centre.
    _pareaok = _ppeakok = False
    if m is not None:
        _marea = float(m.sum().item())
        _pareaok = (_marea > 500.0) and (_marea < 0.60 * IMAGE_SIZE * IMAGE_SIZE)
        if peaks:
            py, px = peaks[0]
            _ppeakok = ((abs(py - 150) < 20 and abs(px - 150) < 20) or
                        (abs(py - 360) < 20 and abs(px - 380) < 20))
    print(f"  [persist] mask found={m is not None} area_ok={_pareaok} peak_on_blob={_ppeakok} "
          f"(area={float(m.sum().item()) if m is not None else 0:.0f}px — catches birth/death collapse)")
    ok &= (m is not None and _pareaok and _ppeakok)

    # [v15 FIX] evaluate_localization executed against synthetic GT (v14 only tested
    # the _iou/_center_in primitives, never the end-to-end pointing-game math).
    _hit_doc = PatchDocument(image_name="hit.png", pathology="Atelectasis", scale=100,
        box=(100, 100, 200, 200), visual_embedding=torch.zeros(2), semantic_score=0.5,
        zeroshot_prob=0.9, gradcam_score=0.5, combined_score=0.9, causal=True,
        anatomical_region="x", confidence=1.0, text_snippet="x", selection_source="threshold")
    _miss_doc = PatchDocument(image_name="miss.png", pathology="Atelectasis", scale=64,
        box=(400, 400, 464, 464), visual_embedding=torch.zeros(2), semantic_score=0.5,
        zeroshot_prob=0.9, gradcam_score=0.5, combined_score=0.9, causal=True,
        anatomical_region="x", confidence=1.0, text_snippet="x", selection_source="threshold")
    _rd = {"val": [ImagePatchResult("hit.png", "", "val", causal_patches=[_hit_doc]),
                   ImagePatchResult("miss.png", "", "val", causal_patches=[_miss_doc])]}
    _bgt = {"hit.png": [("Atelectasis", (90, 90, 210, 210))],
            "miss.png": [("Atelectasis", (90, 90, 210, 210))]}
    _ldf = evaluate_localization(_rd, _bgt, splits=("val",))
    _lrow = _ldf[_ldf["class"] == "Atelectasis"].iloc[0]
    _loc_ok = (int(_lrow["n_eval"]) == 2 and abs(float(_lrow["pointing_game"]) - 0.5) < 1e-9)
    print(f"  [loc-e2e] evaluate_localization n_eval={int(_lrow['n_eval'])} "
          f"pointing_game={float(_lrow['pointing_game']):.2f} (expect 2, 0.50): {_loc_ok}")
    ok &= _loc_ok

    # -- control RNG uses default_rng (NOT the brittle RandomState-lacks-integers) --
    def _rand_boxes(n, rng):
        out = []
        for _ in range(n):
            side = min(int(rng.choice([64, 128, 256])), IMAGE_SIZE)
            x1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            y1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            out.append((x1, y1, x1 + side, y1 + side))
        return out
    rb = _rand_boxes(8, np.random.default_rng(42 + 991))
    print(f"  [RNG] control boxes via default_rng OK ({len(rb)}) — no RandomState.integers assertion")
    ok &= (len(rb) == 8)

    # -- localization math + Wilson + BH ---------------------------------------
    gt = (100, 100, 300, 300)
    assert _center_in((120, 120, 280, 280), gt) and not _center_in((350, 350, 400, 400), gt)
    assert _iou((100, 100, 300, 300), gt) == 1.0 and _iou((350, 350, 400, 400), gt) == 0.0
    p, lo, hi = _wilson_ci(4, 4); assert lo < 1.0 and hi <= 1.0 and p == 1.0, "wilson"
    rej, q = _benjamini_hochberg([0.001, 0.2, 0.04, 0.5])
    print(f"  [stats] Wilson(4/4)=({p:.2f},[{lo:.2f},{hi:.2f}]) | BH rejects {int(rej.sum())}/4")

    print("=" * 70)
    print("SELF-TEST: " + ("ALL CHECKS PASSED" if ok else "FAILURES - inspect above"))
    print("=" * 70)
    return ok

def render_selftest_visuals(out_dir=None):
    """[NEW smoke visualization] Produce real figures from SYNTHETIC data only —
    no dataset, no checkpoint, no GPU — so you can eyeball that the plotting +
    the fixed persistence mask work. Saves two PNGs and returns their paths."""
    out_dir = out_dir or OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    import matplotlib.cm as cm
    paths = []
    # (1) persistence mask on the synthetic 2-blob CAM — validates the death fix.
    yy, xx = torch.meshgrid(torch.arange(IMAGE_SIZE), torch.arange(IMAGE_SIZE), indexing="ij")
    g = (torch.exp(-(((xx - 150.) ** 2 + (yy - 150.) ** 2) / (2 * 40. ** 2)))
         + 0.8 * torch.exp(-(((xx - 380.) ** 2 + (yy - 360.) ** 2) / (2 * 30. ** 2)))).float()
    m, peaks = compute_persistent_causal_mask(g)
    heat = g.cpu().numpy(); heat = (heat - heat.min()) / (heat.max() - heat.min() + 1e-8)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
    ax[0].imshow(cm.jet(heat)[..., :3]); ax[0].set_title("(a) synthetic CAM (2 blobs)"); ax[0].axis("off")
    mnp = m.cpu().numpy() if m is not None else np.zeros_like(heat)
    ax[1].imshow(mnp, cmap="gray"); ax[1].set_title(f"(b) persistence mask ({int(mnp.sum())}px)"); ax[1].axis("off")
    ov = 0.6 * np.stack([heat] * 3, -1) + 0.4 * np.stack([mnp] * 3, -1)
    ax[2].imshow(np.clip(ov, 0, 1))
    for (py, px) in peaks: ax[2].plot(px, py, "r+", ms=14, mew=2)
    ax[2].set_title("(c) overlay + peaks"); ax[2].axis("off")
    plt.tight_layout(); p1 = f"{out_dir}/selftest_persistence.png"
    fig.savefig(p1, dpi=120, bbox_inches="tight"); plt.close(fig); paths.append(p1)
    # (2) saturation histogram: raw logit_scale vs BINARY_PROB_TEMP — justifies FIX 7.
    global BINARY_PROB_TEMP
    torch.manual_seed(0)
    pos = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    neg = F.normalize(torch.randn(N_CLASSES, CLIP_EMBED_DIM), dim=-1)
    emb = F.normalize(torch.randn(4000, CLIP_EMBED_DIM), dim=-1)
    ls = torch.tensor(100.0)
    _save = BINARY_PROB_TEMP
    BINARY_PROB_TEMP = None; p_raw = binary_class_probs_gpu(emb, pos, neg, ls).flatten().numpy()
    BINARY_PROB_TEMP = 14.0; p_tmp = binary_class_probs_gpu(emb, pos, neg, ls).flatten().numpy()
    BINARY_PROB_TEMP = _save
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].hist(p_raw, bins=40, color="#d62728"); ax[0].set_title(f"(a) raw logit_scale (sat={_saturation_fraction(p_raw):.2f})")
    ax[1].hist(p_tmp, bins=40, color="#1f77b4"); ax[1].set_title(f"(b) temp=14 (sat={_saturation_fraction(p_tmp):.2f})")
    for a in ax: a.set_xlabel("P(present)"); a.set_xlim(0, 1)
    plt.tight_layout(); p2 = f"{out_dir}/selftest_saturation.png"
    fig.savefig(p2, dpi=120, bbox_inches="tight"); plt.close(fig); paths.append(p2)
    print(f"[selftest-viz] wrote {len(paths)} synthetic figures: {paths}")
    return paths

if RUN_SELFTEST:
    _smoke_ok = run_internal_smoke_test()
    if RUN_SELFTEST_VIZ:
        try:
            render_selftest_visuals()
        except Exception as _e:
            print(f"[selftest-viz] skipped ({_e})")

# ==================== SECTION 7.3 — Stage 2 + 3 driver ====================
if RUN_STAGE23:
    print(f"\nDevice: {Device}")
    if USE_CUDA:
        print(f"GPU : {torch.cuda.get_device_name(0)} | "
              f"VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    if DATASET == "NIH":
        print("\n" + "=" * 70 + "\nREADING CSV\n" + "=" * 70)
        train_df = read_pairs_csv(TRAIN_CSV); val_df = read_pairs_csv(VAL_CSV)
        test_df = read_pairs_csv(TEST_CSV) if (RUN_TEST_EVAL and TEST_CSV and os.path.exists(TEST_CSV)) else None
        print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | "
              f"Test: {len(test_df):,}" if test_df is not None else
              f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: (skipped)")
        _view_default = "PA"; _view_from_report = False
    else:
        print("\n" + "=" * 70 + "\nREADING MIMIC-CXR COMBINED CSV\n" + "=" * 70)
        assert os.path.exists(COMBINED_CSV), f"CSV not found: {COMBINED_CSV}"
        full_df = pd.read_csv(COMBINED_CSV)
        def _norm_split(s):
            s = str(s).strip().lower()
            if s in ("train", "training"): return "train"
            if s in ("validate", "valid", "val", "validation", "dev"): return "val"
            if s in ("test", "testing", "eval"): return "test"
            return s
        full_df["_split"] = full_df["split"].apply(_norm_split) if "split" in full_df.columns else "train"
        train_df = full_df[full_df["_split"] == "train"].reset_index(drop=True)
        val_df = full_df[full_df["_split"] == "val"].reset_index(drop=True)
        test_df = (full_df[full_df["_split"] == "test"].reset_index(drop=True)
                   if RUN_TEST_EVAL else None)
        if len(val_df) == 0 and len(train_df) > 0:
            n_val = max(1, int(len(train_df) * VAL_CARVE_FRAC))
            val_df = train_df.sample(n_val, random_state=SEED)
            train_df = train_df.drop(val_df.index).reset_index(drop=True); val_df = val_df.reset_index(drop=True)
        _view_default = "AUTO"; _view_from_report = True

    # [FIX 4] SINGLE-LABEL cohort (exactly 1 of 5 positive) applied to every split.
    train_df = apply_cohort_filter(train_df, "Train")
    val_df   = apply_cohort_filter(val_df, "Val")
    if test_df is not None: test_df = apply_cohort_filter(test_df, "Test")

    # [FIX 11 — implemented] Load the GT-box image names ONCE and force them into
    # the val/test cohort during size-sampling (sample_keeping_gt), so the
    # localization eval set is never randomly sampled away.
    _GT_NAMES = set(load_bbox_gt(BBOX_CSV).keys()) if BBOX_CSV else set()
    if MINI_REAL_RUN:
        train_df = train_df.sample(min(len(train_df), MINI_TRAIN_N), random_state=SEED).reset_index(drop=True)
        val_df   = sample_keeping_gt(val_df, MINI_VAL_N, _GT_NAMES, "Val")
        if test_df is not None:
            test_df = sample_keeping_gt(test_df, MINI_VAL_N, _GT_NAMES, "Test")
        print(f"MINI_REAL_RUN: train={len(train_df)} val={len(val_df)}")
    else:
        if MAX_TRAIN_IMAGES is not None and len(train_df) > MAX_TRAIN_IMAGES:
            train_df = train_df.sample(MAX_TRAIN_IMAGES, random_state=SEED).reset_index(drop=True)
        val_df = sample_keeping_gt(val_df, MAX_VAL_IMAGES, _GT_NAMES, "Val")     # [FIX 11]
        if test_df is not None:
            test_df = sample_keeping_gt(test_df, MAX_TEST_IMAGES, _GT_NAMES, "Test")  # [FIX 11]

    print(f"Processing - Train: {len(train_df):,} | Val: {len(val_df):,}"
          + (f" | Test: {len(test_df):,}" if test_df is not None else ""))

    print("\n" + "=" * 70 + "\nSTAGE 2 - RULE-BASED QUERY PLANS\n" + "=" * 70)
    VOCAB = mine_vocab_from_reports(train_df)
    train_plans = run_stage2(train_df, VOCAB, "Stage 2 - train", USE_LABEL_BACKUP_TRAIN,
                             view_default=_view_default, view_from_report=_view_from_report)
    val_plans = run_stage2(val_df, VOCAB, "Stage 2 - val", USE_LABEL_BACKUP_VAL,
                           view_default=_view_default, view_from_report=_view_from_report)
    test_plans = (run_stage2(test_df, VOCAB, "Stage 2 - test", USE_LABEL_BACKUP_VAL,
                             view_default=_view_default, view_from_report=_view_from_report)
                  if test_df is not None else [])
    split_stats(train_plans, "Train"); split_stats(val_plans, "Val")
    if test_plans: split_stats(test_plans, "Test")
    with open(STAGE2_OUT, "wb") as f:
        pickle.dump({"train": train_plans, "val": val_plans, "test": test_plans, "vocab": VOCAB}, f, protocol=4)

    print("\n" + "=" * 70 + "\nSTAGE 3 - PATCH DISCOVERY (binary prompts + global ensemble)\n" + "=" * 70)
    chexzero, cz_prep = load_chexzero(CHEXZERO_CKPT, Device)
    ctm_gpu, ls_gpu = build_class_text_matrix(chexzero, Device)
    POS_MAT, NEG_MAT = build_binary_prompt_matrices(chexzero, Device)   # [v14 FIX A]
    train_plans = fill_query_vectors(train_plans, chexzero, Device, ctm=ctm_gpu)
    val_plans = fill_query_vectors(val_plans, chexzero, Device, ctm=ctm_gpu)
    if test_plans:
        test_plans = fill_query_vectors(test_plans, chexzero, Device, ctm=ctm_gpu)
    gradcam = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu)

    # [HYBRID] The "WHERE": load the frozen RadJEPA SSL backbone, train ONLY its
    # linear probe on NIH train, and build a probe-driven Grad-CAM. CheXzero above
    # is loaded frozen (zero-shot) and is NEVER trained. If RadJEPA can't load the
    # notebook proceeds CheXzero-only (RADJEPA_AVAILABLE=False forces hybrid off).
    radjepa_backbone, radjepa_prep, _rj_dim, _rj_prefix = load_radjepa(RADJEPA_CKPT, Device)
    radjepa_probe = train_radjepa_probe(radjepa_backbone, radjepa_prep, _rj_prefix,
                                        train_df, Device, save_path=RADJEPA_PROBE_PT)
    radjepa_gc = (RadJEPAGradCAM(radjepa_backbone, radjepa_prep, radjepa_probe, Device, _rj_prefix)
                  if (RADJEPA_AVAILABLE and radjepa_probe is not None) else None)
    print(f"[stage3] hybrid causal gate ACTIVE: "
          f"{bool(ENABLE_RADJEPA_HYBRID and RADJEPA_AVAILABLE and radjepa_gc is not None)} "
          f"(RadJEPA where + frozen CheXzero what)")

    train_results, val_results, test_results = [], [], []
    train_completed = val_completed = False
    try:
        train_results, train_completed, _ = run_stage3(
            train_plans, "train", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
            deadline=_DEADLINE, out_dir=OUT_DIR, pos_mat=POS_MAT, neg_mat=NEG_MAT,
            radjepa_gc=radjepa_gc)
        if train_completed and _budget_left_h(_DEADLINE) > 0.15:
            val_results, val_completed, _ = run_stage3(
                val_plans, "val", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
                deadline=_DEADLINE, out_dir=OUT_DIR, pos_mat=POS_MAT, neg_mat=NEG_MAT,
                radjepa_gc=radjepa_gc)
        else:
            print("Skipping val this session - re-run to resume.")
        if test_plans and val_completed and _budget_left_h(_DEADLINE) > 0.15:
            test_results, _, _ = run_stage3(
                test_plans, "test", gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
                deadline=_DEADLINE, out_dir=OUT_DIR, pos_mat=POS_MAT, neg_mat=NEG_MAT,
                radjepa_gc=radjepa_gc)
    finally:
        gradcam.remove_hooks()
        if radjepa_gc is not None: radjepa_gc.remove_hooks()

    # [FIX 3] splits are kept SEPARATE — never pooled for metrics.
    rd = {"train": train_results, "val": val_results, "test": test_results}
    print("\n" + "=" * 70 + "\nSTAGE 3 SUMMARY\n" + "=" * 70)
    summarize_stage3(train_results, "Train")
    if val_results: summarize_stage3(val_results, "Val")
    if test_results: summarize_stage3(test_results, "Test")
    print(f"\nCounters: {dict(_COUNTERS)}")

    with open(STAGE3_RESULTS, "wb") as f: pickle.dump(rd, f, protocol=4)
    cp, sp, op = split_embeddings_three_ways(rd)
    torch.save(cp, STAGE3_CAUSAL_PT); torch.save(sp, STAGE3_SPIN_PT); torch.save(op, STAGE3_SPOUT_PT)
    with open(STAGE3_META, "wb") as f:
        pickle.dump({"version": VERSION, "dataset": DATASET_NAME, "cohort": COHORT_MODE,
                     "target_classes": TARGET_CLASSES, "use_binary_prompts": USE_BINARY_PROMPTS,
                     "global_ensemble_alpha": GLOBAL_ENSEMBLE_ALPHA, "binary_prob_temp": BINARY_PROB_TEMP,
                     "allow_fallback": ALLOW_FALLBACK,
                     "hybrid": {"enabled": bool(ENABLE_RADJEPA_HYBRID and RADJEPA_AVAILABLE),
                                "radjepa_available": RADJEPA_AVAILABLE, "spatial_thr": HYBRID_SPATIAL_THR,
                                "semantic_norm_thr": HYBRID_SEMANTIC_NORM_THR,
                                "keep_persistence_and": RADJEPA_KEEP_PERSISTENCE_AND,
                                "radjepa_ckpt": RADJEPA_CKPT, "radjepa_arch": RADJEPA_ARCH},
                     "counts": {"causal": cp["n"], "spur_in": sp["n"], "spur_out": op["n"]},
                     "counters": dict(_COUNTERS)}, f, protocol=4)
    print(f"Saved causal={cp['n']:,} spur_in={sp['n']:,} spur_out={op['n']:,}")

    # [HYBRID / next-round] Save the COMPLETE per-image records + flattened patch
    # bank: image/report paths, split, sex/age/view/patient-id/raw columns, the 5
    # GT labels, queried classes, and every causal/spurious patch with all scores
    # (semantic, zeroshot, gradcam, RadJEPA spatial, combined + normalized), gate,
    # and fp16 embedding. This is the artifact the CheXpert transfer round loads.
    save_full_records(rd)

    # ---------------- SECTION 7.4 — A* metrics on a HELD-OUT split ----------
    # [FIX 3] report on ONE held-out split (test if produced, else val) — never pooled.
    if test_results:
        EVAL_SPLIT = ("test",); eval_plans = test_plans; eval_results = test_results
    else:
        EVAL_SPLIT = ("val",);  eval_plans = val_plans;  eval_results = val_results
    print(f"\n[eval] primary metrics on held-out split: {EVAL_SPLIT[0]} "
          f"(cohort={COHORT_MODE}, n_plans={len(eval_plans)})")
    plans_by_name = {p.image_name: p for p in eval_plans}
    GT_LUT = build_gt_lookup(train_df, val_df, test_df)   # *dfs — test_df may be None
    BBOX_GT = load_bbox_gt(BBOX_CSV)
    # whole-image baseline FIRST, on the same cohort (the missing experiment)
    _base_df, _name_scores = global_baseline_metrics(
        eval_plans, GT_LUT, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
        pos_mat=POS_MAT, neg_mat=NEG_MAT, splits_label=EVAL_SPLIT[0])
    # the circularity confound, made explicit
    report_parser_agreement(eval_plans, GT_LUT)
    # de-circularised image metric (restricted to queried images) + saturation
    _img_df_q, _img_df_a, _img_raw = image_level_metrics(
        rd, GT_LUT, plans_by_name=plans_by_name, splits=EVAL_SPLIT)
    # A* localization on the held-out split, strict causal-only, Wilson CI
    _loc_df = evaluate_localization(rd, BBOX_GT, splits=EVAL_SPLIT, causal_only=True)
    _live_gc = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu)
    try:
        plot_causal_gradcam_grid(eval_results, plans_by_name, _live_gc,
                                 save_path=f"{OUT_DIR}/causal_gradcam_grid.png")
    finally:
        _live_gc.remove_hooks()
    plot_bboxes_by_bucket(eval_results, n_examples=4, save_path=f"{OUT_DIR}/bboxes_by_bucket.png")
    # ---- reviewer plot suite ----
    plot_per_class_curves(_img_raw, save_path=f"{OUT_DIR}/per_class_curves.png")           # #6-#10
    plot_embedding_separation(cp, sp, op, save_path=f"{OUT_DIR}/embedding_separation.png") # #11
    plot_localization_ci(_loc_df, save_path=f"{OUT_DIR}/localization_ci.png")              # #13
    plot_persistence_diag(save_path=f"{OUT_DIR}/persistence_diag.png")                     # #14
    _eval_df = test_df if (EVAL_SPLIT[0] == "test" and test_df is not None) else val_df
    plot_fairness_subgroups(_eval_df, _name_scores, GT_LUT,
                            save_path=f"{OUT_DIR}/fairness_subgroups.png")                 # #12
    print("\nStage 3 + A* reporting complete. Figures in:", OUT_DIR)


## Cell 8  ablations (factorial + BH-FDR)

Ablation grid on the **held-out VAL split only**, resumable, with negative controls. Includes the **full factorial** over the three score components (arm group `M`: sem/prob/gc, every pair, and all three), the honest `G:with_fallback` and `K:persistence_off` arms, temperature sweep, and **Benjamini–Hochberg** correction across the arm family (**FIX 10**). Emits the localization-quality bars and the paired-ΔAUROC forest plot.

In [ ]:
# =============================================================================
# CELL 8 / 8  —PATCH-MINING ABLATIONS  +  NEGATIVE CONTROLS  +  REPORT
# =============================================================================
# [FIX 3] the ablation pool is the HELD-OUT VAL split ONLY (never pooled with
# train). [FIX 10] arm-vs-FULL comparisons are Benjamini-Hochberg corrected.
# [FIX 6] fallback is default-OFF; arm G turns it ON to quantify its (gate-
# bypassing) uplift. [FIX 8] arm J zeroes the class-prompt blend to test how much
# ranking signal the correlated semantic term actually adds.
if RUN_ABLATIONS:
    # ---------------- SECTION 8.1 — config + guards -----------------------
    ABL_SUBSET_N             = 400
    ABL_SEEDS                = [42, 1, 7]
    ABL_PRIOR_THR            = 0.10
    ABL_TOPK                 = 3
    ABL_WALLCLOCK_BUDGET_MIN = 180
    ABL_BOOTSTRAP            = 1000
    ABL_PAIRED_BOOT          = 2000
    ABL_CTRL_RANDOM_K        = 8
    ABL_BH_ALPHA             = 0.05
    ABL_OUT_CSV              = f"{OUT_DIR}/patch_ablations_{VERSION}.csv"
    ABL_PERCLASS_CSV         = f"{OUT_DIR}/patch_ablations_{VERSION}_perclass.csv"
    ABL_PAIRED_CSV           = f"{OUT_DIR}/patch_ablations_{VERSION}_paired.csv"

    _ABL_T0 = time.time()
    def _abl_left_min():
        return ABL_WALLCLOCK_BUDGET_MIN - (time.time() - _ABL_T0) / 60.0

    try:
        chexzero, cz_prep, ctm_gpu, ls_gpu
    except NameError:
        chexzero, cz_prep = load_chexzero(CHEXZERO_CKPT, Device)
        ctm_gpu, ls_gpu   = build_class_text_matrix(chexzero, Device)
        train_plans = fill_query_vectors(train_plans, chexzero, Device, ctm=ctm_gpu)
        val_plans   = fill_query_vectors(val_plans,   chexzero, Device, ctm=ctm_gpu)
    try:
        POS_MAT, NEG_MAT
    except NameError:
        POS_MAT, NEG_MAT = build_binary_prompt_matrices(chexzero, Device)
    # [HYBRID] ensure a RadJEPA GradCAM so the FULL ablation config uses the same
    # RadJEPA-where + frozen-CheXzero-what gate as Stage 3. Reuse the driver's
    # object if present; else rebuild from the saved probe. Falls back to None
    # (CheXzero-only) when RadJEPA is unavailable.
    try:
        radjepa_gc
    except NameError:
        _abl_rj_bb, _abl_rj_prep, _abl_rj_dim, _abl_rj_prefix = load_radjepa(RADJEPA_CKPT, Device)
        _abl_rj_probe = train_radjepa_probe(_abl_rj_bb, _abl_rj_prep, _abl_rj_prefix,
                                            train_df, Device, save_path=RADJEPA_PROBE_PT)
        radjepa_gc = (RadJEPAGradCAM(_abl_rj_bb, _abl_rj_prep, _abl_rj_probe, Device, _abl_rj_prefix)
                      if (RADJEPA_AVAILABLE and _abl_rj_probe is not None) else None)

    # [FIX 3] VAL ONLY — never pooled with train. (v14 used val_plans+train_plans.)
    _ALL_PLANS = [p for p in val_plans
                  if p.query_items and any(qi.query_vector is not None for qi in p.query_items)]
    def _subset_for_seed(seed):
        rng = np.random.RandomState(seed)   # subset selection only (choice)
        if len(_ALL_PLANS) > ABL_SUBSET_N:
            idx = rng.choice(len(_ALL_PLANS), ABL_SUBSET_N, replace=False)
            return [_ALL_PLANS[i] for i in sorted(idx)]
        return list(_ALL_PLANS)
    print(f"Ablation pool (VAL only): {len(_ALL_PLANS)} plans | seeds={ABL_SEEDS} | n={ABL_SUBSET_N}")

    # ---------------- SECTION 8.2 — GT + metric helpers -------------------
    GT_LUT = build_gt_lookup(train_df, val_df)
    BBOX_GT = load_bbox_gt(BBOX_CSV)
    print(f"GT label lookup: {len(GT_LUT):,} images | GT-box images: {len(BBOX_GT):,}")

    def _class_prior_hmaps(plan):
        hm = {}
        for qi in plan.query_items:
            if qi.anatomical_prior and qi.pathology in TARGET_CLASSES and qi.pathology not in hm:
                p = qi.anatomical_prior
                h = make_gaussian_heatmap_gpu(p.center_x, p.center_y, p.sigma_x, p.sigma_y,
                                              IMAGE_SIZE, IMAGE_SIZE, Device)
                mx = h.max(); hm[qi.pathology] = (h / mx) if mx > 0 else h
        return hm

    def _auroc(yt, ys):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2:
            return float("nan"), int((yt == 1).sum()), int((yt == 0).sum())
        try:
            return float(roc_auc_score(yt, ys)), int((yt == 1).sum()), int((yt == 0).sum())
        except Exception:
            return float("nan"), int((yt == 1).sum()), int((yt == 0).sum())
    def _auprc(yt, ys):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2: return float("nan")
        try: return float(average_precision_score(yt, ys))
        except Exception: return float("nan")
    def _auroc_ci(yt, ys, n_boot=ABL_BOOTSTRAP, seed=0):
        yt, ys = np.asarray(yt), np.asarray(ys)
        if len(np.unique(yt)) != 2: return (float("nan"), float("nan"))
        r = np.random.default_rng(seed); vals = []; n = len(yt)
        for _ in range(n_boot):
            i = r.integers(0, n, n)
            if len(np.unique(yt[i])) == 2:
                try: vals.append(roc_auc_score(yt[i], ys[i]))
                except Exception: pass
        if not vals: return (float("nan"), float("nan"))
        return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))

    # ---------------- SECTION 8.3 — controls (default_rng) -----------------
    def _rand_boxes(n, rng):
        out = []
        for _ in range(n):
            side = min(int(rng.choice([64, 128, 256])), IMAGE_SIZE)
            x1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            y1 = int(rng.integers(0, max(1, IMAGE_SIZE - side)))
            out.append((x1, y1, x1 + side, y1 + side))
        return out
    def _peak_box(hmap_gpu, side=128):
        idx = int(torch.argmax(hmap_gpu).item()); H = hmap_gpu.shape[-1]
        cy, cx = idx // H, idx % H
        return _clamp_box_square(cx - side // 2, cy - side // 2, side)

    def _control_scores(plans, mode, gradcam, seed):
        rng = np.random.default_rng(seed + 991)
        scores = {c: [] for c in TARGET_CLASSES}; gts = {c: [] for c in TARGET_CLASSES}
        for plan in plans:
            gt = GT_LUT.get(os.path.basename(str(plan.image_name)))
            if gt is None: continue
            try:
                pil = Image.open(plan.image_path).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            except Exception:
                continue
            comp = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, Device)
            queried = {qi.pathology: qi for qi in plan.query_items
                       if qi.pathology in TARGET_CLASSES and not qi.negated and qi.query_vector is not None}
            best = {c: 0.0 for c in TARGET_CLASSES}
            if mode == "random":
                boxes = [b for b in _rand_boxes(ABL_CTRL_RANDOM_K, rng) if b is not None]
                if boxes:
                    embs = encode_patch_batch_gpu([pil.crop(b) for b in boxes], chexzero, cz_prep, Device)
                    probs = class_probs_gpu(embs, ctm_gpu, ls_gpu, POS_MAT, NEG_MAT)
                    for c in TARGET_CLASSES:
                        ti = TARGET_CLASSES.index(c); best[c] = float(probs[:, ti].max().item())
            else:
                for c, qi in queried.items():
                    ti = TARGET_CLASSES.index(c)
                    gcam = gradcam.compute_combined(pil, qi.query_vector, ti) if ENABLE_GRADCAM else None
                    hmap = gcam if gcam is not None else comp; b = _peak_box(hmap)
                    if b is None: continue
                    embs = encode_patch_batch_gpu([pil.crop(b)], chexzero, cz_prep, Device)
                    probs = class_probs_gpu(embs, ctm_gpu, ls_gpu, POS_MAT, NEG_MAT)
                    best[c] = float(probs[0, ti].item())
            for c in TARGET_CLASSES:
                lv = gt.get(c, float("nan"))
                if lv == lv: scores[c].append(best[c]); gts[c].append(lv)
        return scores, gts

    # ---------------- SECTION 8.4 — one-config evaluator ------------------
    def _eval_current_config(plans, gradcam, label="", with_ci=False, seed=0):
        t0 = time.time(); n_img = len(plans)
        n_causal = n_true = n_fb = n_img_causal = n_in_prior = n_leak = 0
        sims = []; n_spin = 0; n_spout = 0; band_hits = 0; band_tot = 0
        scores_max = {c: [] for c in TARGET_CLASSES}; scores_topk = {c: [] for c in TARGET_CLASSES}
        gts = {c: [] for c in TARGET_CLASSES}
        pc = {c: dict(causal=0, true=0, fb=0, prior=0, leak=0, spin=0, img=0, queried=0, sims=[])
              for c in TARGET_CLASSES}
        loc = {c: dict(n=0, hit=0, iou10=0, iou25=0) for c in TARGET_CLASSES}
        for plan in plans:
            res = discover_patches_for_plan(plan, gradcam, chexzero, cz_prep, Device, ctm_gpu, ls_gpu,
                                            "abl", pos_mat=POS_MAT, neg_mat=NEG_MAT,
                                            radjepa_gc=radjepa_gc)
            cps = res.causal_patches
            chm = get_composite_heatmap_gpu(plan, IMAGE_SIZE, IMAGE_SIZE, Device)
            cpr = _class_prior_hmaps(plan)
            queried = {qi.pathology for qi in plan.query_items
                       if qi.pathology in TARGET_CLASSES and not qi.negated}
            for c in queried: pc[c]["queried"] += 1
            if cps: n_img_causal += 1
            n_causal += len(cps); imgs_c = set()
            for p in cps:
                sims.append(p.semantic_score); band_tot += 1
                if 0.24 <= p.semantic_score <= 0.34: band_hits += 1
                if p.selection_source == "fallback": n_fb += 1
                else: n_true += 1
                cx = min((p.box[0] + p.box[2]) // 2, IMAGE_SIZE - 1)
                cy = min((p.box[1] + p.box[3]) // 2, IMAGE_SIZE - 1)
                if float(chm[cy, cx].item()) >= ABL_PRIOR_THR: n_in_prior += 1
                else: n_leak += 1
                c = p.pathology
                if c in pc:
                    pc[c]["causal"] += 1
                    pc[c]["fb" if p.selection_source == "fallback" else "true"] += 1
                    pc[c]["sims"].append(p.semantic_score)
                    hmc = cpr.get(c)
                    if hmc is not None and float(hmc[cy, cx].item()) >= ABL_PRIOR_THR: pc[c]["prior"] += 1
                    else: pc[c]["leak"] += 1
                    imgs_c.add(c)
            for c in imgs_c: pc[c]["img"] += 1
            for p in res.spurious_patches:
                if p.spurious_source == "outside_anatomy": n_spout += 1
                else:
                    n_spin += 1
                    if p.pathology in pc: pc[p.pathology]["spin"] += 1
            gbx = BBOX_GT.get(os.path.basename(str(plan.image_name)))
            if gbx:
                for cc in _BBOX_LABEL_MAP:
                    cgts = [g[1] for g in gbx if g[0] == cc]
                    if not cgts: continue
                    # [FIX 6] strict causal-only: fallback patches are never counted
                    cand = [p for p in cps if p.pathology == cc and p.selection_source != "fallback"]
                    if not cand: continue
                    bb = max(cand, key=lambda p: p.combined_score).box
                    loc[cc]["n"] += 1
                    if any(_center_in(bb, g) for g in cgts): loc[cc]["hit"] += 1
                    mi = max(_iou(bb, g) for g in cgts)
                    if mi >= 0.10: loc[cc]["iou10"] += 1
                    if mi >= 0.25: loc[cc]["iou25"] += 1
            gt = GT_LUT.get(os.path.basename(str(plan.image_name)))
            if gt is not None:
                probs_by_c = {c: [] for c in TARGET_CLASSES}
                for p in cps:
                    if p.pathology in probs_by_c: probs_by_c[p.pathology].append(float(p.zeroshot_prob))
                for c in TARGET_CLASSES:
                    lv = gt.get(c, float("nan"))
                    if lv == lv:
                        pr = sorted(probs_by_c[c], reverse=True)
                        scores_max[c].append(pr[0] if pr else 0.0)
                        scores_topk[c].append(float(np.mean(pr[:ABL_TOPK])) if pr else 0.0)
                        gts[c].append(lv)
        pc_rows = []
        for c in TARGET_CLASSES:
            au, npos, nneg = _auroc(gts[c], scores_max[c]); au_tk, _, _ = _auroc(gts[c], scores_topk[c])
            ap = _auprc(gts[c], scores_max[c])
            lo, hi = _auroc_ci(gts[c], scores_max[c], seed=seed) if with_ci else (float("nan"), float("nan"))
            q = max(pc[c]["queried"], 1); ncz = max(pc[c]["causal"], 1)
            pc_rows.append({"label": label, "seed": seed, "class": c, "auroc": au, "auroc_topk": au_tk,
                            "auprc": ap, "ci_lo": lo, "ci_hi": hi, "support_pos": npos, "support_neg": nneg,
                            "causal": pc[c]["causal"], "causal_per_qimg": pc[c]["causal"]/q,
                            "pct_qimg_causal": 100.0*pc[c]["img"]/q, "abstain_pct": 100.0*(1.0 - pc[c]["img"]/q),
                            "true_pct": 100.0*pc[c]["true"]/ncz, "fallback_pct": 100.0*pc[c]["fb"]/ncz,
                            "prior_pct": 100.0*pc[c]["prior"]/ncz, "leak_pct": 100.0*pc[c]["leak"]/ncz,
                            "mean_sim": float(np.mean(pc[c]["sims"])) if pc[c]["sims"] else 0.0,
                            "spur_in": pc[c]["spin"], "loc_n": loc[c]["n"],
                            "pointing_game": (loc[c]["hit"]/loc[c]["n"]) if loc[c]["n"] else float("nan"),
                            "iou@0.1":  (loc[c]["iou10"]/loc[c]["n"]) if loc[c]["n"] else float("nan"),
                            "iou@0.25": (loc[c]["iou25"]/loc[c]["n"]) if loc[c]["n"] else float("nan")})
        macro = float(np.nanmean([r["auroc"] for r in pc_rows]))
        macro_tk = float(np.nanmean([r["auroc_topk"] for r in pc_rows]))
        macro_ap = float(np.nanmean([r["auprc"] for r in pc_rows]))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            macro_pg    = float(np.nanmean([r["pointing_game"] for r in pc_rows]))
            macro_iou25 = float(np.nanmean([r["iou@0.25"] for r in pc_rows]))
        overall = {"label": label, "seed": seed, "causal_per_img": n_causal / max(n_img, 1),
                   "pct_img_causal": 100.0 * n_img_causal / max(n_img, 1),
                   "abstain_pct": 100.0 * (1.0 - n_img_causal / max(n_img, 1)),
                   "true_causal_pct": 100.0 * n_true / max(n_causal, 1),
                   "fallback_pct": 100.0 * n_fb / max(n_causal, 1),
                   "prior_overlap_pct": 100.0 * n_in_prior / max(n_causal, 1),
                   "leak_pct": 100.0 * n_leak / max(n_causal, 1),
                   "band_occ_pct": 100.0 * band_hits / max(band_tot, 1),
                   "mean_sim": float(np.mean(sims)) if sims else 0.0, "spur_in": n_spin, "spur_out": n_spout,
                   "macroAUROC": macro, "macroAUROC_topk": macro_tk, "macroAUPRC": macro_ap,
                   "macro_pointing_game": macro_pg, "macro_iou@0.25": macro_iou25,
                   "n_auc_classes": int(np.sum(~np.isnan([r["auroc"] for r in pc_rows]))),
                   "sec_img": (time.time() - t0) / max(n_img, 1)}
        raw = {"scores_max": scores_max, "scores_topk": scores_topk, "gts": gts}
        return overall, pc_rows, raw

    # ---------------- SECTION 8.5 — paired dAUROC bootstrap ----------------
    def _paired_delta_vs_full(raw_cfg, raw_full, n_boot=ABL_PAIRED_BOOT, seed=0):
        r = np.random.default_rng(seed + 4242); deltas = []; cls_arrays = []
        for c in TARGET_CLASSES:
            g = np.asarray(raw_full["gts"][c]); sc = np.asarray(raw_cfg["scores_max"][c]); sf = np.asarray(raw_full["scores_max"][c])
            if len(g) == len(sc) == len(sf) and len(np.unique(g)) == 2: cls_arrays.append((g, sc, sf))
        if not cls_arrays: return (float("nan"),)*4
        for _ in range(n_boot):
            per_cls = []
            for g, sc, sf in cls_arrays:
                n = len(g); i = r.integers(0, n, n)
                if len(np.unique(g[i])) != 2: continue
                try: per_cls.append(roc_auc_score(g[i], sc[i]) - roc_auc_score(g[i], sf[i]))
                except Exception: pass
            if per_cls: deltas.append(float(np.mean(per_cls)))
        if not deltas: return (float("nan"),)*4
        deltas = np.asarray(deltas)
        # two-sided bootstrap p as 2*min tail mass (used for BH input)
        p_two = float(2.0 * min(np.mean(deltas <= 0.0), np.mean(deltas >= 0.0)))
        return (float(deltas.mean()), float(np.percentile(deltas, 2.5)),
                float(np.percentile(deltas, 97.5)), min(1.0, p_two))

    # ---------------- SECTION 8.6 — apply/restore + main loop --------------
    _ABL_KEYS = ["CAUSAL_CONTAIN_MODE", "CAUSAL_MASK_COVER_FRAC", "CAUSAL_MASK_DILATE_PX",
                 "PERSISTENCE_TOP_K", "PERSISTENCE_N_LEVELS", "PERSISTENCE_ENABLED",
                 "SEMANTIC_THRESHOLD_PER_CLASS", "SEMANTIC_THRESHOLD", "ALLOW_FALLBACK",
                 "GLOBAL_ENSEMBLE_ALPHA", "USE_BINARY_PROMPTS", "BINARY_PROB_TEMP",
                 "CAUSAL_QUERY_CLASS_BLEND", "SCORE_ACTIVE",
                 "ENABLE_RADJEPA_HYBRID", "HYBRID_SPATIAL_THR", "RADJEPA_KEEP_PERSISTENCE_AND"]
    _BASE = {k: copy.deepcopy(globals()[k]) for k in _ABL_KEYS}
    def _apply(ov):
        for k in _ABL_KEYS: globals()[k] = copy.deepcopy(_BASE[k])
        if "sem_scale" in ov:
            s = ov["sem_scale"]
            globals()["SEMANTIC_THRESHOLD_PER_CLASS"] = {c: v*s for c, v in _BASE["SEMANTIC_THRESHOLD_PER_CLASS"].items()}
            globals()["SEMANTIC_THRESHOLD"] = _BASE["SEMANTIC_THRESHOLD"] * s
        if "sem_abs" in ov:
            a = ov["sem_abs"]
            globals()["SEMANTIC_THRESHOLD_PER_CLASS"] = {c: a for c in _BASE["SEMANTIC_THRESHOLD_PER_CLASS"]}
            globals()["SEMANTIC_THRESHOLD"] = a
        for k, v in ov.items():
            if k in ("sem_scale", "sem_abs"): continue
            globals()[k] = v
    def _restore():
        for k in _ABL_KEYS: globals()[k] = copy.deepcopy(_BASE[k])

    # A*-LEVEL ABLATION GRID — each arm isolates ONE design decision.
    CONFIGS = [
        ("FULL",                {}),
        # A — causal-containment rule
        ("A:contain=center",    {"CAUSAL_CONTAIN_MODE": "center"}),
        ("A:contain=peak",      {"CAUSAL_CONTAIN_MODE": "peak"}),
        ("A:contain=cover",     {"CAUSAL_CONTAIN_MODE": "cover"}),
        # B/C/D/E — smoothed-CAM component selection (honest name; see FIX 5)
        ("B:topK=1",            {"PERSISTENCE_TOP_K": 1}),
        ("B:topK=3",            {"PERSISTENCE_TOP_K": 3}),
        ("C:levels=16",         {"PERSISTENCE_N_LEVELS": 16}),
        ("C:levels=48",         {"PERSISTENCE_N_LEVELS": 48}),
        ("D:dilate=0",          {"CAUSAL_MASK_DILATE_PX": 0}),
        ("D:dilate=12",         {"CAUSAL_MASK_DILATE_PX": 12}),
        ("E:cover=0.20",        {"CAUSAL_MASK_COVER_FRAC": 0.20}),
        ("E:cover=0.50",        {"CAUSAL_MASK_COVER_FRAC": 0.50}),
        ("K:persistence_off",   {"PERSISTENCE_ENABLED": False}),   # [FIX 5] drop the PH gate entirely
        # F — semantic threshold
        ("F:sem_abs=0.24",      {"sem_abs": 0.24}),
        ("F:sem_abs=0.30",      {"sem_abs": 0.30}),
        ("F:sem_abs=0.34",      {"sem_abs": 0.34}),
        # G — [FIX 6] fallback is OFF by default; this arm turns it ON to quantify
        # the (gate-bypassing) uplift it was silently providing in v14.
        ("G:with_fallback",     {"ALLOW_FALLBACK": True}),
        # H/I — the two v14 scoring-model fixes
        ("H:5way_softmax",      {"USE_BINARY_PROMPTS": False}),
        ("I:ensemble=0.0",      {"GLOBAL_ENSEMBLE_ALPHA": 0.0}),
        ("I:ensemble=1.0",      {"GLOBAL_ENSEMBLE_ALPHA": 1.0}),
        # J — [FIX 8] zero the class-prompt blend => decorrelate semantic_score
        # from zeroshot_prob; tests how much ranking signal the sem term adds.
        ("J:decorrelate",       {"CAUSAL_QUERY_CLASS_BLEND": 0.0}),
        # L — [FIX 7] probability temperature sweep
        ("L:temp=raw",          {"BINARY_PROB_TEMP": None}),
        ("L:temp=25",           {"BINARY_PROB_TEMP": 25.0}),
        # M — FULL FACTORIAL over the 3 score components (semantic / prob / gradcam):
        # all 7 non-empty subsets. Answers "does the ablation test every combination
        # of sem/gc/prob?" — yes. M:sem+prob+gc is identical to FULL (sanity check).
        ("M:sem_only",          {"SCORE_ACTIVE": ("sem",)}),
        ("M:prob_only",         {"SCORE_ACTIVE": ("prob",)}),
        ("M:gc_only",           {"SCORE_ACTIVE": ("gc",)}),
        ("M:sem+prob",          {"SCORE_ACTIVE": ("sem", "prob")}),
        ("M:sem+gc",            {"SCORE_ACTIVE": ("sem", "gc")}),
        ("M:prob+gc",           {"SCORE_ACTIVE": ("prob", "gc")}),
        ("M:sem+prob+gc",       {"SCORE_ACTIVE": ("sem", "prob", "gc")}),
        # N — [HYBRID] the RadJEPA(where) x CheXzero(what) overlap gate. N:hybrid_off
        # reverts to the v15 CheXzero-only smoothed-CAM 'where'; the spatial-threshold
        # sweep tests how strict the RadJEPA overlap must be; N:and_persistence
        # requires BOTH RadJEPA spatial AND CLIP persistence containment.
        ("N:hybrid_off",        {"ENABLE_RADJEPA_HYBRID": False}),
        ("N:spatial=0.50",      {"HYBRID_SPATIAL_THR": 0.50}),
        ("N:spatial=0.70",      {"HYBRID_SPATIAL_THR": 0.70}),
        ("N:and_persistence",   {"RADJEPA_KEEP_PERSISTENCE_AND": True}),
    ]
    CONTROLS = ["CTRL:random", "CTRL:gcam_peak"]
    _ORDER = [lab for lab, _ in CONFIGS] + CONTROLS

    ROWS, done = [], set()
    if os.path.exists(ABL_OUT_CSV):
        prev = pd.read_csv(ABL_OUT_CSV); ROWS = prev.to_dict("records")
        done = set(prev["label"].astype(str) + "|" + prev["seed"].astype(str))
        print(f"Resuming: {len(done)} (config,seed) cells already saved")
    PC_ROWS = pd.read_csv(ABL_PERCLASS_CSV).to_dict("records") if os.path.exists(ABL_PERCLASS_CSV) else []
    PAIRED_ROWS = pd.read_csv(ABL_PAIRED_CSV).to_dict("records") if os.path.exists(ABL_PAIRED_CSV) else []
    def _save():
        for rows, path in [(ROWS, ABL_OUT_CSV), (PC_ROWS, ABL_PERCLASS_CSV), (PAIRED_ROWS, ABL_PAIRED_CSV)]:
            tmp = path + ".tmp"; pd.DataFrame(rows).to_csv(tmp, index=False); os.replace(tmp, path)

    _stopped = False; FULL_AUROC = float("nan"); full_pc = []
    for seed in ABL_SEEDS:
        if _stopped: break
        np.random.seed(seed); PLANS = _subset_for_seed(seed)
        print(f"\n### SEED {seed} - {len(PLANS)} plans ###")
        gradcam = ViTGradCAM(chexzero, cz_prep, Device, ctm_gpu, ls_gpu); raw_full = None
        try:
            key = f"FULL|{seed}"
            if key not in done:
                _apply({}); r0, pc0, raw_full = _eval_current_config(PLANS, gradcam, "FULL", with_ci=True, seed=seed)
                ROWS.append(r0); PC_ROWS += pc0; done.add(key); _save()
                if seed == ABL_SEEDS[0]: FULL_AUROC = float(r0["macroAUROC"]); full_pc = pc0
                print(f"FULL[{seed}] AUROC(max)={r0['macroAUROC']:.4f} AUPRC={r0['macroAUPRC']:.4f} "
                      f"| abstain={r0['abstain_pct']:.1f}%")
            else:
                _apply({}); _, _, raw_full = _eval_current_config(PLANS, gradcam, "FULL", with_ci=False, seed=seed)
                if seed == ABL_SEEDS[0]:
                    r0 = next(r for r in ROWS if str(r["label"]) == "FULL" and int(r["seed"]) == seed)
                    FULL_AUROC = float(r0["macroAUROC"])
                    full_pc = [r for r in PC_ROWS if str(r["label"]) == "FULL" and int(r["seed"]) == seed]
            for lab, ov in CONFIGS:
                if lab == "FULL": continue
                key = f"{lab}|{seed}"
                if key in done: continue
                if _abl_left_min() <= 0:
                    print(f"\nBudget reached before '{lab}' (seed {seed})."); _stopped = True; break
                _apply(ov)
                try:
                    r, pcr, raw_c = _eval_current_config(PLANS, gradcam, lab, with_ci=False, seed=seed)
                finally:
                    _restore()
                dmean, dlo, dhi, p_two = _paired_delta_vs_full(raw_c, raw_full, seed=seed)
                PAIRED_ROWS.append({"label": lab, "seed": seed, "d_macroAUROC": dmean,
                                    "d_lo": dlo, "d_hi": dhi, "p_two": p_two})
                ROWS.append(r); PC_ROWS += pcr; done.add(key)
                print(f"  {lab:<20} AUROC={r['macroAUROC']:.4f} dvsFULL={dmean:+.4f} "
                      f"[{dlo:+.4f},{dhi:+.4f}] p={p_two:.3f} | abstain={r['abstain_pct']:.1f}%")
                _save()
            for ctrl in CONTROLS:
                key = f"{ctrl}|{seed}"
                if key in done or _stopped: continue
                mode = "random" if ctrl.endswith("random") else "gcam_peak"
                cs, cg = _control_scores(PLANS, mode, gradcam, seed); rows_c = []
                for c in TARGET_CLASSES:
                    au, npos, nneg = _auroc(cg[c], cs[c]); ap = _auprc(cg[c], cs[c])
                    rows_c.append({"label": ctrl, "seed": seed, "class": c, "auroc": au,
                                   "auroc_topk": float("nan"), "auprc": ap, "ci_lo": float("nan"),
                                   "ci_hi": float("nan"), "support_pos": npos, "support_neg": nneg,
                                   "causal": 0, "causal_per_qimg": 0.0, "pct_qimg_causal": 0.0,
                                   "abstain_pct": float("nan"), "true_pct": float("nan"),
                                   "fallback_pct": float("nan"), "prior_pct": float("nan"),
                                   "leak_pct": float("nan"), "mean_sim": 0.0, "spur_in": 0})
                macro = float(np.nanmean([r["auroc"] for r in rows_c]))
                macro_ap = float(np.nanmean([r["auprc"] for r in rows_c]))
                ROWS.append({"label": ctrl, "seed": seed, "macroAUROC": macro, "macroAUROC_topk": float("nan"),
                             "macroAUPRC": macro_ap, "causal_per_img": 0.0, "pct_img_causal": 0.0,
                             "abstain_pct": float("nan"), "true_causal_pct": float("nan"),
                             "fallback_pct": float("nan"), "prior_overlap_pct": float("nan"),
                             "leak_pct": float("nan"), "band_occ_pct": float("nan"), "mean_sim": 0.0,
                             "spur_in": 0, "spur_out": 0, "n_auc_classes": len(TARGET_CLASSES), "sec_img": 0.0})
                PC_ROWS += rows_c; done.add(key)
                print(f"  {ctrl:<20} macroAUROC={macro:.4f} macroAUPRC={macro_ap:.4f}  [NEGATIVE CONTROL]")
                _save()
        finally:
            gradcam.remove_hooks(); _restore()

    # ---------------- master tables + BH correction + plot -----------------
    dfR = pd.DataFrame(ROWS)
    print("\n" + "=" * 118)
    print(f"{'config':<20}{'AUROCmax':>9}{'AUPRC':>8}{'PG%':>6}{'IoU25%':>7}{'true%':>7}{'fb%':>6}"
          f"{'abst%':>7}{'prior%':>8}{'leak%':>7}{'band%':>7}{'sim':>6}")
    print("=" * 118)
    def _m_of(sub, col):
        return float(np.nanmean(pd.to_numeric(sub[col], errors="coerce"))) if col in sub else float("nan")
    for lab in _ORDER:
        sub = dfR[dfR["label"].astype(str) == lab]
        if sub.empty: continue
        _m = lambda col: _m_of(sub, col)
        print(f"{lab:<20}{_m('macroAUROC'):>9.4f}{_m('macroAUPRC'):>8.4f}"
              f"{_m('macro_pointing_game')*100:>6.1f}{_m('macro_iou@0.25')*100:>7.1f}"
              f"{_m('true_causal_pct'):>7.1f}{_m('fallback_pct'):>6.1f}{_m('abstain_pct'):>7.1f}"
              f"{_m('prior_overlap_pct'):>8.1f}{_m('leak_pct'):>7.1f}{_m('band_occ_pct'):>7.1f}{_m('mean_sim'):>6.3f}")
    print("=" * 118)
    print("An arm that cannot beat CTRL:* AUROC is not causal signal; PG/IoU are the direct patch-quality test.")

    if PAIRED_ROWS:
        dfp = pd.DataFrame(PAIRED_ROWS)
        arms = [l for l, _ in CONFIGS if l != "FULL"]
        agg = []
        for lab in arms:
            s = dfp[dfp["label"] == lab]
            if s.empty: continue
            agg.append({"label": lab, "d": float(np.nanmean(s["d_macroAUROC"])),
                        "lo": float(np.nanmean(s["d_lo"])), "hi": float(np.nanmean(s["d_hi"])),
                        "p": float(np.nanmean(s["p_two"]))})
        # [FIX 10] Benjamini-Hochberg across the arm family.
        rej, q = _benjamini_hochberg([a["p"] for a in agg], alpha=ABL_BH_ALPHA)
        print("\n" + "=" * 92 + f"\nPAIRED dAUROC vs FULL — BH-FDR corrected (alpha={ABL_BH_ALPHA})\n" + "=" * 92)
        print(f"{'config':<20}{'d_macroAUROC':>14}{'95% CI':>24}{'p':>8}{'q(BH)':>8}{'sig':>5}")
        for a, rj, qq in zip(agg, rej, q):
            ci = f"[{a['lo']:+.4f},{a['hi']:+.4f}]"
            print(f"{a['label']:<20}{a['d']:>+14.4f}{ci:>24}{a['p']:>8.3f}{qq:>8.3f}{'  *' if rj else '   '}")
        print("* = survives Benjamini-Hochberg at the family level (NOT the raw bootstrap tail).")
        # [plot #5] paired ΔAUROC forest plot — point + 95% CI, filled if BH-sig.
        figf, axf = plt.subplots(figsize=(9, max(4, 0.32 * len(agg))))
        yy = np.arange(len(agg))
        for i, (a, rj) in enumerate(zip(agg, rej)):
            axf.errorbar(a["d"], i, xerr=[[a["d"] - a["lo"]], [a["hi"] - a["d"]]], fmt="o",
                         color=("#2ca02c" if rj else "#999"), mfc=("#2ca02c" if rj else "white"),
                         capsize=3, ms=6)
        axf.axvline(0, color="k", ls=":"); axf.set_yticks(yy)
        axf.set_yticklabels([a["label"] for a in agg], fontsize=7); axf.invert_yaxis()
        axf.set_xlabel("ΔmacroAUROC vs FULL"); axf.set_title("Paired ΔAUROC forest (filled = survives BH-FDR)")
        plt.tight_layout(); figf.savefig(f"{OUT_DIR}/paired_forest.png", dpi=120, bbox_inches="tight"); plt.close(figf)
        print(f"Saved: {OUT_DIR}/paired_forest.png")

    # [plot #4] localization ablation bars — pointing-game / IoU@0.25 per arm,
    # the direct patch-quality companion to the AUROC bars (biggest v14 gap).
    _labs_loc = [l for l in _ORDER if l in set(dfR["label"].astype(str)) and not l.startswith("CTRL")]
    if _labs_loc:
        def _mm(lab, col):
            s = dfR[dfR["label"].astype(str) == lab]
            return float(np.nanmean(pd.to_numeric(s[col], errors="coerce"))) if col in s else float("nan")
        pg = [_mm(l, "macro_pointing_game") for l in _labs_loc]
        iou = [_mm(l, "macro_iou@0.25") for l in _labs_loc]
        xx = np.arange(len(_labs_loc))
        figL, axL = plt.subplots(figsize=(16, 5))
        axL.bar(xx - 0.2, pg, 0.4, label="pointing game", color="#9467bd")
        axL.bar(xx + 0.2, iou, 0.4, label="IoU@0.25", color="#8c564b")
        axL.set_xticks(xx); axL.set_xticklabels(_labs_loc, rotation=55, ha="right", fontsize=8)
        axL.set_ylabel("rate"); axL.set_ylim(0, 1); axL.legend(fontsize=9)
        axL.set_title(f"Localization quality per ablation arm (seed set, VAL, cohort={COHORT_MODE})")
        plt.tight_layout(); figL.savefig(f"{OUT_DIR}/ablation_localization.png", dpi=110, bbox_inches="tight"); plt.close(figL)
        print(f"Saved: {OUT_DIR}/ablation_localization.png")

    if full_pc:
        print("\n" + "=" * 104 + "\nFULL PER-CLASS BENCHMARK  [95% CI]  vs CONTROLS\n" + "=" * 104)
        ctrl_pc = {r["class"]: r for r in PC_ROWS
                   if str(r["label"]) == "CTRL:random" and int(r["seed"]) == ABL_SEEDS[0]}
        def _f(x, w=7, p=2):
            return (f"{x:>{w}.{p}f}" if (x == x) else f"{'-':>{w}}")
        print(f"{'class':<16}{'AUROC':>8}{'95% CI':>18}{'AUPRC':>8}{'rand.AU':>9}"
              f"{'PG':>7}{'IoU25':>7}{'locN':>6}{'pos/neg':>10}{'abst%':>7}")
        for r in full_pc:
            ci = f"[{r['ci_lo']:.3f},{r['ci_hi']:.3f}]" if r['ci_lo'] == r['ci_lo'] else "  n/a"
            cr = ctrl_pc.get(r["class"], {})
            print(f"{r['class']:<16}{r['auroc']:>8.3f}{ci:>18}{r['auprc']:>8.3f}"
                  f"{cr.get('auroc', float('nan')):>9.3f}"
                  f"{_f(r.get('pointing_game', float('nan')))}{_f(r.get('iou@0.25', float('nan')))}"
                  f"{int(r.get('loc_n', 0)):>6}"
                  f"{str(r['support_pos'])+'/'+str(r['support_neg']):>10}{r.get('abstain_pct', float('nan')):>7.1f}")

    dfR_full = dfR[dfR["seed"] == ABL_SEEDS[0]]
    labels = [l for l in _ORDER if l in set(dfR_full["label"].astype(str))]
    def _rowval(lab, col):
        s = dfR_full[dfR_full["label"].astype(str) == lab]
        return float(pd.to_numeric(s[col], errors="coerce").mean()) if not s.empty else float("nan")
    auroc_max = [_rowval(l, "macroAUROC") for l in labels]; auprc_v = [_rowval(l, "macroAUPRC") for l in labels]
    ctrl_rand = _rowval("CTRL:random", "macroAUROC"); x = np.arange(len(labels))
    fig, ax1 = plt.subplots(figsize=(16, 6))
    ax1.bar(x - 0.2, auroc_max, 0.4, label="macroAUROC (max)", color="#2ca02c")
    ax1.bar(x + 0.2, auprc_v, 0.4, label="macroAUPRC", color="#1f77b4")
    if ctrl_rand == ctrl_rand:
        ax1.axhline(ctrl_rand, color="#d62728", ls="--", lw=1.5, label=f"random-patch AUROC={ctrl_rand:.3f}")
    ax1.axhline(0.5, color="#888", ls=":", lw=1); ax1.set_ylabel("score"); ax1.set_ylim(0, 1.0)
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=55, ha="right", fontsize=8)
    ax1.legend(loc="upper right", fontsize=9)
    ax1.set_title(f"Patch-mining ablation (seed {ABL_SEEDS[0]}, n={ABL_SUBSET_N}, VAL, cohort={COHORT_MODE})")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/ablation_overall.png", dpi=110, bbox_inches="tight"); plt.close(fig)

    _all_done = all(f"{lab}|{s}" in done for lab in _ORDER for s in ABL_SEEDS)
    print(f"\n{'Ablations COMPLETE' if _all_done else 'PARTIAL - re-run to finish'}")
    print(f"   overall -> {ABL_OUT_CSV}\n   perclass -> {ABL_PERCLASS_CSV}\n   paired -> {ABL_PAIRED_CSV}")

print("[cell 8] A*-level ablations with BH-FDR, honest fallback/decorrelation arms ready.")
